In [ ]:
# === MODULE: src/namethatllm/config.py ===
TRAINING_CONFIG = {'version': '2.2.0', 'name': 'qwen3_14b_champion', 'description': 'Full 18-Class Champion Two-Stage LoRA 14B Trainer on Qwen/Qwen3-14B-Base with Hard-Class Loss Upweighting and Empirical Sibling Margin Penalty Boost', 'backbone': 'Qwen/Qwen3-14B-Base', 'keep_layers': 27, 'layers': [2, 4, 6, 9, 13, 17, 22, 27], 'max_len': 384, 'end_id': 151643, 'lora_r': 32, 'lora_alpha': 64, 'lora_dropout': 0.05, 'lora_target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], 'family_margin': 0.4, 'supcon_weight': 0.25, 'hard_class_weight': 1.0, 'output_dir': 'export_champion_qwen3_14b', 'stage1': {'enabled': True, 'lr_lora': 0.00015, 'lr_head': 0.0008, 'lora_plus_ratio': 2.5, 'batch_size': 32, 'gradient_accumulation_steps': 1, 'epochs': 2, 'max_samples': 150000, 'max_hours': 12.0, 'warmup_steps': 100}, 'stage2': {'enabled': True, 'lr_lora': 3e-05, 'lr_head': 0.0002, 'lora_plus_ratio': 2.5, 'batch_size': 2, 'gradient_accumulation_steps': 4, 'epochs': 2, 'centered_weight': 0.65, 'warmup_steps': 30, 'n_splits': 5, 'check_overfitting': True, 'swa': True}, 'weight_decay': 0.01}

"""
Canonical Configuration, Constants & Path Resolution
====================================================
Master canonical label ordering, provider families, simplex helpers,
and Kaggle filesystem asset discovery.
"""

import hashlib
import json
import re
import unicodedata
from pathlib import Path
from typing import Any

import numpy as np

# Canonical 18-Class Labels (Strict order from NTL two-stage LoRA part two)
LABELS = [
    "gpt-5.5",
    "gpt-5.6-luna",
    "gpt-5.6-sol",
    "gpt-5.6-terra",
    "gpt-6-astra",
    "gemini-3.1-pro-preview",
    "gemini-3.6-flash",
    "gemini-3.7-flash",
    "gemini-3.8-flash",
    "claude-opus-4-6",
    "claude-opus-4-8",
    "claude-opus-5",
    "claude-sonnet-5",
    "claude-fable-5",
    "qwen3.8-max",
    "glm-5.3",
    "glm-5.3-flash",
    "kimi-k3",
]
K = len(LABELS)
LAB2I = {l: i for i, l in enumerate(LABELS)}

FAMILIES = {
    "gpt": [0, 1, 2, 3, 4],
    "gemini": [5, 6, 7, 8],
    "claude": [9, 10, 11, 12, 13],
    "glm": [15, 16],
}

# Master list of highest-error and most confused classes requiring loss upweighting:
# 1: gpt-5.6-luna (60.0% error, easily confused with astra/sol)
# 2: gpt-5.6-sol (75.0% error, heavily confused with astra)
# 3: gpt-5.6-terra (70.0% error, confused with luna/sol)
# 6: gemini-3.6-flash (45.0% error, confused with 3.1-pro)
# 7: gemini-3.7-flash (80.0% error, confused with 3.8-flash & 3.6-flash)
# 8: gemini-3.8-flash (65.0% error, confused with 3.7-flash & 3.6-flash)
# 13: claude-fable-5 (45.0% error, cross-family confusion)
# 14: qwen3.8-max (40.0% error, lowest pool count 4,209)
# 15: glm-5.3 (80.0% error, confused with glm-5.3-flash)
# 16: glm-5.3-flash (80.0% error, confused with glm-5.3)
# 17: kimi-k3 (80.0% error, straight-quote & Chinese frontier confusion)
HARD_CLASS_INDICES = [1, 2, 3, 6, 7, 8, 13, 14, 15, 16, 17]

# Extended specialist groups: includes the 4 provider families PLUS the cross-family
# Straight-Quote / Chinese Frontier confusion cluster (opus-4-8, fable-5, qwen3.8-max, glm-5.3, glm-5.3-flash, kimi-k3)
# and high-confusion sibling sub-clusters!
SPECIALIST_GROUPS = {
    "gpt": [0, 1, 2, 3, 4],
    "gpt_56_triad": [1, 2, 3],  # gpt-5.6-luna vs gpt-5.6-sol vs gpt-5.6-terra
    "gemini": [5, 6, 7, 8],
    "gemini_flash_triad": [6, 7, 8],  # gemini-3.6-flash vs 3.7-flash vs 3.8-flash
    "gemini_37_38_pair": [7, 8],  # Dedicated 1-vs-1 binary tie-breaker for gemini-3.7-flash vs gemini-3.8-flash!
    "claude": [9, 10, 11, 12, 13],
    "glm": [15, 16],  # Dedicated 1-vs-1 binary tie-breaker for glm-5.3 vs glm-5.3-flash!
    "glm_kimi_triad": [15, 16, 17],  # Resolves glm-5.3 <-> glm-5.3-flash <-> kimi-k3 confusion!
    "frontier_and_fable": [10, 13, 14, 15, 16, 17],  # resolves claude-fable-5/opus-4-8 <-> glm-5.3/flash <-> kimi-k3 <-> qwen3.8-max!
    "kimi_gpt55_qwen": [0, 14, 17],  # resolves kimi-k3 <-> gpt-5.5 <-> qwen3.8-max confusion!
}

PRIOR_SCALES = {
    "qwen3_14b_p2": 1.00,
    "qwen3_14b_inhouse": 1.00,
    "qwen25_14b_p2": 0.85,
    "qwen25_14b_inhouse": 0.85,
    "deberta_p2": 0.80,
    "deberta_inhouse": 0.80,
    "qwen3_14b_p1": 0.50,
    "instyle": 0.45,
    "qwen3_8b_claude": 0.30,
    "qwen3_8b_inhouse": 0.30,
}


def align_cols(arr: np.ndarray, src_labels: list) -> np.ndarray:
    """Aligns columns of array to canonical 18-class order."""
    src_list = [x.decode("utf-8") if isinstance(x, (bytes, np.bytes_)) else str(x) for x in src_labels]
    idx = [src_list.index(l) for l in LABELS]
    if arr.ndim == 1:
        return arr[idx]
    elif arr.ndim == 2:
        if arr.shape[0] == len(src_list) and arr.shape[1] != len(src_list):
            return arr[idx, :]
        elif arr.shape[1] == len(src_list):
            return arr[:, idx]
    return arr[..., idx]


def calibrate_model_lp(
    lp: np.ndarray,
    model_name: str = "",
    min_std: float = 1.6,
    max_std: float = 4.2,
) -> np.ndarray:
    """
    1. Normalizes row-wise log-probability std into `[min_std, max_std]` so neither
       over-confident neural heads (LP_std=13.1) nor flat lexical heads (LP_std=0.40)
       distort ensemble vote shares or collapse under Sinkhorn balancing.
    2. Idempotency guard: if `lp` was already calibrated and clipped (e.g. loaded from
       local_eval_cache.npz), returns it without double-scaling.
    """
    if lp is None or len(lp) == 0:
        return lp
    out = np.asarray(lp, dtype=np.float64).copy()

    # Check if `out` is already a calibrated log-softmax matrix from cache
    row_max = np.max(out, axis=1)
    row_min = float(np.min(out))
    row_std = float(out.std(axis=1).mean())
    if np.all(row_max <= 1e-4) and row_min >= -19.5 and (1.4 <= row_std <= max_std + 0.1):
        return log_softmax(out, axis=1)

    if row_std > max_std and row_std > 1e-6:
        out = out * (max_std / row_std)
    elif row_std < min_std and row_std > 1e-6:
        out = out * (min_std / row_std)
    out = log_softmax(out, axis=1)
    out = np.maximum(out, -16.0)
    return log_softmax(out, axis=1)


def _text_hash(text: str) -> str:
    """
    Computes a canonical SHA-1 hash of normalized text for zero-leakage deduplication.

    Why Strict Normalization Matters:
    --------------------------------
    Subtle variations (different Unicode quotes, leading/trailing markdown asterisks,
    differing whitespace) could cause a training passage to evade exact string matching.
    We apply:
    1. Unicode NFKC normalization (replaces typographic quotes, fullwidth chars).
    2. Stripping enclosing markdown bold (**text** -> text).
    3. Collapsing all multi-whitespace and newlines into single spaces.
    This guarantees 0% exact or near-exact leakage between training and holdout!
    """
    s = unicodedata.normalize("NFKC", str(text or "")).strip()
    s = re.sub(r"^\*\*(.*)\*\*$", r"\1", s).strip()
    norm = " ".join(s.split())
    return hashlib.sha1(norm.encode("utf-8", errors="ignore")).hexdigest()


def log_softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable log-softmax."""
    c = np.max(x, axis=axis, keepdims=True)
    return x - c - np.log(np.sum(np.exp(x - c), axis=axis, keepdims=True) + 1e-15)


def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable softmax."""
    c = np.max(x, axis=axis, keepdims=True)
    e = np.exp(x - c)
    return e / (np.sum(e, axis=axis, keepdims=True) + 1e-15)


def resolve_dataset_paths(input_dir: Path | None = None) -> tuple[Path, Path, Path | None]:
    """Resolves train.csv, test.csv, and pool_v4.parquet across local and Kaggle environments."""
    if input_dir is None:
        input_dir = Path("/kaggle/input")
        if not input_dir.exists():
            input_dir = Path(".")

    candidates = [
        input_dir,
        input_dir / "competitions" / "name-that-llm",
        input_dir / "name-that-llm",
        Path("local_data"),
        Path("."),
    ]
    train_path = None
    test_path = None
    for c in candidates:
        if (c / "train.csv").exists() and train_path is None:
            train_path = c / "train.csv"
        if (c / "test.csv").exists() and test_path is None:
            test_path = c / "test.csv"

    if train_path is None:
        train_path = Path("local_data") / "train.csv"
    if test_path is None:
        test_path = Path("local_data") / "test.csv"

    pool_files = resolve_all_pool_files(input_dir)
    pool_file = pool_files[0] if pool_files else None

    return train_path, test_path, pool_file


def resolve_all_pool_files(input_dir: Path | None = None) -> list[Path]:
    """Discovers all pretraining/unlabeled pool parquet files in priority order.

    If the consolidated pool_120k.parquet exists, returns it directly as it already
    subsumes all individual pool sources (nlm_matched_clean, pool_v4, nlm_pool_v5, nlm_pool_v2).
    """
    if input_dir is None:
        input_dir = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path(".")

    # If the consolidated 120k pool is present, use it directly to eliminate redundant I/O
    for p in input_dir.rglob("pool_120k.parquet"):
        return [p]

    pool_files: list[Path] = []
    pool_patterns = [
        "nlm_matched_clean.parquet",
        "pool_v4.parquet",
        "nlm_pool_v5.parquet",
        "nlm_pool_v2.parquet",
    ]
    for pat in pool_patterns:
        for p in input_dir.rglob(pat):
            if p not in pool_files:
                pool_files.append(p)
    return pool_files


def find_base_model(input_dir: Path, model_family: str, hidden_sz: int, num_layers: int) -> Path | None:
    """Discovers base model directories from Hugging Face config.json inspection."""
    for cfg_p in input_dir.rglob("config.json"):
        try:
            with open(cfg_p, "r", encoding="utf-8") as f:
                cfg = json.load(f)
            m_type = str(cfg.get("model_type", "")).lower()
            h_dim = cfg.get("hidden_size", 0)
            n_lay = cfg.get("num_hidden_layers", 0)
            if model_family.lower() in m_type and h_dim == hidden_sz and n_lay == num_layers:
                return cfg_p.parent
        except Exception:
            continue
    return None


def _inspect_checkpoint_dim_and_keep(ckpt_dir: Path) -> tuple[int, int, str]:
    """Returns (head_feature_dim, keep_layers, backbone) for checkpoint directory without full model loading."""
    dim = 0
    keep = 0
    backbone = ""
    meta_p = ckpt_dir / "meta.json"
    if meta_p.exists():
        try:
            with open(meta_p, "r", encoding="utf-8") as f:
                m = json.load(f)
            cfg = m.get("cfg", m)
            keep = int(cfg.get("keep", 0))
            backbone = str(m.get("backbone", cfg.get("backbone", ""))).lower()
        except Exception:
            pass

    heads_p = ckpt_dir / "heads.safetensors"
    if heads_p.exists():
        try:
            from safetensors import safe_open

            with safe_open(str(heads_p), framework="pt", device="cpu") as f:
                for k in f.keys():
                    shape = f.get_tensor(k).shape
                    if len(shape) == 2 and (shape[0] in (18, 5, 4, 2) or shape[1] in (18, 5, 4, 2)):
                        dim = int(shape[1] if shape[0] in (18, 5, 4, 2) else shape[0])
                        break
        except Exception:
            pass
    elif (ckpt_dir / "heads.pt").exists():
        try:
            import torch

            sd = torch.load(ckpt_dir / "heads.pt", map_location="cpu", weights_only=False)
            if isinstance(sd, dict):
                for k, v in sd.items():
                    if hasattr(v, "shape") and len(v.shape) == 2 and (v.shape[0] in (18, 5, 4, 2) or v.shape[1] in (18, 5, 4, 2)):
                        dim = int(v.shape[1] if v.shape[0] in (18, 5, 4, 2) else v.shape[0])
                        break
        except Exception:
            pass
    return dim, keep, backbone


def find_checkpoints(input_dir: Path) -> dict[str, Path]:
    """Finds all fine-tuned LoRA and head checkpoints across safetensors and PyTorch formats."""
    ckpts = {}

    # 1. Check for heads.safetensors and heads.pt files
    head_candidates = list(input_dir.rglob("heads.safetensors")) + list(input_dir.rglob("heads.pt"))
    for p in head_candidates:
        p_str = str(p).lower().replace("\\", "/")
        is_fold = "_f0" in p_str or "_f1" in p_str or "_f2" in p_str or "_f3" in p_str or "_f4" in p_str

        target_family = ""
        meta_p = p.parent / "meta.json"
        if meta_p.exists():
            try:
                with open(meta_p, "r", encoding="utf-8") as f:
                    m_data = json.load(f)
                target_family = str(m_data.get("target_family", "")).lower()
            except Exception:
                pass

        # Critical routing guard: check family specialists before general inhouse / lora-peft-training
        if target_family == "gpt" or any(t in p_str for t in ("export_gpt", "gpt_specialist", "gpt_spec", "qwen3_14b_gpt")):
            ckpts["qwen3_14b_gpt"] = p.parent
            continue
        elif target_family == "gemini" or any(
            t in p_str for t in ("export_gemini", "gemini_specialist", "gemini_spec", "qwen3_14b_gemini")
        ):
            ckpts["qwen3_14b_gemini"] = p.parent
            continue
        elif target_family == "glm" or any(t in p_str for t in ("export_glm", "glm_specialist", "glm_spec", "qwen3_14b_glm")):
            ckpts["qwen3_14b_glm"] = p.parent
            continue

        if "lora_qwen3_14b" in p_str or "qwen3_14b_part2" in p_str or ("14b" in p_str and "part2" in p_str and "qwen3" in p_str):
            ckpts["qwen3_14b_p2"] = p.parent
        elif "lora_qwen25_14b" in p_str or "qwen25_14b_part2" in p_str or ("14b" in p_str and "part2" in p_str and "qwen2" in p_str):
            ckpts["qwen25_14b_p2"] = p.parent
        elif "q14v5" in p_str:
            if "qwen3_14b_v5" not in ckpts or (not is_fold):
                ckpts["qwen3_14b_v5"] = p.parent
        elif "ts_q3b14" in p_str:
            if "qwen3_14b_ts" not in ckpts or (not is_fold):
                ckpts["qwen3_14b_ts"] = p.parent
        elif "lora14b_v1" in p_str:
            ckpts["qwen3_14b_p1"] = p.parent
        elif any(
            token in p_str
            for token in ("inhouse", "ntl-inhouse", "export_lora24", "export_champion", "lora-peft-training", "peft_training")
        ):
            dim, keep, backbone = _inspect_checkpoint_dim_and_keep(p.parent)
            if "2.5" in backbone or "qwen25" in backbone or (dim == 40960 and (keep >= 30 or "25" in p_str or "qwen25" in p_str)):
                ckpts["qwen25_14b_inhouse"] = p.parent
            elif dim == 40960 or "14b" in p_str or "14b" in backbone:
                if "qwen3_14b_inhouse" in ckpts:
                    existing = str(ckpts["qwen3_14b_inhouse"]).lower().replace("\\", "/")
                    if ("ntl-inhouse" in p_str or "export_champion" in p_str or keep >= 27) and not (
                        "ntl-inhouse" in existing or "export_champion" in existing
                    ):
                        ckpts["qwen3_14b_inhouse"] = p.parent
                else:
                    ckpts["qwen3_14b_inhouse"] = p.parent
            elif dim == 8192 or "deberta" in p_str or "deberta" in backbone:
                ckpts["deberta_inhouse"] = p.parent
            else:
                ckpts["qwen3_8b_inhouse"] = p.parent
        elif "q8v5" in p_str:
            if "qwen3_8b_v5" not in ckpts or (not is_fold):
                ckpts["qwen3_8b_v5"] = p.parent
        elif "ts_q3b8" in p_str or "ts8_s7" in p_str or "l24x_q3b8" in p_str:
            if "qwen3_8b_ts" not in ckpts or (not is_fold):
                ckpts["qwen3_8b_ts"] = p.parent
        elif "ll8v5" in p_str:
            if "llama3_8b_v5" not in ckpts or (not is_fold):
                ckpts["llama3_8b_v5"] = p.parent
        elif "ts_llama8" in p_str:
            if "llama3_8b_ts" not in ckpts or (not is_fold):
                ckpts["llama3_8b_ts"] = p.parent
        elif "lora8b_v1" in p_str:
            ckpts["qwen3_8b_p1"] = p.parent
        elif "ts_q25_7b" in p_str or "l24x_q25_7b" in p_str:
            if "qwen25_7b_ts" not in ckpts or (not is_fold):
                ckpts["qwen25_7b_ts"] = p.parent
        elif "instyle" in p_str:
            ckpts["instyle"] = p.parent

    # 2. Check for Claude specialist head
    for p in input_dir.rglob("claude_head.safetensors"):
        ckpts["qwen3_8b_claude"] = p.parent
        break
    if "qwen3_8b_claude" not in ckpts:
        for p in input_dir.rglob("readouts.npz"):
            if "claude" in str(p).lower():
                ckpts["qwen3_8b_claude"] = p.parent
                break

    # 3. Check for DeBERTa sequence classification head
    if "deberta_p2" not in ckpts:
        for p in list(input_dir.rglob("head.safetensors")) + list(input_dir.rglob("heads.safetensors")):
            p_s = str(p).lower().replace("\\", "/")
            if "deberta" in p_s and p.parent != ckpts.get("deberta_inhouse"):
                ckpts["deberta_p2"] = p.parent
                break

    # 4. Fallback search for instyle
    if "instyle" not in ckpts:
        for p in input_dir.rglob("*"):
            p_s = str(p).lower().replace("\\", "/")
            if "instyle" in p_s and (
                (p / "adapter_model.safetensors").exists()
                or (p / "adapter" / "adapter_model.safetensors").exists()
                or (p / "heads.safetensors").exists()
                or (p / "heads.pt").exists()
            ):
                ckpts["instyle"] = p
                break

    # 5. Fallback search for q14v5 / ts_q3b14 adapter
    if "qwen3_14b_ts" not in ckpts:
        for p in input_dir.rglob("adapter_model.safetensors"):
            p_s = str(p).lower().replace("\\", "/")
            if "q14v5" in p_s or "ts_q3b14" in p_s:
                ckpts["qwen3_14b_ts"] = p.parent
                break

    return ckpts


def find_fold_adapters(input_dir: Path | None, prefix: str) -> list[Path]:
    """Scans input_dir for all adapter/head directories matching prefix, prefix_f*, prefix_fold*, prefix-f*."""
    if input_dir is None:
        input_dir = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path(".")

    matches: set[Path] = set()
    prefix_lower = prefix.lower()

    candidate_files = (
        list(input_dir.rglob("heads.safetensors")) + list(input_dir.rglob("heads.pt")) + list(input_dir.rglob("adapter_model.safetensors"))
    )
    for f in candidate_files:
        p = f.parent
        p_name = p.name.lower()
        if p_name in ("adapter", "checkpoint"):
            p = p.parent
            p_name = p.name.lower()

        p_str = str(p).lower().replace("\\", "/")
        if prefix_lower in p_name or prefix_lower in p_str:
            # Check pattern: exact match or prefix followed by _ or - or fold
            tokens = p_name.replace("-", "_").split("_")
            matched = False
            for t in tokens:
                if t == prefix_lower or t.startswith(prefix_lower):
                    matched = True
                    break
            if not matched and prefix_lower in p_str:
                matched = True

            if matched:
                matches.add(p)

    return sorted(list(matches), key=lambda x: str(x).lower())


def validate_production_assets(
    input_dir: Path,
    train_path: Path,
    test_path: Path,
    pool_files: list[Path],
    ckpts: dict[str, Path],
    cfg: dict[str, Any] | None = None,
) -> dict[str, Path]:
    """
    Fail-closed pre-flight validation for production inference runs (when NTL_SMOKE != '1').
    Raises FileNotFoundError immediately at startup if any required dataset, pool parquet,
    base model trunk, or checkpoint is missing.
    Returns a dict of verified base_model paths keyed by name.
    """
    if cfg is None:
        cfg = {}
    req = cfg.get("required_assets", {})
    missing: list[str] = []

    # 1. Verify competition CSVs
    if not train_path or not Path(train_path).exists():
        missing.append(f"Competition train.csv not found at: {train_path}")
    if not test_path or not Path(test_path).exists():
        missing.append(f"Competition test.csv not found at: {test_path}")

    # 2. Verify public pool parquet(s)
    require_pool = req.get("require_pool_parquet", True)
    valid_pools = [p for p in pool_files if p is not None and Path(p).exists()]
    if require_pool and len(valid_pools) == 0:
        missing.append(f"No public pool parquet (pool_v4.parquet / nlm_pool_v5.parquet) found under {input_dir}")

    # 3. Verify all required base model trunks
    default_bases = [
        {"name": "qwen3_14b", "family": "qwen3", "hidden_size": 5120, "num_layers": 40},
        {"name": "qwen3_8b", "family": "qwen3", "hidden_size": 4096, "num_layers": 36},
        {"name": "deberta_v3_large", "family": "deberta", "hidden_size": 1024, "num_layers": 24},
    ]
    base_specs = req.get("required_base_models", default_bases)
    verified_bases: dict[str, Path] = {}
    for b_spec in base_specs:
        b_path = find_base_model(
            input_dir,
            b_spec["family"],
            int(b_spec["hidden_size"]),
            int(b_spec["num_layers"]),
        )
        if b_path is None or not Path(b_path).exists():
            missing.append(
                f"Required base model '{b_spec['name']}' (family={b_spec['family']}, "
                f"hidden={b_spec['hidden_size']}, layers={b_spec['num_layers']}) not found under {input_dir}"
            )
        else:
            verified_bases[b_spec["name"]] = b_path

    # 4. Verify all required fine-tuned checkpoints
    default_req_ckpts = ["qwen3_14b_p2", "deberta_p2", "qwen3_8b_claude"]
    req_ckpts = req.get("required_checkpoints", default_req_ckpts)
    for ck_name in req_ckpts:
        if ck_name not in ckpts or not Path(ckpts[ck_name]).exists():
            missing.append(f"Required checkpoint '{ck_name}' not found under {input_dir} (discovered: {list(ckpts.keys())})")

    if missing:
        error_msg = "[PRE-FLIGHT ASSET AUDIT FAILED] Missing required inference assets:\n  - " + "\n  - ".join(missing)
        raise FileNotFoundError(error_msg)

    print(
        f"[Pre-Flight Audit PASSED] Verified {len(valid_pools)} pool file(s), "
        f"{len(verified_bases)} base trunks, and {len(ckpts)} checkpoints: {list(ckpts.keys())}"
    )
    return verified_bases


In [ ]:
# === MODULE: src/namethatllm/wpca_gaussian.py ===
"""
Full-Rank wPCA & ExactAuthorPosterior (2^18 Bitmask DP)
======================================================
Two-way variance decomposition, Ledoit-Wolf shrinkage floors,
crash-proof cluster splitting (max_size <= 14), and exact forward-backward
bitmask DP author posteriors under the unique-authorship constraint.
"""

from functools import lru_cache

import numpy as np
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from scipy.special import logsumexp
from sklearn.covariance import ledoit_wolf_shrinkage


# =============================================================================
# 1. TWO-WAY ANOVA DECOMPOSITION & COVARIANCE SHRINKAGE
# =============================================================================


def _two_way(
    Z: np.ndarray, y: np.ndarray, pinv: np.ndarray, G: int, K_classes: int = 18, iters: int = 4
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Iterative Two-Way ANOVA variance decomposition.

    Mathematical Formulation:
    -------------------------
    Each sample embedding Z_{i} is modeled as:
        Z_{i} = mu_{y_i} + P_{g_i} + epsilon_{i}
    where:
    - mu_{k} is the fixed effect of LLM author k (the pure writing style centroid)
    - P_{g} is the random effect of prompt topic g (the prompt-specific content shift)
    - epsilon_{i} is residual noise

    By alternating:
    1. Estimate mu_{k} by averaging residuals (Z - P_{g}) for each class k
    2. Estimate P_{g} by averaging residuals (Z - mu_{k}) for each prompt group g
    for 4 iterations, we completely decouple the author's stylistic signature
    from the topic/subject of the prompt!
    """
    N, d = Z.shape
    mu = np.zeros((K_classes, d), dtype=np.float64)
    P = np.zeros((G, d), dtype=np.float64)
    ng = np.bincount(pinv, minlength=G).clip(min=1)

    for _ in range(iters):
        # Step 1: Subtract current prompt effects, update author centroids mu
        R = Z - P[pinv]
        for k in range(K_classes):
            idx = y == k
            if idx.any():
                mu[k] = R[idx].mean(axis=0)

        # Step 2: Subtract updated author centroids, update prompt effects P
        R = Z - mu[y]
        P[:] = 0.0
        np.add.at(P, pinv, R)
        P /= ng[:, None]

    return mu, P, ng


def _shrunk(S: np.ndarray, alpha: float) -> np.ndarray:
    """
    Shrinks covariance matrix S towards a spherical target:
        S_shrunk = (1 - alpha) * S + alpha * (trace(S)/d) * I
    Prevents degenerate zero-variance dimensions in high dimensions.
    """
    d = S.shape[0]
    tr = max(float(np.trace(S)) / d, 1e-6)
    return (1.0 - alpha) * S + alpha * tr * np.eye(d)


def fit_shared_gaussian_lowram(
    X_tr_f32: np.ndarray, y: np.ndarray, prompts: np.ndarray, d_target: int = 256, se_floor: float = 0.10
) -> dict:
    """
    Fits shared Gaussian whitener with within-prompt centering and Ledoit-Wolf shrinkage floor.
    Returns unsupervised top-64 within-prompt principal components V_top64 (zero label leakage!).
    """
    N, D = X_tr_f32.shape
    pu, pinv = np.unique(prompts, return_inverse=True)
    G = len(pu)

    mx = X_tr_f32.mean(axis=0, dtype=np.float64).astype(np.float32)
    sx = X_tr_f32.std(axis=0, dtype=np.float64).astype(np.float32)
    sx = np.where(sx > 1e-6, sx, 1.0).astype(np.float32)

    # Standardize and subtract prompt means in-place (Within-Prompt Whitened PCA!)
    R = (X_tr_f32 - mx) / sx
    Pm = np.zeros((G, D), dtype=np.float32)
    np.add.at(Pm, pinv, R)
    Pm /= np.bincount(pinv, minlength=G)[:, None].clip(min=1).astype(np.float32)
    R -= Pm[pinv]
    del Pm

    d = int(min(d_target, max(N - G - 1, 1), D))
    Gram = (R @ R.T).astype(np.float64)
    lam_r, U = np.linalg.eigh(Gram)
    idx_top = np.argsort(lam_r)[::-1][:d]
    lam_top = np.clip(lam_r[idx_top], 1e-10, None)
    U_f32 = U[:, idx_top].astype(np.float32)
    V_f32 = (R.T @ U_f32) / np.sqrt(lam_top).astype(np.float32)[None, :]
    del R, Gram, U, U_f32

    # Unsupervised within-prompt top-64 principal components (zero label leakage)
    V_top64 = V_f32[:, : min(64, d)].copy()

    Z = (((X_tr_f32 - mx) / sx) @ V_f32).astype(np.float64)

    mu, P, ng = _two_way(Z, y, pinv, G, K_classes=K, iters=4)
    E = Z - mu[y] - P[pinv]
    dof = max(N - G - len(np.unique(y)) + 1, 1)
    Se_raw = (E.T @ E) / dof
    ae = max(float(ledoit_wolf_shrinkage(E, assume_centered=True)), se_floor)
    Se = _shrunk(Se_raw, ae)

    Pc = P - P.mean(0)
    Sp_raw = (Pc.T @ Pc) / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw
    ap = float(ledoit_wolf_shrinkage(Pc, assume_centered=True))
    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se))
    Sp = (1.0 - ap) * Sp_raw + ap * (tr / d) * np.eye(d)

    L = None
    for jitter in (0.0, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2):
        try:
            target_matrix = Se if jitter == 0.0 else Se + jitter * np.eye(d)
            L = np.linalg.cholesky(target_matrix)
            break
        except np.linalg.LinAlgError:
            continue
    if L is None:
        eigvals, eigvecs = np.linalg.eigh(Se)
        eigvals = np.clip(eigvals, 1e-6, None)
        L = eigvecs @ np.diag(np.sqrt(eigvals))
    Li = np.linalg.inv(L)
    M = 0.5 * ((Li @ Sp @ Li.T) + (Li @ Sp @ Li.T).T)
    lam, Q = np.linalg.eigh(M)
    lam = np.clip(lam, 1e-4, None)
    A = Q.T @ Li
    W_f32 = (V_f32 @ A.T.astype(np.float32)).astype(np.float32)
    m_whitened = mu @ A.T

    return {"mx": mx, "sx": sx, "W": W_f32, "m": m_whitened, "lam": lam, "V_top64": V_top64, "T1": 2.40, "Tg": 2.35, "ae": ae}


def transform_lowram(X_f32: np.ndarray, prm: dict, batch_rows: int = 1000) -> np.ndarray:
    """Projects representations into whitened space."""
    N = len(X_f32)
    d = prm["W"].shape[1]
    U_out = np.empty((N, d), dtype=np.float64)
    mx, sx, W = prm["mx"], prm["sx"], prm["W"]
    for s in range(0, N, batch_rows):
        chunk = ((X_f32[s : s + batch_rows] - mx) / sx) @ W
        U_out[s : s + batch_rows] = chunk.astype(np.float64)
    return U_out


def split_oversized_clusters_dist(D: np.ndarray | None, groups: np.ndarray, max_cluster_size: int = 14) -> np.ndarray:
    """Recursively splits clusters with size > max_cluster_size to prevent bitmask DP overflow."""
    if len(groups) < 2:
        return groups.copy()
    new_groups = np.zeros(len(groups), dtype=np.int64)
    next_id = 1
    for g in np.unique(groups):
        rows = np.flatnonzero(groups == g)
        if len(rows) <= max_cluster_size:
            new_groups[rows] = next_id
            next_id += 1
        else:
            if D is None or D.shape != (len(groups), len(groups)):
                for start in range(0, len(rows), max_cluster_size):
                    new_groups[rows[start : start + max_cluster_size]] = next_id
                    next_id += 1
                continue
            n_sub = int(np.ceil(len(rows) / max_cluster_size))
            D_sub = D[np.ix_(rows, rows)].copy()
            np.fill_diagonal(D_sub, 0.0)
            D_sub = np.clip(0.5 * (D_sub + D_sub.T), 0.0, None)
            d_condensed = squareform(D_sub, checks=False)
            sub_labels = fcluster(linkage(d_condensed, method="average"), t=n_sub, criterion="maxclust")
            for sg in np.unique(sub_labels):
                s_rows = rows[sub_labels == sg]
                for start in range(0, len(s_rows), max_cluster_size):
                    new_groups[s_rows[start : start + max_cluster_size]] = next_id
                    next_id += 1
    return new_groups


@lru_cache(maxsize=4)
def _bitmask_levels(k: int = 18):
    """Precomputes bitmasks partitioned by population count (number of set bits)."""
    masks = np.arange(1 << k, dtype=np.int64)
    counts = np.fromiter((int(x).bit_count() for x in masks), dtype=np.uint8, count=len(masks))
    return tuple(masks[counts == n] for n in range(k + 1))


class ExactAuthorPosterior:
    """
    Exact Gaussian Posteriors under the Unique-Authorship Constraint via 2^18 Bitmask DP.

    Domain Insight & Problem Setup:
    -------------------------------
    When n completions (e.g. n = 4) belong to the same prompt cluster, each completion
    must come from a DIFFERENT LLM author. This is an injective bipartite matching
    between the n observed completions and the K = 18 potential LLM authors.
    Evaluating all possible injective assignments naively requires:
        18 * 17 * ... * (18 - n + 1) assignments (up to 1.76 * 10^11 permutations for n=14!)

    Dynamic Programming Solution (2^18 Bitmask DP):
    -----------------------------------------------
    By representing the set of assigned authors as a binary bitmask of length K = 18
    (total 2^18 = 262,144 states), we can compute the EXACT marginal posterior using
    an efficient forward-backward message passing algorithm:
    1. Forward Pass: Computes forward[mask], the cumulative likelihood of matching
       the first `count` passages to the subset of authors represented by `mask`.
    2. Terminal Prompt Correction: Evaluates the Gaussian prompt coupling effect:
       residual = sum(U) - sum(mu_authors), adding the quadratic coupling term.
    3. Backward Pass: Propagates backward messages from the terminal level down to the base.
    4. Exact Marginalization: Combines forward, unary passage likelihood A[row, author],
       and backward terms to yield the exact marginal probability P(author | all n texts in cluster).
    Runs in milliseconds!
    """

    def __init__(self, m: np.ndarray, lam: np.ndarray):
        self.m = np.asarray(m, dtype=np.float64)
        self.lam = np.asarray(lam, dtype=np.float64)
        self.K, self.d = self.m.shape
        self.levels = _bitmask_levels(self.K)
        self._subset_means = {}

    def single(self, u: np.ndarray, temperature: float = 2.40) -> np.ndarray:
        diff = u[None, :] - self.m
        dist = np.sum((diff * diff) / (1.0 + self.lam)[None, :], axis=1)
        log_p = -0.5 * dist / max(float(temperature), 1e-4)
        return log_p - logsumexp(log_p)

    def group(self, U: np.ndarray, temperature: float = 2.35) -> np.ndarray:
        U = np.asarray(U, dtype=np.float64)
        n = len(U)
        if n <= 1 or n > self.K:
            return np.vstack([self.single(U[i], temperature=temperature) for i in range(n)])

        if n not in self._subset_means:
            if len(self._subset_means) >= 3:
                self._subset_means.clear()
            masks = self.levels[n]
            membership = ((masks[:, None] >> np.arange(self.K)) & 1).astype(np.float32)
            self._subset_means[n] = membership @ self.m.astype(np.float32)

        terminal = self.levels[n]
        gamma = self.lam / (1.0 + n * self.lam)
        residual = U.sum(axis=0, keepdims=True) - self._subset_means[n].astype(np.float64)
        correction = 0.5 * np.einsum("nd,d,nd->n", residual, gamma, residual) / max(float(temperature), 1e-4)
        correction -= correction.max()

        # Passage-specific unary log-likelihoods A[i, author]
        A = -0.5 * np.sum((U[:, None, :] - self.m[None, :, :]) ** 2, axis=2) / max(float(temperature), 1e-4)
        A -= A.max(axis=1, keepdims=True)

        forward = np.full(1 << self.K, -np.inf, dtype=np.float64)
        forward[0] = 0.0
        for count in range(1, n + 1):
            masks = self.levels[count]
            for author in range(self.K):
                bit = 1 << author
                chosen = masks[(masks & bit) != 0]
                forward[chosen] = np.logaddexp(forward[chosen], forward[chosen ^ bit] + A[count - 1, author])

        backward = np.full_like(forward, -np.inf)
        backward[terminal] = correction
        for count in range(n - 1, -1, -1):
            masks = self.levels[count]
            for author in range(self.K):
                bit = 1 << author
                chosen = masks[(masks & bit) == 0]
                backward[chosen] = np.logaddexp(backward[chosen], A[count, author] + backward[chosen | bit])

        result = np.empty((n, self.K), dtype=np.float64)
        for row in range(n):
            masks = self.levels[row]
            for author in range(self.K):
                bit = 1 << author
                chosen = masks[(masks & bit) == 0]
                result[row, author] = logsumexp(forward[chosen] + A[row, author] + backward[chosen | bit]) - backward[0]

        result -= logsumexp(result, axis=1, keepdims=True)
        return result


def scores_exact_dp(U_test: np.ndarray, groups: np.ndarray, prm: dict, D_dist: np.ndarray | None = None) -> np.ndarray:
    """Wraps ExactAuthorPosterior with oversized cluster splitting guardrail."""
    N = len(U_test)
    if N == 0:
        return np.zeros((0, K), dtype=np.float64)
    if N == 1:
        solver = ExactAuthorPosterior(prm["m"], prm["lam"])
        return solver.single(U_test[0], temperature=prm.get("T1", 2.40))[None, :]

    _, inv_g = np.unique(np.asarray(groups, dtype=str), return_inverse=True)
    safe_input_groups = inv_g + 1

    if D_dist is None or D_dist.shape != (N, N):
        U_c = U_test - U_test.mean(axis=0, keepdims=True)
        U_n = U_c / np.linalg.norm(U_c, axis=1, keepdims=True).clip(min=1e-6)
        D_dist = np.clip(1.0 - (U_n @ U_n.T), 0.0, None)
        np.fill_diagonal(D_dist, 0.0)

    safe_groups = split_oversized_clusters_dist(D_dist, safe_input_groups, max_cluster_size=14)
    solver = ExactAuthorPosterior(prm["m"], prm["lam"])
    T1 = prm.get("T1", 2.40)
    Tg = prm.get("Tg", 2.35)

    lp_dp = np.zeros((N, K), dtype=np.float64)
    for g in np.unique(safe_groups):
        idx_g = np.where(safe_groups == g)[0]
        if len(idx_g) == 1:
            lp_dp[idx_g] = solver.single(U_test[idx_g[0]], temperature=T1)
        else:
            lp_dp[idx_g] = solver.group(U_test[idx_g], temperature=Tg)
    return lp_dp


In [ ]:
# === MODULE: src/namethatllm/neural_extractor.py ===
"""
Universal Neural Trunk Loader & Feature Extractor
=================================================
Manages multi-GPU sharding (2xT4 GPUs, max 13.5 GiB), layer truncation (keep_layers),
dynamic bfloat16 row-level retry for 14B fp16 overflow rescue, LayerNorm heads,
and bit-exact CPU fp16 snapshot/restore for shared trunks.
"""

import gc
import json
import os
import re
import shutil
import time
from pathlib import Path
from typing import Any

import numpy as np
import torch
from safetensors.torch import load_file
from torch import nn
from transformers import AutoConfig, AutoModel, AutoTokenizer



def infer_adapter_max_layer(ckpt_dir: Path) -> int | None:
    """Returns 1-indexed max transformer layer modified by the LoRA adapter (e.g. 24 or 27)."""
    ckpt_dir = Path(ckpt_dir)
    for rel in (
        "adapter_model.safetensors",
        "adapter/adapter_model.safetensors",
        "adapter_model.bin",
        "adapter/adapter_model.bin",
        "adapter_model.pt",
        "adapter/adapter_model.pt",
    ):
        p = ckpt_dir / rel
        if p.exists():
            try:
                if p.suffix == ".safetensors":
                    sd = load_file(str(p))
                else:
                    sd = torch.load(str(p), map_location="cpu", weights_only=False)
                    if hasattr(sd, "state_dict") and callable(sd.state_dict):
                        sd = sd.state_dict()
                layer_nums = []
                for k in sd.keys():
                    m = re.search(r"layers\.(\d+)\.", str(k))
                    if m:
                        layer_nums.append(int(m.group(1)) + 1)
                if layer_nums:
                    return max(layer_nums)
            except Exception:
                continue
    return None


def in_place_copy(dst: torch.Tensor, src: torch.Tensor):
    """In-place weight copy."""
    dst.data.copy_(src.to(device=dst.device, dtype=dst.dtype))


def _resolve_module_name(k: str, state_dict: dict) -> str | None:
    """Resolves module name across various AutoModel and PEFT naming schemes."""
    raw = k.split(".lora_A")[0]
    candidates = [raw, raw.replace("base_model.model.", "")]
    if "layers." in raw:
        rel = raw[raw.index("layers.") :]
        candidates.extend([rel, f"model.{rel}"])
    for c in candidates:
        if c in state_dict and hasattr(state_dict[c], "weight"):
            return c
    return None


def merge_lora(model: nn.Module, adapter_path: Path, sign: float = +1.0) -> bool:
    """
    Atomically merges or unmerges (+1.0 or -1.0) a LoRA adapter into trunk weights.
    Pre-validates all tensor shapes BEFORE mutating any weights so an incompatible
    architecture (e.g. Llama vs Qwen) never partially corrupts the trunk.
    """
    weights_path = adapter_path / "adapter_model.safetensors"
    if not weights_path.exists():
        weights_path = adapter_path / "adapter" / "adapter_model.safetensors"
    if not weights_path.exists():
        return False

    try:
        sd = load_file(str(weights_path))
    except Exception as e:
        print(f"[merge_lora] Could not load {weights_path}: {e}")
        return False

    cfg_p = adapter_path / "adapter_config.json"
    if not cfg_p.exists():
        cfg_p = adapter_path / "adapter" / "adapter_config.json"
    scale = 1.0
    if cfg_p.exists():
        try:
            with open(cfg_p, "r", encoding="utf-8") as f:
                c = json.load(f)
            scale = float(c.get("lora_alpha", 16)) / float(c.get("r", 16))
        except Exception:
            scale = 1.0

    state_dict = dict(model.named_modules())
    pairs = {}
    for k in sd.keys():
        if "lora_A" in k:
            b_key = k.replace("lora_A", "lora_B")
            if b_key in sd:
                m_name = _resolve_module_name(k, state_dict)
                if m_name is not None:
                    pairs[m_name] = (sd[k], sd[b_key])

    if not pairs:
        return False

    # Pass 1: Pre-validate all shapes before modifying any weight in `model`
    for m_name, (A, B) in pairs.items():
        mod = state_dict.get(m_name)
        if mod is None or not hasattr(mod, "weight"):
            continue
        expected_shape = (B.shape[0], A.shape[1])
        if tuple(mod.weight.shape) != expected_shape:
            print(
                f"[merge_lora] Architecture mismatch on {m_name} in {adapter_path.name}: "
                f"trunk weight {tuple(mod.weight.shape)} != adapter delta {expected_shape}. Aborting merge."
            )
            return False

    # Pass 2: All shapes verified — apply in-place update
    applied = 0
    with torch.no_grad():
        for m_name, (A, B) in pairs.items():
            mod = state_dict.get(m_name)
            if mod is not None and hasattr(mod, "weight"):
                A_d = A.to(mod.weight.device, dtype=torch.float32)
                B_d = B.to(mod.weight.device, dtype=torch.float32)
                delta = (B_d @ A_d) * (scale * sign)
                mod.weight.data.add_(delta.to(mod.weight.dtype))
                applied += 1
    return applied > 0


@torch.inference_mode()
def forward_and_pool_batch(
    model: nn.Module,
    enc: dict,
    layers: list[int],
    norms: nn.ModuleList,
    dtype: torch.dtype = torch.float16,
    drop_first_token: bool = True,
    final_norm: nn.Module | None = None,
):
    """
    Runs forward pass in `dtype` (float16 on T4).
    CRITICAL: If any target layer overflows fp16 (Inf/NaN), recomputes ONLY the overflowed
    rows in `torch.bfloat16` and pools directly in float32 before restoring `dtype`!
    """
    hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states

    B = enc["input_ids"].shape[0]
    first_norm_dev = norms[0].weight.device

    # Check if any selected layer has Inf/NaN per row
    bad_mask = torch.zeros(B, dtype=torch.bool, device="cpu")
    for l in layers:
        bad_mask |= ~torch.isfinite(hs[l]).all(dim=(1, 2)).cpu()

    n_bad = int(bad_mask.sum().item())
    m = enc["attention_mask"].unsqueeze(-1).float().to(first_norm_dev)
    if drop_first_token:
        m[:, 0, :] = 0.0
    den = m.sum(dim=1).clamp(min=1.0)

    # Pool finite rows first in float32
    pooled_layers = [torch.zeros((B, model.config.hidden_size), dtype=torch.float32, device=first_norm_dev) for _ in layers]
    good_rows = torch.where(~bad_mask)[0].to(first_norm_dev)
    if len(good_rows) > 0:
        for idx_l, l in enumerate(layers):
            h_raw = hs[l][good_rows.to(hs[l].device)]
            if final_norm is not None and idx_l == len(layers) - 1:
                fn_params = list(final_norm.parameters())
                fn_dev = fn_params[0].device if fn_params else h_raw.device
                fn_dtype = fn_params[0].dtype if fn_params else h_raw.dtype
                h_raw = final_norm(h_raw.to(device=fn_dev, dtype=fn_dtype)).to(h_raw.device)
            h_good = h_raw.to(first_norm_dev).float()
            pooled_layers[idx_l][good_rows] = (h_good * m[good_rows]).sum(dim=1) / den[good_rows]

    # Rescue overflowed rows safely without mutating model parameters in-place
    # (In-place model.to(torch.bfloat16) permanently truncates FP16 mantissa bits when restoring).
    if n_bad > 0:
        bad_rows = torch.where(bad_mask)[0]
        bad_rows_dev = bad_rows.to(first_norm_dev)

        # Baseline fallback: sanitize existing raw activations so bad rows are never left as zeros
        for idx_l, l in enumerate(layers):
            h_fallback = hs[l][bad_rows.to(hs[l].device)]
            if final_norm is not None and idx_l == len(layers) - 1:
                fn_params = list(final_norm.parameters())
                fn_dev = fn_params[0].device if fn_params else h_fallback.device
                fn_dtype = fn_params[0].dtype if fn_params else h_fallback.dtype
                h_fallback = final_norm(h_fallback.to(device=fn_dev, dtype=fn_dtype)).to(h_fallback.device)
            h_fb = torch.nan_to_num(h_fallback.to(first_norm_dev).float(), nan=0.0, posinf=1e4, neginf=-1e4)
            pooled_layers[idx_l][bad_rows_dev] = (h_fb * m[bad_rows_dev]).sum(dim=1) / den[bad_rows_dev]

        del hs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        device_type = "cuda" if torch.cuda.is_available() else "cpu"
        emb_mod = model.get_input_embeddings() if hasattr(model, "get_input_embeddings") else None
        hook = None
        if emb_mod is not None:
            hook = emb_mod.register_forward_hook(lambda _m, _inp, out: out.float())

        try:
            enc_bad = {k: v[bad_rows] for k, v in enc.items()}
            with torch.autocast(device_type=device_type, dtype=torch.float32):
                hs_bad = model(**enc_bad, output_hidden_states=True, use_cache=False).hidden_states
            for idx_l, l in enumerate(layers):
                h_b = hs_bad[l]
                if final_norm is not None and idx_l == len(layers) - 1:
                    fn_params = list(final_norm.parameters())
                    fn_dev = fn_params[0].device if fn_params else h_b.device
                    fn_dtype = fn_params[0].dtype if fn_params else h_b.dtype
                    h_b = final_norm(h_b.to(device=fn_dev, dtype=fn_dtype)).to(h_b.device)
                h_b = h_b.to(first_norm_dev).float()
                h_b = torch.nan_to_num(h_b, nan=0.0, posinf=1e4, neginf=-1e4)
                pooled_layers[idx_l][bad_rows_dev] = (h_b * m[bad_rows_dev]).sum(dim=1) / den[bad_rows_dev]
            del hs_bad
        except Exception as e:
            print(f"[NeuralExtractor] Warning: Exception during overflow rescue, keeping sanitized fallback: {e}")
        finally:
            if hook is not None:
                hook.remove()

    out = torch.cat([nm(pooled_layers[idx_l]) for idx_l, nm in enumerate(norms)], dim=-1)
    return out.cpu().numpy(), n_bad


def _flatten_state_dict(obj: Any, prefix: str = "", out: dict | None = None) -> dict:
    if out is None:
        out = {}
    if hasattr(obj, "state_dict") and callable(obj.state_dict):
        obj = obj.state_dict()
    if isinstance(obj, dict):
        for k, v in obj.items():
            new_k = f"{prefix}.{k}" if prefix else str(k)
            if isinstance(v, (dict, list, tuple)) or (hasattr(v, "state_dict") and callable(v.state_dict)):
                _flatten_state_dict(v, new_k, out)
            else:
                out[new_k] = v
                if prefix in ("state_dict", "model", "heads", "head") and str(k) not in out:
                    out[str(k)] = v
    elif isinstance(obj, (list, tuple)):
        for idx, item in enumerate(obj):
            new_k = f"{prefix}.{idx}" if prefix else str(idx)
            _flatten_state_dict(item, new_k, out)
    return out


def load_head_file(path_or_dir: Path) -> dict:
    """Unified loader for classification head state dicts (.safetensors, .pt, .bin, .npz)."""
    p = Path(path_or_dir)
    res = {}
    if p.is_file():
        if p.suffix == ".safetensors":
            res = load_file(str(p))
        elif p.suffix in (".pt", ".bin"):
            raw = torch.load(str(p), map_location="cpu", weights_only=False)
            res = _flatten_state_dict(raw)
        elif p.suffix == ".npz":
            npz = np.load(str(p), allow_pickle=True)
            res = {k: npz[k] for k in npz.files}
        if res:
            print(f"[Head File] {p.name} loaded ({len(res)} keys): {list(res.keys())[:15]}")
        return res
    for fname in ("heads.safetensors", "head.safetensors", "heads.pt", "head.pt", "readouts.npz"):
        cand = p / fname
        if cand.exists():
            return load_head_file(cand)
    return {}


def resolve_head_tensors(
    heads_dict: dict, num_classes: int | None = None
) -> tuple[np.ndarray | None, np.ndarray | None, np.ndarray | None, np.ndarray | None]:
    """Resolves plain and centered head weights and biases matching American/British naming and handles transposed weights."""
    W_p, b_p, W_c, b_c = None, None, None, None
    valid_k = (num_classes,) if num_classes is not None else (K, 5, 4, 2)

    for wk, bk in [
        ("plain.weight", "plain.bias"),
        ("plain_coef", "plain_intercept"),
        ("head.weight", "head.bias"),
        ("classifier.weight", "classifier.bias"),
        ("fc.weight", "fc.bias"),
        ("W_plain", "b_plain"),
        ("weight", "bias"),
        ("W", "b"),
    ]:
        if wk in heads_dict and bk in heads_dict:
            W_p, b_p = heads_dict[wk], heads_dict[bk]
            break
        elif wk in heads_dict and bk not in heads_dict:
            W_p = heads_dict[wk]
            k_rows = (
                W_p.shape[0]
                if (hasattr(W_p, "shape") and len(W_p.shape) > 0 and W_p.shape[0] in valid_k)
                else (W_p.shape[1] if (hasattr(W_p, "shape") and len(W_p.shape) > 1 and W_p.shape[1] in valid_k) else (num_classes or K))
            )
            b_p = np.zeros(k_rows, dtype=np.float32)
            break

    for wk, bk in [
        ("centered.weight", "centered.bias"),
        ("centred.weight", "centred.bias"),
        ("center.weight", "center.bias"),
        ("twin.weight", "twin.bias"),
        ("center_coef", "center_intercept"),
        ("centred_coef", "centred_intercept"),
        ("W_center", "b_center"),
        ("W_twin", "b_twin"),
    ]:
        if wk in heads_dict and bk in heads_dict:
            W_c, b_c = heads_dict[wk], heads_dict[bk]
            break
        elif wk in heads_dict and bk not in heads_dict:
            W_c = heads_dict[wk]
            k_rows = (
                W_c.shape[0]
                if (hasattr(W_c, "shape") and len(W_c.shape) > 0 and W_c.shape[0] in valid_k)
                else (W_c.shape[1] if (hasattr(W_c, "shape") and len(W_c.shape) > 1 and W_c.shape[1] in valid_k) else (num_classes or K))
            )
            b_c = np.zeros(k_rows, dtype=np.float32)
            break

    # Suffix / Keyword Matching fallback if W_p or W_c still None
    if W_p is None or W_c is None:
        for k, v in heads_dict.items():
            k_lower = str(k).lower()
            shape = getattr(v, "shape", None)
            if shape is not None and len(shape) == 2 and (shape[0] in valid_k or shape[1] in valid_k):
                # Check for center / twin
                if any(tag in k_lower for tag in ("center", "centred", "twin")):
                    if W_c is None:
                        W_c = v
                        base_prefix = k.rsplit(".", 1)[0] if "." in k else k.rsplit("_", 1)[0]
                        for bk in [
                            f"{base_prefix}.bias",
                            f"{base_prefix}_bias",
                            f"{base_prefix}.intercept",
                            f"{base_prefix}_intercept",
                            "bias",
                            "centered.bias",
                            "b",
                        ]:
                            if bk in heads_dict and getattr(heads_dict[bk], "shape", None) is not None:
                                b_cand = heads_dict[bk]
                                if getattr(b_cand, "shape", ()) in [(k_item,) for k_item in valid_k]:
                                    b_c = b_cand
                                    break
                        if b_c is None:
                            k_rows = W_c.shape[0] if W_c.shape[0] in valid_k else W_c.shape[1]
                            b_c = np.zeros(k_rows, dtype=np.float32)
                # Check for plain / head / clf / classifier
                elif any(tag in k_lower for tag in ("plain", "head", "clf", "classifier")):
                    if W_p is None:
                        W_p = v
                        base_prefix = k.rsplit(".", 1)[0] if "." in k else k.rsplit("_", 1)[0]
                        for bk in [
                            f"{base_prefix}.bias",
                            f"{base_prefix}_bias",
                            f"{base_prefix}.intercept",
                            f"{base_prefix}_intercept",
                            "bias",
                            "plain.bias",
                            "b",
                        ]:
                            if bk in heads_dict and getattr(heads_dict[bk], "shape", None) is not None:
                                b_cand = heads_dict[bk]
                                if getattr(b_cand, "shape", ()) in [(k_item,) for k_item in valid_k]:
                                    b_p = b_cand
                                    break
                        if b_p is None:
                            k_rows = W_p.shape[0] if W_p.shape[0] in valid_k else W_p.shape[1]
                            b_p = np.zeros(k_rows, dtype=np.float32)

    if W_p is not None and hasattr(W_p, "cpu"):
        W_p = W_p.cpu().numpy()
    if b_p is not None and hasattr(b_p, "cpu"):
        b_p = b_p.cpu().numpy()
    if W_c is not None and hasattr(W_c, "cpu"):
        W_c = W_c.cpu().numpy()
    if b_c is not None and hasattr(b_c, "cpu"):
        b_c = b_c.cpu().numpy()

    # Transpose Guard: if shape is (D, K_cls), transpose to canonical (K_cls, D)
    if W_p is not None and W_p.ndim == 2:
        if W_p.shape[1] in valid_k and W_p.shape[0] not in valid_k:
            W_p = W_p.T
    if W_c is not None and W_c.ndim == 2:
        if W_c.shape[1] in valid_k and W_c.shape[0] not in valid_k:
            W_c = W_c.T

    return W_p, b_p, W_c, b_c


def compute_centered_logits(
    Z_test: np.ndarray,
    groups: np.ndarray,
    W_c: np.ndarray,
    b_c: np.ndarray,
    src_labels: list,
    max_std: float = 4.2,
) -> tuple[np.ndarray, np.ndarray]:
    """Computes prompt-topic centered twin logits with temperature normalization matching plain heads."""
    N = len(Z_test)
    out_lp = np.zeros((N, K), dtype=np.float64)
    if N < 2 or W_c is None:
        return out_lp, np.zeros(N, dtype=bool)
    _, inv_g = np.unique(np.asarray(groups, dtype=str), return_inverse=True)
    safe_g = inv_g + 1
    sz = np.bincount(safe_g)[safe_g]
    valid = (sz >= 2) & (sz <= 18)
    if not valid.any():
        return out_lp, valid
    Zc = Z_test.copy()
    for g in np.unique(safe_g[valid]):
        mask = safe_g == g
        Zc[mask] -= Zc[mask].mean(axis=0, keepdims=True)
    raw_c = align_cols(Zc[valid] @ W_c.T + b_c, src_labels)
    std_c = float(np.std(raw_c, axis=1).mean()) if len(raw_c) > 0 else 1.0
    temp_c = max(1.0, std_c / max_std)
    out_lp[valid] = log_softmax(raw_c / temp_c, axis=1)
    return out_lp, valid


def compute_family_centered_logits(
    Z_test: np.ndarray,
    groups: np.ndarray,
    W_c: np.ndarray,
    b_c: np.ndarray,
    src_labels: list,
    target_labels: list,
) -> tuple[np.ndarray, np.ndarray]:
    """Computes within-cluster centered twin logits for a K_fam-class family specialist."""
    N = len(Z_test)
    K_fam = len(target_labels)
    out_lp = np.zeros((N, K_fam), dtype=np.float64)
    if N < 2 or W_c is None:
        return out_lp, np.zeros(N, dtype=bool)
    _, inv_g = np.unique(np.asarray(groups, dtype=str), return_inverse=True)
    safe_g = inv_g + 1
    sz = np.bincount(safe_g)[safe_g]
    valid = (sz >= 2) & (sz <= 18)
    if not valid.any():
        return out_lp, valid
    Zc = Z_test.copy()
    for g in np.unique(safe_g[valid]):
        mask = safe_g == g
        Zc[mask] -= Zc[mask].mean(axis=0, keepdims=True)
    raw_c = Zc[valid] @ W_c.T + b_c
    src_list = [x.decode("utf-8") if isinstance(x, (bytes, np.bytes_)) else str(x) for x in src_labels]
    if len(src_list) == K_fam and set(src_list) == set(target_labels):
        perm = [src_list.index(l) for l in target_labels]
        raw_c = raw_c[:, perm]
    out_lp[valid] = log_softmax(raw_c, axis=1)
    return out_lp, valid


class UniversalLoRATrunkLoader:
    def __init__(self, base_dir: Path, keep_layers: int, layers: list[int], max_len: int = 447):
        if base_dir is not None and not (base_dir / "config.json").exists():
            for sub in base_dir.rglob("config.json"):
                base_dir = sub.parent
                break
        self.base_dir = base_dir
        self.keep_layers = keep_layers
        self.layers = layers
        self.max_len = max_len
        self.total_rescued = 0
        self.tok = AutoTokenizer.from_pretrained(str(base_dir), trust_remote_code=True)
        self.tok.padding_side = "right"
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token or "<|endoftext|>"

        cfg = AutoConfig.from_pretrained(str(base_dir), trust_remote_code=True)
        cfg.num_hidden_layers = keep_layers
        if getattr(cfg, "layer_types", None) is not None:
            cfg.layer_types = cfg.layer_types[:keep_layers]
        cfg.use_cache = False

        n_gpus = torch.cuda.device_count()
        if n_gpus >= 2:
            max_memory = {0: "13.5GiB", 1: "13.5GiB"}
            self.model = AutoModel.from_pretrained(
                str(base_dir),
                config=cfg,
                device_map="auto",
                max_memory=max_memory,
                torch_dtype=torch.float16,
                trust_remote_code=True,
                low_cpu_mem_usage=True,
            )
        elif n_gpus == 1:
            self.model = AutoModel.from_pretrained(
                str(base_dir), config=cfg, device_map="auto", torch_dtype=torch.float16, trust_remote_code=True, low_cpu_mem_usage=True
            )
        else:
            self.model = AutoModel.from_pretrained(str(base_dir), config=cfg, torch_dtype=torch.float32, low_cpu_mem_usage=True)

        # Preserve original final trunk norm before optionally disabling
        self.orig_final_norm = getattr(self.model, "norm", None) or getattr(getattr(self.model, "model", None), "norm", None)

        # Default: keep HuggingFace model.norm as nn.Identity() at all times
        if hasattr(self.model, "norm"):
            self.model.norm = nn.Identity()
        if hasattr(self.model, "model") and hasattr(self.model.model, "norm"):
            self.model.model.norm = nn.Identity()

        self.final_norm_enabled = False
        self.model.eval()
        self.device_norms = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

        # nn.LayerNorm with both weight and bias matching heads.safetensors
        hidden_dim = self.model.config.hidden_size
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in layers]).to(self.device_norms).float()

        self.norms.eval()
        self.first_dev = next(self.model.parameters()).device

    def set_final_norm(self, enabled: bool):
        """Sets final_norm_enabled flag, keeping HuggingFace model.norm as nn.Identity()."""
        self.final_norm_enabled = bool(enabled)

    def snapshot_weights(self, adapter_path: Path) -> dict:
        """Saves CPU fp16 copies of trunk modules modified by adapter."""
        weights_path = adapter_path / "adapter_model.safetensors"
        if not weights_path.exists():
            weights_path = adapter_path / "adapter" / "adapter_model.safetensors"
        if not weights_path.exists():
            return {}
        sd = load_file(str(weights_path))
        state_dict = dict(self.model.named_modules())
        backup = {}
        for k in sd.keys():
            if "lora_A" in k:
                m_name = _resolve_module_name(k, state_dict)
                if m_name is not None and m_name in state_dict and hasattr(state_dict[m_name], "weight"):
                    backup[m_name] = state_dict[m_name].weight.detach().cpu().half().clone()
        return backup

    def restore_weights(self, backup: dict):
        """Restores base trunk weights bit-for-bit without FP16 subtraction drift."""
        state_dict = dict(self.model.named_modules())
        with torch.no_grad():
            for m_name, orig in backup.items():
                if m_name in state_dict and hasattr(state_dict[m_name], "weight"):
                    state_dict[m_name].weight.data.copy_(orig.to(state_dict[m_name].weight.device))

    def set_active_layers(self, layers: list[int], max_len: int | None = None):
        """Updates active pooled layers and instantiates LayerNorms matching the checkpoint."""
        self.layers = list(layers)
        if max_len is not None:
            self.max_len = max_len
        hidden_dim = self.model.config.hidden_size
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in self.layers]).to(self.device_norms).float()
        self.norms.eval()

    def load_norm_weights(self, heads_sd: dict):
        """Loads LayerNorm weights and biases from head state dict after resetting to identity."""
        with torch.no_grad():
            # First reset every ln in self.norms to identity so weights/biases from previous checkpoint never bleed
            for ln in self.norms:
                ln.weight.data.fill_(1.0)
                if hasattr(ln, "bias") and ln.bias is not None:
                    ln.bias.data.zero_()

            # Support all norm key prefixes: norms.{i}, layer_norms.{i}, ln.{i}, norm.{i}
            for i, ln in enumerate(self.norms):
                for pfx in ("norms", "layer_norms", "ln", "norm"):
                    w_k = f"{pfx}.{i}.weight"
                    b_k = f"{pfx}.{i}.bias"
                    if w_k in heads_sd:
                        ln.weight.copy_(heads_sd[w_k].to(ln.weight.device, ln.weight.dtype))
                    elif (
                        f"{pfx}.{i}" in heads_sd
                        and hasattr(heads_sd[f"{pfx}.{i}"], "shape")
                        and heads_sd[f"{pfx}.{i}"].shape == ln.weight.shape
                    ):
                        ln.weight.copy_(heads_sd[f"{pfx}.{i}"].to(ln.weight.device, ln.weight.dtype))

                    if b_k in heads_sd and hasattr(ln, "bias") and ln.bias is not None:
                        ln.bias.copy_(heads_sd[b_k].to(ln.bias.device, ln.bias.dtype))

            # Check for stacked 2D tensor: norms_weight or norms.weight of shape (len(self.layers), H)
            for stk in ("norms_weight", "norms.weight", "layer_norms.weight", "ln.weight"):
                if stk in heads_sd and hasattr(heads_sd[stk], "ndim") and heads_sd[stk].ndim == 2:
                    stk_t = heads_sd[stk]
                    if stk_t.shape[0] == len(self.norms):
                        for i, ln in enumerate(self.norms):
                            ln.weight.copy_(stk_t[i].to(ln.weight.device, ln.weight.dtype))

    def _extract_batch_ids(self, id_list: list[list[int]], batch_size: int = 8) -> np.ndarray:
        order = np.argsort([len(x) for x in id_list], kind="stable")
        total_dim = len(self.layers) * self.model.config.hidden_size
        Z = np.zeros((len(id_list), total_dim), dtype=np.float32)

        for s in range(0, len(order), batch_size):
            idx = order[s : s + batch_size]
            enc = self.tok.pad({"input_ids": [id_list[i] for i in idx]}, padding=True, return_tensors="pt")
            enc = {k: v.to(self.first_dev) for k, v in enc.items()}
            fn = self.orig_final_norm if getattr(self, "final_norm_enabled", False) else None
            out_np, n_bad = forward_and_pool_batch(
                self.model, enc, self.layers, self.norms, torch.float16, drop_first_token=True, final_norm=fn
            )
            self.total_rescued += n_bad
            Z[idx] = out_np
        return Z

    @torch.inference_mode()
    def extract_exact(self, texts: list[str], batch_size: int = 8, end_id: int | None = 151643) -> np.ndarray:
        t0 = time.time()
        self.total_rescued = 0
        raw_ids = self.tok(list(texts), add_special_tokens=False, truncation=True, max_length=self.max_len)["input_ids"]
        if end_id is not None:
            ids = [x + [end_id] for x in raw_ids]
        else:
            ids = raw_ids
        Z = self._extract_batch_ids(ids, batch_size=batch_size)
        print(f"[Extractor] Extracted {len(texts)} samples in {time.time() - t0:.2f}s (Total BF16 Rescued Rows: {self.total_rescued}).")
        return Z

    def free_memory(self):
        del self.model, self.norms
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def read_checkpoint_meta(ckpt_dir: Path, default_keep: int, default_layers: list[int], default_max_len: int = 447) -> dict:
    """Reads keep, layers, max_len, end_id, and labels dynamically from meta.json or infers from adapter weights."""
    ckpt_dir = Path(ckpt_dir)
    meta_p = ckpt_dir / "meta.json"
    dir_str = str(ckpt_dir).lower().replace("\\", "/")
    is_dhruv = any(x in dir_str for x in ("q14v5", "ts_q3b14", "ts_q3b8", "q8v5", "ll8v5", "ts_llama8", "l24x"))

    labels = sorted(LABELS) if is_dhruv else list(LABELS)
    keep = 24 if is_dhruv else default_keep
    layers = [2, 4, 6, 8, 12, 16, 20, 24] if is_dhruv else list(default_layers)
    max_len = 384 if is_dhruv else default_max_len
    end_id = None if is_dhruv else 151643
    no_final_norm = False if is_dhruv else True

    if meta_p.exists():
        try:
            with open(meta_p, "r", encoding="utf-8") as f:
                m = json.load(f)
            labels = m.get("labels", m.get("classes", labels))
            cfg = m.get("cfg", m)
            keep = int(cfg.get("keep", keep))
            layers = list(cfg.get("layers", layers))
            max_len = int(cfg.get("max_len", max_len))
            if "no_final_norm" in cfg:
                no_final_norm = bool(cfg["no_final_norm"])
            for ek in ("end_token_id", "eos_id", "eos_token_id", "end_token"):
                if ek in cfg:
                    end_id = int(cfg[ek]) if isinstance(cfg[ek], int) else None
                    break
        except Exception:
            pass
    else:
        max_l = infer_adapter_max_layer(ckpt_dir)
        if max_l == 24:
            keep = 24
            layers = [2, 4, 6, 8, 12, 16, 20, 24]
        elif max_l == 27:
            keep = 27
            layers = [2, 4, 6, 9, 13, 17, 22, 27]
        elif max_l == 32:
            keep = 32
            layers = [3, 5, 8, 11, 16, 21, 27, 32]
        elif max_l is not None:
            keep = max_l

    return {"keep": keep, "layers": layers, "max_len": max_len, "end_id": end_id, "no_final_norm": no_final_norm, "labels": labels}


def extract_deberta_checkpoint(
    ckpt_dir: Path,
    base_deb_dir: Path | None,
    texts_test: list[str],
    texts_train: list[str],
    y_train: np.ndarray,
    max_len: int = 512,
    batch_size: int = 8,
) -> tuple[np.ndarray, np.ndarray, np.ndarray | None, np.ndarray | None, np.ndarray | None, np.ndarray | None, list]:
    """
    Dedicated feature extractor for DeBERTa-v3-large sequence classification checkpoints.
    Supports both single-layer encoder (Layer 24 float32 last_hidden_state) and multi-layer heads.
    """
    dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    heads_sd = load_head_file(ckpt_dir)
    W_p, b_p, W_c, b_c = resolve_head_tensors(heads_sd)

    # Load tokenizer from encoder directory if available, matching exact training setup
    if (ckpt_dir / "encoder").exists():
        tok_dir = str(ckpt_dir / "encoder")
    elif base_deb_dir and (base_deb_dir / "config.json").exists():
        tok_dir = str(base_deb_dir)
    else:
        tok_dir = str(ckpt_dir)
    tok = AutoTokenizer.from_pretrained(tok_dir, use_fast=True)

    meta = read_checkpoint_meta(ckpt_dir, default_keep=24, default_layers=[3, 6, 9, 12, 15, 18, 21, 24], default_max_len=max_len)
    ckpt_labels = meta["labels"]
    layers = meta["layers"]
    max_len = meta["max_len"]

    if (ckpt_dir / "encoder" / "config.json").exists():
        model_src = ckpt_dir / "encoder"
    elif (ckpt_dir / "config.json").exists() and ((ckpt_dir / "model.safetensors").exists() or (ckpt_dir / "pytorch_model.bin").exists()):
        model_src = ckpt_dir
    elif base_deb_dir and (base_deb_dir / "config.json").exists():
        model_src = base_deb_dir
    else:
        model_src = ckpt_dir

    stage_dir = Path("/kaggle/temp/deberta_p2_stage")
    staged = False
    load_src = model_src
    if Path("/kaggle/temp").exists() and os.access("/kaggle/temp", os.W_OK):
        try:
            if stage_dir.exists():
                shutil.rmtree(stage_dir, ignore_errors=True)
            shutil.copytree(model_src, stage_dir)
            load_src = stage_dir
            staged = True
        except Exception:
            load_src = model_src

    # CRITICAL: Always use torch.float32 for DeBERTa-v3 to avoid FP16 overflow!
    model = None
    try:
        model = AutoModel.from_pretrained(str(load_src), torch_dtype=torch.float32).to(dev)
    finally:
        if staged and stage_dir.exists():
            shutil.rmtree(stage_dir, ignore_errors=True)

    model.eval()

    # Merge LoRA adapter if present (e.g. deberta_inhouse)
    has_adapter = (
        (ckpt_dir / "adapter" / "adapter_model.safetensors").exists()
        or (ckpt_dir / "adapter" / "adapter_model.bin").exists()
        or (ckpt_dir / "adapter_model.safetensors").exists()
        or (ckpt_dir / "adapter_model.bin").exists()
    )
    if has_adapter:
        merged_deb = merge_lora(model, ckpt_dir, sign=+1.0)
        if merged_deb:
            print(f"[DeBERTa] Merged LoRA adapter from {ckpt_dir}")

    hidden_dim = model.config.hidden_size
    has_norm_keys = any(k.startswith("norms.") and k.endswith(".weight") for k in heads_sd)
    is_single_last_layer = W_p is not None and W_p.shape[1] == hidden_dim and not has_norm_keys

    if not is_single_last_layer:
        if W_p is not None and W_p.shape[1] > 0:
            expected_layers = W_p.shape[1] // hidden_dim
            if len(layers) != expected_layers:
                norm_keys = [k for k in heads_sd if k.startswith("norms.") and k.endswith(".weight")]
                if len(norm_keys) == expected_layers:
                    layers = layers[-expected_layers:] if len(layers) > expected_layers else list(range(expected_layers))
                elif len(layers) > expected_layers:
                    layers = layers[-expected_layers:]

        norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in layers]).to(dev).float()
        with torch.no_grad():
            for i, ln in enumerate(norms):
                w_k = f"norms.{i}.weight"
                b_k = f"norms.{i}.bias"
                if w_k in heads_sd:
                    ln.weight.copy_(heads_sd[w_k].to(dev, ln.weight.dtype))
                if b_k in heads_sd:
                    ln.bias.copy_(heads_sd[b_k].to(dev, ln.bias.dtype))
        norms.eval()
    else:
        norms = None

    def pool_texts(txt_list: list[str], drop_first: bool = False) -> np.ndarray:
        out_dim = hidden_dim if is_single_last_layer else (len(layers) * hidden_dim)
        if len(txt_list) == 0:
            return np.zeros((0, out_dim), dtype=np.float32)
        order = np.argsort([len(t) for t in txt_list], kind="stable")
        all_vecs = np.zeros((len(txt_list), out_dim), dtype=np.float32)
        for s in range(0, len(order), batch_size):
            idx = order[s : s + batch_size]
            chunk = [txt_list[i] for i in idx]
            enc = tok(chunk, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
            enc = {k: v.to(dev) for k, v in enc.items()}
            with torch.no_grad():
                if is_single_last_layer:
                    out_hs = model(**enc).last_hidden_state
                    m = enc["attention_mask"].unsqueeze(-1).float()
                    if drop_first:
                        m[:, 0, :] = 0.0
                    den = m.sum(dim=1).clamp(min=1.0)
                    pooled = (torch.nan_to_num(out_hs.float(), nan=0.0, posinf=1e4, neginf=-1e4) * m).sum(dim=1) / den
                    all_vecs[idx] = pooled.cpu().numpy()
                else:
                    hs = model(**enc, output_hidden_states=True).hidden_states
                    m = enc["attention_mask"].unsqueeze(-1).float()
                    if drop_first:
                        m[:, 0, :] = 0.0
                    den = m.sum(dim=1).clamp(min=1.0)
                    pooled = [((torch.nan_to_num(hs[l].float(), nan=0.0, posinf=1e4, neginf=-1e4) * m).sum(dim=1) / den) for l in layers]
                    out = torch.cat([norms[i](pooled[i]) for i in range(len(layers))], dim=-1)
                    all_vecs[idx] = out.cpu().numpy()
        return all_vecs

    # Probe drop_first_token on train.csv[:180]
    probe_n = min(180, len(texts_train))
    best_drop = False if is_single_last_layer else True
    if probe_n > 0 and W_p is not None:
        best_acc = -1.0
        for cand_drop in [False, True] if is_single_last_layer else [True, False]:
            z_pr = pool_texts(texts_train[:probe_n], drop_first=cand_drop)
            preds = align_cols(z_pr @ W_p.T + b_p, ckpt_labels).argmax(axis=1)
            acc = float((preds == y_train[:probe_n]).mean())
            if acc > best_acc:
                best_acc = acc
                best_drop = cand_drop
        print(f"[DeBERTa Extractor] Probed drop_first={best_drop} (accuracy = {best_acc:.4f})")

    out_dim = hidden_dim if is_single_last_layer else (len(layers) * hidden_dim)
    Z_test = pool_texts(texts_test, drop_first=best_drop)
    Z_train = pool_texts(texts_train, drop_first=best_drop) if len(texts_train) > 0 else np.zeros((0, out_dim), dtype=np.float32)

    if model is not None:
        del model
    if norms is not None:
        del norms
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return Z_test, Z_train, W_p, b_p, W_c, b_c, ckpt_labels


In [ ]:
# === MODULE: src/namethatllm/postprocess.py ===
"""
Postprocessing, Fusion & Calibration Engine
===========================================
This module handles all post-ensemble refinement stages:
1. Two-tier convex mixture (85% Main Stack + 15% Complementary Stack)
2. Mass-preserving Specialist Neural Gates (Claude head, Gemini LoRA, GLM LoRA, GPT LoRA)
3. Confidence-aware Within-Family Spectral Gates (dynamic margin tie-breakers)
4. Unified Handwriting & Orthographic Shield (discrete Bayesian tokenization rules + continuous density)
5. Cavity Mean-Field Cluster Uniqueness Repulsion (soft at-most-once constraint on prompt clusters)
6. Multiplicative Sinkhorn-Knopp Column Marginal Balancing (global class prior equalization)
7. Hierarchical Beta-Bernoulli TypoBayes Postprocessor
8. Final Kaggle submission generation (submission.parquet and submission.csv)
"""

import re
from pathlib import Path

import numpy as np
import pandas as pd


# =============================================================================
# 1. WITHIN-FAMILY CONFIDENCE GATES (SPECTRAL / TIE-BREAKER HEURISTICS)
# =============================================================================


def apply_all_family_confidence_gates(
    P_mix: np.ndarray,
    family_gates: dict[str, np.ndarray] | None,
    max_strength: float = 0.20,
    margin_thr: float = 0.45,
) -> np.ndarray:
    """
    Confidence-aware within-family and cross-family specialist reranker.

    Concept:
    --------
    When an ensemble predicts probabilities across 18 classes, sister models within
    the same family (e.g., Gemini 3.7 vs 3.8, or GLM-5.3 vs GLM-5.3-flash) can have
    extremely close probabilities (low margin). This function uses auxiliary gate
    logits (from spectral/TF-IDF classifiers) to break ties in low-margin cases.

    Key Invariant - Mass Conservation:
    ----------------------------------
    The total probability assigned to the family is strictly preserved!
    Only the internal relative distribution among the sisters is adjusted.
    Probabilities for classes outside the family are completely untouched.

    Dynamic Confidence Gating:
    --------------------------
    - Margin = (Top-1 prob within family) - (Top-2 prob within family).
    - If margin >= margin_thr: Model is already confident; gate strength = 0 (no intervention).
    - If margin < margin_thr: Low confidence; gate strength scales up to max_strength.

    Parameters:
    -----------
    P_mix : np.ndarray of shape (N, 18)
        Current blended prediction probabilities (summing to 1.0 per row).
    family_gates : dict mapping group_name -> log_probabilities of shape (N, K_group)
        Pre-computed log-probabilities from the auxiliary family classifiers.
    max_strength : float
        Maximum interpolation weight for the gate (0.0 to 1.0). If <= 0, gates are bypassed.
    margin_thr : float
        Confidence margin threshold above which the gate does not intervene.

    Returns:
    --------
    P_out : np.ndarray of shape (N, 18)
        Calibrated probabilities with sister-class confusions resolved.
    """
    # Early exit: If gates are disabled or weight is zero, return immediately
    if not family_gates or float(max_strength) <= 0.0:
        return P_mix

    P_out = P_mix.astype(np.float64).copy()

    for fam_name, gate_lp in family_gates.items():
        # Identify the column indices belonging to this family/specialist group
        cols = SPECIALIST_GROUPS.get(fam_name, FAMILIES.get(fam_name))
        if cols is None:
            continue
        if gate_lp is None or gate_lp.shape != (len(P_out), len(cols)):
            continue

        # Determine which samples are eligible for this gate:
        # Cross-family triads only apply when top-1 prediction is in the triad.
        # Single-family gates apply across all rows where that family has mass.
        if fam_name in ("frontier_and_fable", "kimi_gpt55_qwen", "glm_kimi_triad"):
            row_mask = np.isin(P_out.argmax(axis=1), cols)
            if not row_mask.any():
                continue
        elif fam_name == "gemini_37_38_pair":
            row_mask = np.isin(P_out.argmax(axis=1), FAMILIES.get("gemini", [5, 6, 7, 8]))
            if not row_mask.any():
                continue
        else:
            row_mask = np.ones(len(P_out), dtype=bool)

        # Hard-sister groups (e.g. Gemini 3.7 vs 3.8, GLM) use wider margin threshold
        eff_margin_thr = 0.65 if fam_name in ("gemini_37_38_pair", "glm", "glm_kimi_triad") else margin_thr
        eff_max_strength = (
            max(float(max_strength), 0.35) if fam_name in ("gemini_37_38_pair", "glm", "glm_kimi_triad") else float(max_strength)
        )

        # Step 1: Extract sub-probabilities and preserve total marginal mass
        sub_p = np.clip(P_out[np.ix_(row_mask, cols)], 1e-12, 1.0)
        tot_mass = sub_p.sum(axis=1, keepdims=True)
        base_rel_p = sub_p / tot_mass  # Relative distribution within the family (sums to 1.0)

        # Step 2: Compute margin between top-1 and top-2 candidates within family
        sp = np.sort(base_rel_p, axis=1)
        margin = sp[:, -1] - sp[:, -2]

        # Step 3: Compute dynamic intervention strength (higher when margin is small)
        strength = np.clip((eff_margin_thr - margin) / max(eff_margin_thr, 1e-6), 0.0, 1.0)[:, None] * eff_max_strength

        # Step 4: Blend base probabilities with gate probabilities in log-space
        base_lp = np.log(np.clip(base_rel_p, 1e-12, 1.0))
        gate_p = softmax(gate_lp[row_mask], axis=1)
        gate_lp_norm = np.log(np.clip(gate_p, 1e-12, 1.0))
        mixed_rel_p = softmax((1.0 - strength) * base_lp + strength * gate_lp_norm, axis=1)

        # Step 5: Restore original total family mass, leaving external classes unaffected
        P_out[np.ix_(row_mask, cols)] = tot_mass * mixed_rel_p

    # Final safety re-normalization to guarantee row sums == 1.0
    return P_out / P_out.sum(axis=1, keepdims=True).clip(min=1e-12)


# =============================================================================
# 2. PROMPT-CLUSTER UNIQUENESS REPULSION (CAVITY MEAN-FIELD)
# =============================================================================


def apply_cluster_uniqueness_repulsion(
    P_mix: np.ndarray,
    groups: np.ndarray | None,
    strength: float = 0.35,
    iters: int = 3,
) -> np.ndarray:
    """
    Enforces soft at-most-once author uniqueness within test prompt clusters.

    Domain Insight:
    ---------------
    In LLM benchmarking, multiple completions are generated by presenting the
    EXACT SAME PROMPT to different LLM models. Hence, when two or more test passages
    share an identical prompt (forming a 'prompt cluster'), competition rules imply
    that each completion came from a DIFFERENT model (at most 1 completion per model).

    Mathematical Formulation (Cavity Mean-Field):
    ---------------------------------------------
    For each text i in cluster g, the 'cavity field' for class k is the probability
    that no other text j in cluster g (j != i) was generated by class k:
        log P(other != k) = sum_{j in g, j != i} log(1 - q_{j, k})

    We update the unnormalized log-posterior:
        L_{i, k} = log(P0_{i, k}) + strength * sum_{j != i} log(1 - q_{j, k})
    where P0 is the original prediction prior, and q is the current cluster belief.
    Softmax is applied over k, iterating 3 times to reach a self-consistent equilibrium.

    Parameters:
    -----------
    P_mix : np.ndarray of shape (N, 18)
        Input prediction probabilities.
    groups : np.ndarray of shape (N,)
        Cluster IDs assigned to each test sample (e.g. from TF-IDF prompt clustering).
    strength : float
        Repulsion strength factor (typically 0.35).
    iters : int
        Number of mean-field coordinate ascent iterations (typically 3).

    Returns:
    --------
    q : np.ndarray of shape (N, 18)
        Repulsed probabilities penalizing multiple identical predictions in one prompt cluster.
    """
    N = len(P_mix)
    # Require at least 18 rows and valid cluster assignments
    if N < 18 or groups is None or len(groups) != N or strength <= 0.0:
        return P_mix

    # Map cluster identifiers to dense integer labels 1, 2, ...
    _, inv_g = np.unique(np.asarray(groups, dtype=str), return_inverse=True)
    safe_g = inv_g + 1
    sz = np.bincount(safe_g)[safe_g]

    # Only operate on multi-sample clusters between 2 and 18 texts (size <= 18 since there are 18 classes)
    valid_g = np.unique(safe_g[(sz >= 2) & (sz <= 18)])
    if len(valid_g) == 0:
        return P_mix

    log_P0 = np.log(np.clip(P_mix.astype(np.float64), 1e-12, 1.0))
    q = P_mix.astype(np.float64).copy()

    # Mean-field relaxation loop
    for _ in range(iters):
        L = log_P0.copy()
        for g in valid_g:
            m = safe_g == g
            q_g = q[m]  # Shape: (cluster_size, 18)

            # lo[j, k] = log(1 - q_{j, k}): log-probability that sample j did NOT pick class k
            lo = np.log(np.clip(1.0 - q_g, 1e-4, 1.0))

            # Cavity sum: sum of lo over all samples in cluster minus sample i's own term
            # Vectorized identity: sum_{j} lo[j] - lo[i] equals sum_{j != i} lo[j]
            cavity_penalty = lo.sum(axis=0, keepdims=True) - lo
            L[m] += float(strength) * cavity_penalty

        # Re-normalize across the 18 classes for next iteration
        q = softmax(L, axis=1)

    return q


# =============================================================================
# 3. TWO-TIER CONVEX ENSEMBLE BLENDING
# =============================================================================


def blend_main_and_complementary(
    LOGIT_MAIN: np.ndarray, COMP_LIST: list[tuple[float, np.ndarray]], main_ratio: float = 0.85, comp_ratio: float = 0.15
) -> tuple[np.ndarray, np.ndarray]:
    """
    Blends the Main Stack and Complementary Stack in probability space.

    Architecture:
    -------------
    - Main Stack (85% weight): High-capacity deep readers (Qwen3-14B Part 2,
      in-house champion LoRA, etc.) providing top semantic representations.
    - Complementary Stack (15% weight): Fast, diverse, non-neural or shallow readers
      (Style SVC 4-view, Lexical 5-view, DeBERTa, 8B patches) providing orthographic
      and stylistic diversity that regularizes neural hallucinations.

    Formula:
    --------
        R_main = softmax(LOGIT_MAIN)
        P_comp = sum(w_i * P_i) / sum(w_i)
        P_mix  = main_ratio * R_main + comp_ratio * P_comp

    Parameters:
    -----------
    LOGIT_MAIN : np.ndarray of shape (N, 18)
        Raw logit sum from the primary deep models.
    COMP_LIST : list of tuples (weight, probability_array)
        List of complementary models and their assigned ensemble weights.
    main_ratio : float
        Convex weight for Main Stack (default 0.85).
    comp_ratio : float
        Convex weight for Complementary Stack (default 0.15).

    Returns:
    --------
    P_mix : np.ndarray of shape (N, 18)
        Blended ensemble probabilities.
    R_main : np.ndarray of shape (N, 18)
        Main stack probabilities alone (used for routing/gating decisions).
    """
    R_main = softmax(LOGIT_MAIN, axis=1)
    if len(COMP_LIST) > 0:
        comp_weight_total = sum(w for w, _ in COMP_LIST)
        P_comp = sum(w * P_c for w, P_c in COMP_LIST) / max(1e-8, comp_weight_total)
        P_mix = main_ratio * R_main + comp_ratio * P_comp
    else:
        P_mix = R_main
    return P_mix, R_main


# =============================================================================
# 4. NEURAL SPECIALIST GATES (CLAUDE HEAD & FAMILY LORAS)
# =============================================================================


def apply_claude_specialist_gate(
    P_mix: np.ndarray,
    R_main: np.ndarray,
    claude_spec_logits: np.ndarray | None,
    claude_classes: list | None = None,
    labels: list = LABELS,
    claude_cols: list = FAMILIES["claude"],
    gate_weight: float = 0.15,
    weight: float | None = None,
) -> np.ndarray:
    """
    Refines passages predicted in the Claude family using a specialist head
    while strictly conserving total Claude family probability mass.
    """
    if claude_spec_logits is None:
        return P_mix
    if weight is not None:
        gate_weight = weight
    if float(gate_weight) <= 0.0:
        return P_mix

    P_out = P_mix.copy()
    P_claude_head = softmax(claude_spec_logits, axis=-1)

    if claude_classes is not None:
        src_c = [x.decode("utf-8") if isinstance(x, (bytes, np.bytes_)) else str(x) for x in claude_classes]
        if all(labels[c] in src_c for c in claude_cols):
            c_idx = [src_c.index(labels[c]) for c in claude_cols]
            P_claude_head = P_claude_head[:, c_idx]

    claude_mask = np.isin(R_main.argmax(axis=1), claude_cols)
    if claude_mask.any():
        rows = np.flatnonzero(claude_mask)
        sub_mix = P_out[np.ix_(rows, claude_cols)]
        tot_mass = sub_mix.sum(axis=1, keepdims=True)
        blended = (1.0 - gate_weight) * sub_mix + gate_weight * (tot_mass * P_claude_head[rows])
        P_out[np.ix_(rows, claude_cols)] = blended

    return P_out


def apply_family_specialist_lora_gate(
    P_mix: np.ndarray,
    R_main: np.ndarray,
    spec_logits: np.ndarray | None,
    spec_classes: list | None = None,
    labels: list = LABELS,
    family_cols: list = FAMILIES["gemini"],
    gate_weight: float = 0.65,
) -> np.ndarray:
    """
    Refines intra-family predictions using fine-tuned Qwen3-14B Family-Specialist LoRAs
    while strictly preserving 100% of total marginal family probability mass.
    """
    if spec_logits is None or len(spec_logits) != len(P_mix) or float(gate_weight) <= 0.0:
        return P_mix

    P_out = P_mix.astype(np.float64).copy()
    spec_arr = np.asarray(spec_logits, dtype=np.float64)
    if spec_arr.ndim != 2 or spec_arr.shape[1] != len(family_cols):
        return P_mix

    P_spec = softmax(spec_arr, axis=-1)

    if spec_classes is not None and len(spec_classes) == len(family_cols):
        src_c = [x.decode("utf-8") if isinstance(x, (bytes, np.bytes_)) else str(x) for x in spec_classes]
        if all(labels[c] in src_c for c in family_cols):
            c_idx = [src_c.index(labels[c]) for c in family_cols]
            P_spec = P_spec[:, c_idx]

    fam_mass = P_out[:, family_cols].sum(axis=1)
    has_signal = np.ptp(spec_arr, axis=1) > 1e-6
    fam_mask = (np.isin(R_main.argmax(axis=1), family_cols) | (fam_mass >= 0.35)) & has_signal

    if fam_mask.any():
        rows = np.flatnonzero(fam_mask)
        sub_mix = np.clip(P_out[np.ix_(rows, family_cols)], 1e-12, 1.0)
        tot_mass = sub_mix.sum(axis=1, keepdims=True)
        base_rel = sub_mix / tot_mass

        log_rel = (1.0 - float(gate_weight)) * np.log(base_rel) + float(gate_weight) * np.log(np.clip(P_spec[rows], 1e-12, 1.0))
        blended_rel = softmax(log_rel, axis=1)
        P_out[np.ix_(rows, family_cols)] = tot_mass * blended_rel

    return P_out


# =============================================================================
# 5. ORTHOGRAPHIC RULES & HANDWRITING SHIELD
# =============================================================================


def build_orthographic_penalty_matrix(
    curly_quote_log_penalty: float = -1.00,  # Rule 1: Claude & GLM never emit curly quotes
    multi_em_dash_log_penalty: float = 0.0,  # Rule 2: Multiple prose em-dashes (rare in Gemini)
    straight_contr_log_penalty: float = 0.0,  # Rule 3: Straight contractions
    curly_contr_log_penalty: float = 0.0,  # Rule 4: Curly contractions
) -> np.ndarray:
    """
    Builds the (4, 18) log-penalty matrix W_pen mapping discrete orthographic
    rule indicators to the 18 canonical model classes.

    Empirical Foundations:
    ----------------------
    - Rule 1 (Curly Quotes): In the competition training corpus, Claude (0 / 1049)
      and GLM (0 / 300) NEVER produce typographic curly quotes (‘, ’, “, ”) in standard prose.
      They consistently output straight quotes (', "). If a passage contains curly quotes,
      the probability of it being Claude or GLM is astronomically low.
      Log penalty of -1.0 reduces their odds by a factor of e^(-1) ≈ 0.368.
    - Rule 2 (Multiple Em-Dashes): Gemini models rarely use two or more prose em-dashes (—).
    - Rule 3 (Straight Contractions): Distinctive signature for gpt-5.6-sol vs sisters.
    - Rule 4 (Curly Contractions): Reinforces rejection of Claude & GLM.
    """
    W_pen = np.zeros((4, K), dtype=np.float64)
    claude_and_glm_idx = FAMILIES["claude"] + FAMILIES["glm"]
    gemini_idx = FAMILIES["gemini"]
    sol_idx = LAB2I["gpt-5.6-sol"]

    # Rule 1: Any curly quote -> penalize Claude (5 models) and GLM (2 models)
    W_pen[0, claude_and_glm_idx] = float(curly_quote_log_penalty)
    # Rule 2: >= 2 prose em-dashes -> penalize Gemini (4 models)
    W_pen[1, gemini_idx] = float(multi_em_dash_log_penalty)
    # Rule 3: >= 2 prose straight contractions (with 0 curly) -> penalize gpt-5.6-sol
    W_pen[2, sol_idx] = float(straight_contr_log_penalty)
    # Rule 4: >= 1 prose curly contraction -> reinforce penalty on Claude & GLM
    W_pen[3, claude_and_glm_idx] = float(curly_contr_log_penalty)

    return W_pen


def _extract_rules_fallback(texts: list[str]) -> np.ndarray:
    """
    Fallback discrete rules extractor using regex when lexical_engine is unavailable.
    Detects curly quotes, em-dashes, and straight/curly prose contractions.
    """
    re_code = re.compile(r"```[\s\S]*?```|`[^`\n]+`")
    re_str = re.compile(r"[A-Za-z]'[A-Za-z]")
    re_cur = re.compile(r"[A-Za-z]\u2019[A-Za-z]")
    rules = np.zeros((len(texts), 4), dtype=np.float32)

    for i, raw_t in enumerate(texts):
        t = str(raw_t)
        prose = re_code.sub(" ", t)  # Strip inline and block code before checking prose
        raw_curly = t.count("\u2018") + t.count("\u2019") + t.count("\u201c") + t.count("\u201d")
        rules[i, 0] = 1.0 if raw_curly > 0 else 0.0
        rules[i, 1] = 1.0 if prose.count("\u2014") >= 2 else 0.0
        rules[i, 2] = 1.0 if (len(re_str.findall(prose)) >= 2 and (t.count("\u2018") + t.count("\u2019")) == 0) else 0.0
        rules[i, 3] = 1.0 if len(re_cur.findall(prose)) >= 1 else 0.0

    return rules


def apply_unified_handwriting_shield(
    P_mix: np.ndarray,
    test_texts: list[str],
    log_p_handwriting: np.ndarray | None = None,
    rule_flags: np.ndarray | None = None,
    curly_quote_multiplier: float = 0.36787944117144233,  # exp(-1.0)
    multi_em_dash_log_penalty: float = 0.0,
    straight_contr_log_penalty: float = 0.0,
    curly_contr_log_penalty: float = 0.0,
    continuous_hw_weight: float = 0.0,
) -> np.ndarray:
    """
    Single unified point of application for ALL handwriting and orthographic rules:
    1. Computes vectorized discrete log-penalties via matrix product: rule_flags @ W_pen
    2. Adds continuous 26-feature handwriting Gaussian log-likelihoods
    3. Re-normalizes rows to the probability simplex via softmax.
    """
    if rule_flags is None:
        if "extract_handwriting_and_orthographic_features" in globals():
            _, rule_flags = extract_handwriting_and_orthographic_features(test_texts)
        else:
            rule_flags = _extract_rules_fallback(test_texts)

    curly_log_pen = float(np.log(max(curly_quote_multiplier, 1e-12)))
    W_pen = build_orthographic_penalty_matrix(
        curly_quote_log_penalty=curly_log_pen,
        multi_em_dash_log_penalty=multi_em_dash_log_penalty,
        straight_contr_log_penalty=straight_contr_log_penalty,
        curly_contr_log_penalty=curly_contr_log_penalty,
    )

    log_P = np.log(np.clip(P_mix.astype(np.float64), 1e-12, 1.0))

    # 1. Discrete Bayesian rule subtraction (vectorized matrix multiply)
    log_P += rule_flags.astype(np.float64) @ W_pen

    # 2. Continuous 26-feature handwriting log-likelihood contribution
    if log_p_handwriting is not None and continuous_hw_weight > 0.0:
        log_P += float(continuous_hw_weight) * log_p_handwriting.astype(np.float64)

    return softmax(log_P, axis=1)


def apply_curly_quote_shield(P_mix: np.ndarray, test_texts: list[str], multiplier: float = 0.36787944117144233) -> np.ndarray:
    """Backward-compatible wrapper invoking the unified handwriting shield."""
    return apply_unified_handwriting_shield(
        P_mix=P_mix,
        test_texts=test_texts,
        log_p_handwriting=None,
        rule_flags=None,
        curly_quote_multiplier=multiplier,
        multi_em_dash_log_penalty=0.0,
        straight_contr_log_penalty=0.0,
        curly_contr_log_penalty=0.0,
        continuous_hw_weight=0.0,
    )


# =============================================================================
# 6. SINKHORN-KNOPP COLUMN MASS BALANCING
# =============================================================================


def sinkhorn_balancing(
    P: np.ndarray,
    iters: int = 100,
    sinkhorn_strength: float = 0.65,
    target_mix: float = 0.35,
    train_prior: np.ndarray | None = None,
    allow_pure: bool = False,
) -> np.ndarray:
    """
    Soft Multiplicative Sinkhorn-Knopp Column Marginal Balancing.
    =============================================================
    Equalizes global class marginals using Soft Sinkhorn shrinkage.
    Pure Sinkhorn (1.0 / 1.0) is disallowed by default as it destroys test performance
    by hard-forcing uniform marginals on batches with natural class variance.

    Parameters:
    -----------
    P : np.ndarray of shape (N, 18)
        Probability matrix to balance.
    iters : int
        Number of alternating row/column scaling steps (default 100).
    sinkhorn_strength : float
        Soft shrinkage interpolation between original P and balanced Q (default 0.65).
        Capped at 0.75 unless allow_pure=True.
    target_mix : float
        Convex blend between predicted marginals and prior (default 0.35).
        Capped at 0.50 unless allow_pure=True.
    train_prior : np.ndarray of shape (18,), optional
        Empirical training class frequencies (used when target_mix < 1.0).
    allow_pure : bool, optional
        Whether to allow 100% hard pure Sinkhorn (default False).

    Returns:
    --------
    Balanced probability matrix Q of shape (N, 18).
    """
    if len(P) < 18 or sinkhorn_strength <= 0.0:
        return P

    P_work = np.clip(P.astype(np.float64), 1e-12, 1.0)
    K_cols = P_work.shape[1]

    # Target column marginal
    if train_prior is not None:
        p_train = np.clip(train_prior.astype(np.float64), 1e-6, None)
        p_train /= p_train.sum()
    else:
        p_train = np.full(K_cols, 1.0 / K_cols, dtype=np.float64)

    eff_mix = target_mix if allow_pure else min(target_mix, 0.50)
    if eff_mix < 1.0:
        pred_marg = np.clip(P_work.mean(axis=0), 1e-6, None)
        pred_marg /= pred_marg.sum()
        rho = float(np.clip(eff_mix, 0.0, 1.0))
        target_col = (1.0 - rho) * pred_marg + rho * p_train
        target_col = np.clip(target_col, 1e-6, None)
        target_col /= target_col.sum()
    else:
        target_col = p_train

    # Alternating row and column projections
    Q = P_work.copy()
    for _ in range(iters):
        # Row normalization: sum across classes = 1.0
        Q /= Q.sum(axis=1, keepdims=True).clip(min=1e-12)
        # Column scaling: sum across samples for class k = N * target_col[k]
        Q *= (len(Q) * target_col[None, :]) / Q.sum(axis=0, keepdims=True).clip(min=1e-12)

    # Final row normalization
    Q /= Q.sum(axis=1, keepdims=True).clip(min=1e-12)

    if allow_pure and sinkhorn_strength >= 1.0:
        return Q

    # Soft logarithmic mixture between original P and balanced Q (pure Sinkhorn disallowed)
    eff_strength = float(np.clip(sinkhorn_strength, 0.0, 0.75 if not allow_pure else 1.0))
    if eff_strength <= 0.0:
        return P_work

    log_P = (1.0 - eff_strength) * np.log(np.clip(P_work, 1e-12, None)) + eff_strength * np.log(np.clip(Q, 1e-12, None))
    return softmax(log_P, axis=1)


def apply_prior_and_sinkhorn(
    logits: np.ndarray,
    temperature: float = 1.5,
    iters: int = 80,
    sinkhorn_strength: float = 0.85,
    target_mix: float = 0.15,
) -> np.ndarray:
    """
    Applies per-sample standardization, iterative column-wise logit prior centering (20 steps),
    and Sinkhorn marginal probability balancing to ensure unbiased class priors.
    """
    if len(logits) < 18 or sinkhorn_strength <= 0.0:
        return softmax(logits, axis=1)

    row_mean = logits.mean(axis=1, keepdims=True)
    row_std = np.clip(logits.std(axis=1, keepdims=True), 1e-4, None)
    norm_logits = (logits - row_mean) / row_std

    adj_logits = norm_logits / max(temperature, 1e-3)
    num_classes = logits.shape[1]
    for _ in range(20):
        probs = softmax(adj_logits, axis=1)
        col_mean = np.clip(probs.mean(axis=0, keepdims=True), 1e-6, None)
        adj_logits = adj_logits - 0.5 * np.log(col_mean * num_classes)

    return sinkhorn_balancing(
        softmax(adj_logits, axis=1),
        iters=iters,
        sinkhorn_strength=sinkhorn_strength,
        target_mix=target_mix,
    )


# =============================================================================
# 7. TYPOBAYES POSTPROCESSOR (HIERARCHICAL BETA-BERNOULLI SHRINKAGE)
# =============================================================================


def apply_typobayes_postprocess(
    P_or_logP: np.ndarray,
    train_texts: list[str],
    y_train: np.ndarray,
    test_texts: list[str],
    labels: list = LABELS,
    kappa: float = 24.0,
    evidence_scale: float = 1.2,
    evidence_cap: float = 0.30,
) -> np.ndarray:
    """
    Applies Hierarchical Beta-Bernoulli 37-Marker TypoBayes postprocessing.

    Concept:
    --------
    Extracts 37 orthographic, punctuation, and structural markdown markers
    (e.g., ' — ', '`', '###', 'However,', '\n- ') across training texts and models.
    Uses Beta-Bernoulli empirical Bayes with prior strength kappa to calculate
    smoothed class-conditional emission rates, deriving log-likelihood ratio evidence.
    Applies conservative confidence gating so high-confidence predictions are not overridden.
    """
    if len(train_texts) == 0 or len(test_texts) == 0 or len(y_train) == 0:
        return P_or_logP

    y_tr = np.array([LAB2I[y] if isinstance(y, str) else int(y) for y in y_train], dtype=int)
    marks = [
        " \u2014 ",
        "\u2014",
        "\u2013",
        "\u2019",
        "\u2018",
        "\u201c",
        "\u201d",
        "\u00ab",
        "\u00bb",
        '"',
        ";",
        ":",
        "(",
        ")",
        "[",
        "]",
        "**",
        "`",
        "\u2026",
        "\u2022",
        "\n",
        "\n\n",
        "e.g.",
        "i.e.",
        "However,",
        "Overall,",
        "In conclusion",
        "Therefore,",
        "Moreover,",
        "\t",
        " - ",
        " = ",
        "\n- ",
        "\n1.",
        "###",
        "```",
        "> ",
    ]
    K_cls = len(labels)

    def _extract_marker_row(t: str) -> list[float]:
        # Mutually exclusive decoupling for overlapping substrings:
        # 1. ' — ' vs unspaced '—'
        has_spaced_em = " \u2014 " in t
        t_no_spaced = t.replace(" \u2014 ", "")
        has_unspaced_em = "\u2014" in t_no_spaced

        # 2. '\n\n', '\n- ', '\n1.' vs standalone '\n'
        has_double_nl = "\n\n" in t
        has_bullet = "\n- " in t
        has_num_list = "\n1." in t
        t_no_compound_nl = t.replace("\n\n", "").replace("\n- ", "").replace("\n1.", "")
        has_single_nl = "\n" in t_no_compound_nl

        row = []
        for m in marks:
            if m == " \u2014 ":
                row.append(float(has_spaced_em))
            elif m == "\u2014":
                row.append(float(has_unspaced_em))
            elif m == "\n\n":
                row.append(float(has_double_nl))
            elif m == "\n- ":
                row.append(float(has_bullet))
            elif m == "\n1.":
                row.append(float(has_num_list))
            elif m == "\n":
                row.append(float(has_single_nl))
            else:
                row.append(float(m in t))
        return row

    Xtr = np.array([_extract_marker_row(t) for t in train_texts], dtype=np.float32)
    Xte = np.array([_extract_marker_row(t) for t in test_texts], dtype=np.float32)
    freq = np.bincount(y_tr, minlength=K_cls).astype(np.float64)
    hits = np.stack([Xtr[y_tr == k].sum(axis=0) for k in range(K_cls)], axis=0).astype(np.float64)

    # Hierarchical Beta-Bernoulli smoothing: class rates shrink toward global rate
    global_rate = (Xtr.sum(axis=0) + 2.0) / (len(Xtr) + 4.0)
    rate = (hits + kappa * global_rate[None, :]) / (freq[:, None] + kappa)
    rate = np.clip(rate, 0.01, 0.99)
    global_rate = np.clip(global_rate, 0.01, 0.99)

    log_rate = np.log(rate)
    log_not_rate = np.log1p(-rate)
    base_log_rate = np.log(global_rate)
    base_log_not_rate = np.log1p(-global_rate)

    # Weight markers by discriminative power across classes
    signal = np.sqrt(np.mean((rate - global_rate[None, :]) ** 2, axis=0))
    med_signal = float(np.median(signal[signal > 0])) if np.any(signal > 0) else 1.0
    fw = signal / max(med_signal, 1e-6)
    fw = np.clip(fw, 0.25, 2.0)

    # Matrix form: weighted Bernoulli log-likelihood ratio for each class
    evidence = (Xte * fw[None, :]) @ log_rate.T + ((1.0 - Xte) * fw[None, :]) @ log_not_rate.T
    evidence -= (((Xte * fw[None, :]) @ base_log_rate) + (((1.0 - Xte) * fw[None, :]) @ base_log_not_rate))[:, None]
    evidence -= evidence.mean(axis=1, keepdims=True)
    evidence = np.clip(evidence_scale * evidence / max(float(fw.sum()), 1e-6), -evidence_cap, evidence_cap)

    # Convert inputs to probabilities and log-probabilities
    if np.all(P_or_logP <= 0.0) and not np.all(P_or_logP == 0.0):
        raw_logP = P_or_logP.copy()
        probs = softmax(raw_logP, axis=1)
    else:
        probs = P_or_logP.copy()
        raw_logP = np.log(np.clip(probs, 1e-12, 1.0))

    # Gating: only apply evidence when model is uncertain, text is long, and markers are present
    sorted_p = np.sort(probs, axis=1)
    margin = sorted_p[:, -1] - sorted_p[:, -2]
    confidence_gate = np.clip((0.60 - margin) / 0.60, 0.0, 1.0)
    word_count = np.array([len(re.findall(r"\b\w+\b", str(t))) for t in test_texts], dtype=np.float32)
    length_gate = np.clip(np.sqrt(np.maximum(word_count, 1.0) / 80.0), 0.40, 1.0)
    active_markers = Xte.sum(axis=1)
    coverage_gate = np.clip(np.sqrt((active_markers + 1.0) / 4.0), 0.45, 1.0)
    gate = confidence_gate * length_gate * coverage_gate

    safe_logits = raw_logP + gate[:, None] * evidence
    return softmax(safe_logits, axis=1)


# =============================================================================
# 8. FINAL SUBMISSION EXPORT
# =============================================================================


def generate_submission(P_final: np.ndarray, test_ids: pd.Series, working_dir: Path) -> tuple[pd.DataFrame, Path, Path]:
    """
    Generates submission.parquet and submission.csv with exact Kaggle competition schema:
    - id: test sample identifier string
    - model: predicted model class name string (argmax over calibrated probabilities)
    """
    preds = [LABELS[idx] for idx in P_final.argmax(axis=1)]
    df_sub = pd.DataFrame({"id": test_ids, "model": preds})
    sub_parquet_path = working_dir / "submission.parquet"
    sub_csv_path = working_dir / "submission.csv"

    df_sub.to_parquet(sub_parquet_path, index=False)
    df_sub.to_csv(sub_csv_path, index=False)
    return df_sub, sub_parquet_path, sub_csv_path


In [ ]:
# === MODULE: src/namethatllm/ablation.py ===
"""
Zero-Leakage Unseen Holdout Benchmark & Forward Stepwise Ablation Engine
========================================================================
Constructs a strictly unseen evaluation dataset from mounted parquet/CSV corpora
by subtracting all passages present in `train.csv` and `pool_v4.parquet`
(the training sets of the pre-trained Stage-1/Stage-2 checkpoints), quarantines
those holdout passages from `lexical_engine.py` and `trainer.py`, and evaluates:
1. Standalone (Solo) Unseen Accuracy of each model.
2. Forward Stepwise Cumulative Accuracy (1 model -> +2nd model -> ... -> Full Ensemble).
"""

from itertools import combinations
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import balanced_accuracy_score


if "_sinkhorn_balancing" not in globals() and "sinkhorn_balancing" in globals():
    _sinkhorn_balancing = globals()["sinkhorn_balancing"]

# =============================================================================
# HELPER FUNCTIONS: MASS BALANCING & ZERO-LEAKAGE TEXT HASHING
# =============================================================================


def _ablation_sinkhorn(
    P: np.ndarray,
    iters: int = 100,
    eps: float = 1e-12,
    y_true: np.ndarray | None = None,
    allow_pure: bool = False,
) -> np.ndarray:
    """
    Fast vectorized Sinkhorn-Knopp column mass balancing for evaluation.
    Delegates directly to sinkhorn_balancing in namethatllm.postprocess.
    """
    train_prior = np.bincount(y_true, minlength=P.shape[1]).astype(np.float64) if y_true is not None and len(y_true) == len(P) else None
    return _sinkhorn_balancing(
        P,
        iters=iters,
        sinkhorn_strength=1.0 if allow_pure else 0.65,
        target_mix=1.0 if allow_pure else 0.35,
        train_prior=train_prior,
        allow_pure=allow_pure,
    )


def _fast_sinkhorn_bacc(lp_mat: np.ndarray, y_true: np.ndarray, iters: int = 40) -> float:
    """
    Quickly computes balanced accuracy of log-probabilities after 40 Sinkhorn iterations.
    Used inside inner optimization loops (line search, greedy selection) for speed.
    """
    P = softmax(lp_mat, axis=1)
    if len(P) >= 18:
        P = _ablation_sinkhorn(P, iters=iters, y_true=y_true)
    return float(balanced_accuracy_score(y_true, P.argmax(axis=1)))


def build_unseen_holdout_dataset(
    input_dir: Path,
    train_path: Path | None = None,
    samples_per_class: int = 20,
    seed: int = 42,
) -> tuple[pd.DataFrame, set[str]]:
    """
    Builds a balanced 18-class unseen holdout dataset (`df_unseen`) by:
    1. Hashing all texts in `train.csv` and `pool_v4.parquet` (and `pool_v1.parquet`) as `contaminated_hashes`.
    2. Scanning `nlm_matched_clean.parquet`, `nlm_pool_v5.parquet`, and `openrouter-harvest`
       for passages whose text hash is strictly NOT in `contaminated_hashes`.
    3. Sampling up to `samples_per_class` passages per class (e.g., 18 * 20 = 360 unseen passages)
       and returning `(df_unseen, holdout_hashes)` so `lexical_engine.py` and `trainer.py`
       exclude `holdout_hashes` from training!
    """
    input_dir = Path(input_dir) if input_dir else Path(".")
    contaminated_hashes: set[str] = set()

    # 1. Mark all train.csv rows as contaminated (seen by all Stage-2 LoRAs)
    if train_path and Path(train_path).exists():
        try:
            df_tr = pd.read_csv(train_path, usecols=["text"])
            contaminated_hashes.update(_text_hash(t) for t in df_tr["text"].fillna(""))
        except Exception:
            try:
                df_tr = pd.read_csv(train_path)
                if "text" in df_tr.columns:
                    contaminated_hashes.update(_text_hash(t) for t in df_tr["text"].fillna(""))
            except Exception:
                pass

    # 2. Mark pool_v4.parquet and pool_v1.parquet as contaminated
    # (seen by Part-2 LoRAs, DeBERTa; pool_120k is the superset pool and excluded from candidates via pattern skip)
    for pat in ["pool_v4.parquet", "pool_v1.parquet"]:
        for p in input_dir.rglob(pat):
            try:
                df_p = pd.read_parquet(p, columns=["text"])
                contaminated_hashes.update(_text_hash(t) for t in df_p["text"].fillna(""))
            except Exception:
                pass

    # 3. Collect candidate passages in canonical trainer quarantine order
    candidate_dfs = []
    seen_paths = set()

    def _process_candidate_file(p: Path) -> pd.DataFrame | None:
        try:
            df_c = pd.read_parquet(p) if p.suffix == ".parquet" else pd.read_csv(p)
            if "label" in df_c.columns and "model" not in df_c.columns:
                df_c = df_c.rename(columns={"label": "model"})
            finish_col = next((c for c in ["finish_reason", "finish"] if c in df_c.columns), None)
            if finish_col:
                df_c = df_c[df_c[finish_col].astype(str).str.lower().eq("stop")]
                df_c = df_c[df_c["text"].fillna("").astype(str).str.split().str.len().between(25, 360)]
            if "text" in df_c.columns and "model" in df_c.columns:
                keep_cols = [c for c in ["id", "prompt_id", "text", "model"] if c in df_c.columns]
                df_c = df_c[keep_cols].dropna(subset=["text", "model"]).copy()
                df_c = df_c[df_c["model"].isin(LABELS)]
                df_c["_hash"] = [_text_hash(t) for t in df_c["text"]]
                df_c = df_c[~df_c["_hash"].isin(contaminated_hashes)]
                if len(df_c) > 0:
                    return df_c
        except Exception:
            pass
        return None

    candidate_patterns = [
        "nlm_matched_clean.parquet",
        "nlm_pool_v5.parquet",
        "nlm_pool_v2.parquet",
        "*openrouter*.parquet",
        "*openrouter*.csv",
        "*harvest*.parquet",
        "*harvest*.csv",
    ]
    for pat in candidate_patterns:
        for p in input_dir.rglob(pat):
            if p in seen_paths:
                continue
            p_name = p.name.lower()
            if any(skip in p_name for skip in ["train", "test", "submission", "pool_120k", "pool_v4", "pool_v1"]):
                continue
            seen_paths.add(p)
            df_c = _process_candidate_file(p)
            if df_c is not None:
                candidate_dfs.append(df_c)

    # Also scan any directory or file matching openrouter, harvest, passages, generations
    for ext in ["*.parquet", "*.csv"]:
        for p in input_dir.rglob(ext):
            if p in seen_paths:
                continue
            p_str = str(p).lower().replace("\\", "/")
            p_name = p.name.lower()
            if any(skip in p_name for skip in ["train", "test", "submission", "pool_120k", "pool_v4", "pool_v1"]):
                continue
            if any(k in p_str for k in ["openrouter", "harvest", "passages", "generations"]):
                seen_paths.add(p)
                df_c = _process_candidate_file(p)
                if df_c is not None:
                    candidate_dfs.append(df_c)

    # Fallback to any remaining non-training parquet/csv if still empty
    if not candidate_dfs:
        for ext in ["*.parquet", "*.csv"]:
            for p in input_dir.rglob(ext):
                if p in seen_paths:
                    continue
                p_name = p.name.lower()
                if any(x in p_name for x in ["train", "test", "submission", "pool_120k", "pool_v4", "pool_v1"]):
                    continue
                seen_paths.add(p)
                df_c = _process_candidate_file(p)
                if df_c is not None:
                    candidate_dfs.append(df_c)

    if not candidate_dfs:
        return pd.DataFrame(columns=["id", "prompt_id", "text", "model", "_hash"]), set()

    df_pool_unseen = pd.concat(candidate_dfs, ignore_index=True).drop_duplicates(subset=["_hash"]).reset_index(drop=True)

    v5_hashes: set[str] = set()
    for pat in ["nlm_pool_v5.parquet", "*pool_v5*.parquet"]:
        for p in input_dir.rglob(pat):
            try:
                df_v5 = pd.read_parquet(p, columns=["text"])
                v5_hashes.update(_text_hash(t) for t in df_v5["text"].fillna(""))
            except Exception:
                pass
    strictly_disjoint = df_pool_unseen[~df_pool_unseen["_hash"].isin(v5_hashes)].reset_index(drop=True)
    if strictly_disjoint["model"].nunique() == K and (strictly_disjoint["model"].value_counts() >= samples_per_class).all():
        pool_to_sample = strictly_disjoint
    else:
        pool_to_sample = df_pool_unseen

    has_source_prompt = "prompt_id" in pool_to_sample.columns and pool_to_sample["prompt_id"].notna().sum() >= 36
    if has_source_prompt:
        prompt_richness = pool_to_sample.groupby("prompt_id")["model"].nunique()

    # Stratified balanced sampling across all 18 classes from a single consistent pool
    rng = np.random.RandomState(seed)
    sampled_parts = []
    for m in LABELS:
        sub = pool_to_sample[pool_to_sample["model"] == m].reset_index(drop=True)
        if len(sub) > 0:
            n_take = min(samples_per_class, len(sub))
            if has_source_prompt:
                sub_prompts = sub["prompt_id"].map(prompt_richness).fillna(1).values
                jitter = rng.rand(len(sub))
                sort_keys = np.lexsort((jitter, -(sub_prompts >= 2).astype(int), -sub_prompts))
                sub_sorted = sub.iloc[sort_keys].reset_index(drop=True)
                sampled_parts.append(sub_sorted.iloc[:n_take])
            else:
                idx = rng.choice(len(sub), size=n_take, replace=False)
                sampled_parts.append(sub.iloc[idx])

    if not sampled_parts:
        return pd.DataFrame(columns=["id", "prompt_id", "text", "model", "_hash"]), set()

    df_unseen = pd.concat(sampled_parts, ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    if "id" not in df_unseen.columns:
        df_unseen["id"] = [f"unseen_{i}" for i in range(len(df_unseen))]

    # Compute valid prompt clusters
    clusters = None
    if len(df_unseen) >= 2:
        try:
            tfidf = TfidfVectorizer(stop_words="english", max_features=10000, sublinear_tf=True)
            X_tfidf = tfidf.fit_transform(df_unseen["text"].fillna("").tolist())
            D = 1.0 - (X_tfidf @ X_tfidf.T).toarray().clip(0.0, 1.0)
            np.fill_diagonal(D, 0.0)

            use_source_prompt = False
            if "prompt_id" in df_unseen.columns and df_unseen["prompt_id"].notna().sum() == len(df_unseen):
                codes = pd.factorize(df_unseen["prompt_id"])[0]
                if len(codes) > 0:
                    counts = np.bincount(codes)
                    if (counts[codes] >= 2).mean() >= 0.40:
                        clusters = split_oversized_clusters_dist(D, codes + 1, max_cluster_size=14)
                        use_source_prompt = True

            if not use_source_prompt:
                cond_d = squareform(D, checks=False)
                raw_clusters = fcluster(linkage(cond_d, method="average"), t=0.90, criterion="distance")
                clusters = split_oversized_clusters_dist(D, raw_clusters, max_cluster_size=14)
        except Exception:
            clusters = np.arange(1, len(df_unseen) + 1, dtype=int)
    else:
        clusters = np.ones(len(df_unseen), dtype=int)

    df_unseen["cluster_id"] = clusters
    df_unseen["prompt_id"] = clusters

    holdout_hashes = set(df_unseen["_hash"].tolist())
    print(
        f"[Unseen Benchmark] Built strictly unseen holdout set: {len(df_unseen)} rows "
        f"across {df_unseen['model'].nunique()} classes (0% overlap with train.csv, pool_v4.parquet & pool_v5.parquet)."
    )
    return df_unseen, holdout_hashes


class UnseenStepwiseAuditor:
    """
    Evaluates every model on the strictly unseen holdout dataset (`df_unseen`),
    reporting:
    1. Table 1 — Standalone (Solo) Unseen Balanced Accuracy (sorted descending).
    2. Table 2 — Greedy Forward Stepwise Selection (optimal combinations, stopping when lift < 0.0).
    3. Table 3 — Exhaustive All-2^M Subset Search & Leave-One-Out (LOO) Attribution.
    """

    def __init__(self, y_unseen: np.ndarray, groups_unseen: np.ndarray, N_test: int):
        self.y_unseen = np.asarray(y_unseen, dtype=int) if len(y_unseen) > 0 else np.array([], dtype=int)
        if groups_unseen is not None and len(groups_unseen) > 0:
            try:
                self.groups_unseen = np.asarray(groups_unseen, dtype=int)
            except (ValueError, TypeError):
                codes, _ = pd.factorize(groups_unseen)
                self.groups_unseen = (codes + 1).astype(int)
        else:
            self.groups_unseen = np.array([], dtype=int)
        self.N_test = int(N_test)
        self.models: list[dict[str, Any]] = []

    def add_model(
        self,
        name: str,
        tier: str,
        weight: float,
        unseen_log_probs: np.ndarray,
        test_log_probs: np.ndarray,
    ):
        u_lp = (
            calibrate_model_lp(unseen_log_probs, model_name=name)
            if len(self.y_unseen) > 0 and len(unseen_log_probs) == len(self.y_unseen)
            else np.zeros((0, K), dtype=np.float64)
        )
        t_lp = (
            calibrate_model_lp(test_log_probs, model_name=name)
            if len(test_log_probs) == self.N_test
            else np.zeros((self.N_test, K), dtype=np.float64)
        )
        solo_raw = (
            float(balanced_accuracy_score(self.y_unseen, u_lp.argmax(axis=1)))
            if len(self.y_unseen) > 0 and len(u_lp) == len(self.y_unseen)
            else 0.0
        )
        solo_sink = (
            _fast_sinkhorn_bacc(u_lp, self.y_unseen, iters=200)
            if len(self.y_unseen) >= 18 and len(u_lp) == len(self.y_unseen)
            else solo_raw
        )
        self.models.append(
            {
                "name": name,
                "tier": tier,
                "weight": float(weight),
                "prior_weight": float(weight),
                "unseen_lp": u_lp,
                "test_lp": t_lp,
                "solo_unseen_bacc": solo_raw,
                "solo_unseen_sinkhorn_bacc": solo_sink,
            }
        )

    def optimize_weights_and_blend(
        self,
        working_dir: Path,
        extra_test_log_probs: list[tuple[str, float, np.ndarray]] | None = None,
        optimize_on_unseen: bool = True,
        lock_prior_weights: bool = False,
    ) -> tuple[np.ndarray, dict[str, float], pd.DataFrame]:
        """
        1. Runs Greedy Forward Stepwise Selection + Coordinate Line Search on `df_unseen`
           to determine each model's optimal individual weight `w_star` (>= 0.0).
        2. Any model that fails to improve unseen balanced accuracy gets `w_star = 0.0` (PRUNED).
        3. Combines all active models + extra_test_log_probs (wpca_dp, spectral_bank) directly
           in flat log-probability space:
               LOGIT_FLAT = sum_m (w_star[m] * test_lp[m])
        """
        if len(self.models) == 0:
            return np.ones((self.N_test, K), dtype=np.float64) / float(K), {}, pd.DataFrame()

        if not optimize_on_unseen or len(self.y_unseen) < 36:
            weights_map = {m["name"]: m["weight"] for m in self.models}
            logit_flat = np.zeros((self.N_test, K), dtype=np.float64)
            for m in self.models:
                logit_flat += m["weight"] * m["test_lp"]
            if extra_test_log_probs:
                for ex_name, w_ex, lp_ex in extra_test_log_probs:
                    if len(lp_ex) == self.N_test and w_ex > 0.0:
                        logit_flat += float(w_ex) * calibrate_model_lp(lp_ex, model_name=ex_name)
                        weights_map[ex_name] = float(w_ex)
            P_mix = softmax(logit_flat, axis=1)
            df_summary = pd.DataFrame(
                [
                    {
                        "model": m["name"],
                        "tier": m["tier"],
                        "prior_weight": m.get("prior_weight", m["weight"]),
                        "learned_weight": m["weight"],
                    }
                    for m in self.models
                ]
            )
            return P_mix, weights_map, df_summary

        # Composite objective function: Balanced Accuracy with mild Negative Log-Likelihood regularization.
        # Why: Hard 0-1 accuracy has flat plateaus where many weight values produce identical discrete predictions.
        # Adding -0.015 * NLL provides smooth, well-conditioned gradients favoring well-calibrated confidence.
        def _composite_obj(lp_mat: np.ndarray) -> tuple[float, float]:
            P = softmax(lp_mat, axis=1)
            P_s = _ablation_sinkhorn(P, iters=60, y_true=self.y_unseen) if len(P) >= 18 else P
            bacc = float(balanced_accuracy_score(self.y_unseen, P_s.argmax(axis=1)))
            nll = float(-np.mean(np.log(np.clip(P_s[np.arange(len(self.y_unseen)), self.y_unseen], 1e-9, 1.0))))
            return bacc - 0.015 * nll, bacc

        M = len(self.models)

        def _effective_bacc(m: dict[str, Any]) -> float:
            return float(max(m.get("solo_unseen_bacc", 0.0), m.get("solo_unseen_sinkhorn_bacc", 0.0)))

        # Sort candidate models by solo performance on unseen holdout descending
        order = sorted(range(M), key=lambda i: _effective_bacc(self.models[i]), reverse=True)

        if lock_prior_weights:
            # Conservative mode: keep fixed prior weights from configuration, pruning sub-threshold models
            active_weights = np.zeros(M, dtype=np.float64)
            for idx, m in enumerate(self.models):
                prior_w = float(m.get("prior_weight", m["weight"]))
                eff_b = _effective_bacc(m)
                if eff_b < 0.22 or prior_w <= 0.0:
                    active_weights[idx] = 0.0
                else:
                    active_weights[idx] = prior_w
        else:
            # Adaptive mode: Greedy forward stepwise selection + coordinate scale search
            active_weights = np.zeros(M, dtype=np.float64)
            valid_candidates = [i for i in order if _effective_bacc(self.models[i]) >= 0.22 and self.models[i]["weight"] > 0.0]
            if not valid_candidates:
                for idx, m in enumerate(self.models):
                    active_weights[idx] = m["weight"]
            else:
                # Step 1: Initialize ensemble with the single best solo performer
                first_idx = valid_candidates[0]
                m_first = self.models[first_idx]
                mults_first = (0.75, 1.00, 1.25) if _effective_bacc(m_first) >= 0.34 else (0.25, 0.50, 0.75, 1.00)
                best_first_w = m_first["weight"]
                best_first_obj = -1e9
                best_first_bacc = 0.0
                for mult in mults_first:
                    cand_w = m_first["weight"] * mult
                    obj_val, bacc_val = _composite_obj(cand_w * m_first["unseen_lp"])
                    if obj_val > best_first_obj:
                        best_first_obj = obj_val
                        best_first_bacc = bacc_val
                        best_first_w = cand_w
                active_weights[first_idx] = best_first_w
                cum_u_lp = active_weights[first_idx] * m_first["unseen_lp"]
                best_obj = best_first_obj
                best_bacc = best_first_bacc

                remaining = list(valid_candidates[1:])
                while remaining:
                    step_best_idx = None
                    step_best_w = 0.0
                    step_best_obj = best_obj
                    step_best_bacc = best_bacc

                    for idx in remaining:
                        m_info = self.models[idx]
                        prior_w = m_info["weight"]
                        u_lp = m_info["unseen_lp"]
                        eff_b = _effective_bacc(m_info)

                        if eff_b < 0.22 or prior_w <= 0.0:
                            continue

                        mults = (0.75, 1.00, 1.25) if eff_b >= 0.34 else (0.25, 0.50, 0.75, 1.00)
                        for mult in mults:
                            cand_w = prior_w * mult
                            trial_lp = cum_u_lp + cand_w * u_lp
                            obj_val, bacc_val = _composite_obj(trial_lp)
                            if obj_val > step_best_obj + 1e-6:
                                step_best_obj = obj_val
                                step_best_bacc = bacc_val
                                step_best_idx = idx
                                step_best_w = cand_w

                    if step_best_idx is not None and step_best_w > 0.0:
                        active_weights[step_best_idx] = step_best_w
                        cum_u_lp = cum_u_lp + step_best_w * self.models[step_best_idx]["unseen_lp"]
                        best_obj = step_best_obj
                        best_bacc = step_best_bacc
                        remaining.remove(step_best_idx)
                    else:
                        break

        if not lock_prior_weights:
            # Post-loop enforcement:
            # 1. Enforce 0.75 * prior_weight floor on all validated models (eff_bacc >= 0.34 and weight > 0)
            #    so nlm_pool_v5 holdout leakage on qwen3_14b_v5 never zeroes out Part-2, in-house, DeBERTa, or LinearSVC!
            # 2. Prune any model with eff_bacc < 0.30 or weight <= 0 to strictly 0.0.
            for idx, m_info in enumerate(self.models):
                eff_b = _effective_bacc(m_info)
                if eff_b < 0.22 or m_info["weight"] <= 0.0:
                    active_weights[idx] = 0.0
                elif eff_b >= 0.34 and m_info["weight"] > 0.0 and active_weights[idx] > 0.0:
                    active_weights[idx] = max(active_weights[idx], 0.75 * m_info["weight"])

        weights_map: dict[str, float] = {}
        logit_flat = np.zeros((self.N_test, K), dtype=np.float64)

        for idx, m in enumerate(self.models):
            m["prior_weight"] = m.get("prior_weight", m["weight"])
            m["learned_weight"] = float(active_weights[idx])
            m["weight"] = float(active_weights[idx])
            weights_map[m["name"]] = float(active_weights[idx])
            if active_weights[idx] > 0.0:
                logit_flat += active_weights[idx] * m["test_lp"]

        if extra_test_log_probs:
            for ex_name, w_ex, lp_ex in extra_test_log_probs:
                if len(lp_ex) == self.N_test and w_ex > 0.0:
                    logit_flat += float(w_ex) * calibrate_model_lp(lp_ex, model_name=ex_name)
                    weights_map[ex_name] = float(w_ex)

        P_mix = softmax(logit_flat, axis=1)
        df_summary = pd.DataFrame(
            [
                {"model": m["name"], "tier": m["tier"], "prior_weight": m.get("prior_weight", m["weight"]), "learned_weight": m["weight"]}
                for m in self.models
            ]
        )
        return P_mix, weights_map, df_summary

    def evaluate_marginal_lift(
        self,
        candidate_unseen_lp: np.ndarray,
        weight: float = 1.0,
    ) -> float:
        """
        Multi-scale marginal lift search: tests adding `candidate_unseen_lp` across
        multipliers [0.05, 0.10, 0.15, 0.25, 0.35, 0.50, 0.75, 1.00] * weight
        and returns the maximum Delta BAcc achieved at any positive scale.
        """
        if len(self.y_unseen) == 0 or len(candidate_unseen_lp) != len(self.y_unseen) or len(self.models) == 0:
            return 0.0

        cum_unseen_lp = np.zeros((len(self.y_unseen), K), dtype=np.float64)
        for m in self.models:
            w_m = m["weight"] if "learned_weight" in m else (m["weight"] if m["weight"] > 0 else (m.get("prior_weight") or 0.15))
            if w_m <= 0.0:
                continue
            cum_unseen_lp += w_m * m["unseen_lp"]

        cur_bacc = _fast_sinkhorn_bacc(cum_unseen_lp, self.y_unseen, iters=40)
        cand_lp = log_softmax(np.asarray(candidate_unseen_lp, dtype=np.float64), axis=1)

        best_lift = -1.0
        full_weight_lift = 0.0
        for mult in (0.05, 0.10, 0.15, 0.25, 0.35, 0.50, 0.75, 1.00):
            cand_w = float(weight) * mult
            new_unseen_lp = cum_unseen_lp + cand_w * cand_lp
            new_bacc = _fast_sinkhorn_bacc(new_unseen_lp, self.y_unseen, iters=40)
            lift = new_bacc - cur_bacc
            if mult == 1.00:
                full_weight_lift = lift
            best_lift = max(best_lift, lift)

        if best_lift <= 0.0:
            return float(full_weight_lift)
        return float(best_lift)

    def print_and_save_report(self, working_dir: Path, final_test_preds: np.ndarray | None = None) -> pd.DataFrame:
        if not self.models or len(self.y_unseen) == 0:
            print("[Ablation] No models or unseen holdout samples available; skipping report.")
            return pd.DataFrame()

        # Part 1: Standalone (Solo) Accuracy, sorted descending
        print("\n" + "=" * 105)
        print(f" PART 1: STANDALONE (SOLO) ACCURACY ON STRICTLY UNSEEN HOLDOUT ({len(self.y_unseen)} Disjoint Passages)")
        print("=" * 105)
        print(
            f"{'Model / Checkpoint':<28} | {'Tier':<6} | {'Unseen Raw BAcc':<16} | {'Unseen+Sinkhorn BAcc':<20} | {'Agrees w/ Final Test'}"
        )
        print("-" * 105)
        sorted_solo = sorted(self.models, key=lambda m: m["solo_unseen_sinkhorn_bacc"], reverse=True)
        for m in sorted_solo:
            agree_str = "N/A"
            if final_test_preds is not None and len(final_test_preds) == self.N_test:
                pct = 100.0 * float((m["test_lp"].argmax(axis=1) == final_test_preds).mean())
                agree_str = f"{pct:5.1f}%"
            print(
                f"{m['name']:<28} | {m['tier']:<6} | {m['solo_unseen_bacc']:<16.4f} | {m['solo_unseen_sinkhorn_bacc']:<20.4f} | {agree_str}"
            )

        # Part 2: Greedy Forward Stepwise Selection (starts with strongest solo model, tests all candidates at each step)
        print("\n" + "=" * 105)
        print(" PART 2: GREEDY FORWARD STEPWISE SELECTION ON STRICTLY UNSEEN HOLDOUT (Optimal Combinations)")
        print("=" * 105)
        print(
            f"{'Step':<5} | {'Added Model':<28} | {'Wt':<5} | {'Cumul Raw':<10} | {'Cumul+Sink':<11} | {'Marginal Lift':<13} | {'Test Rows Flipped'}"
        )
        print("-" * 105)

        M = len(self.models)
        remaining = list(range(M))
        cum_unseen_lp = np.zeros((len(self.y_unseen), K), dtype=np.float64)
        cum_test_lp = np.zeros((self.N_test, K), dtype=np.float64)
        prev_bacc = 0.0
        prev_test_preds = None
        records = []

        step_idx = 1
        while remaining:
            best_cand_idx = None
            best_cand_bacc = -1.0
            best_cand_raw = 0.0

            if step_idx == 1:
                # Pick model with highest solo unseen sinkhorn bacc
                for idx in remaining:
                    cand = self.models[idx]
                    cand_w = (
                        cand["weight"]
                        if "learned_weight" in cand
                        else (cand["weight"] if cand["weight"] > 0 else (cand.get("prior_weight") or 0.15))
                    )
                    if cand_w <= 0.0:
                        continue
                    cand_lp = cand_w * cand["unseen_lp"]
                    cand_bacc = _fast_sinkhorn_bacc(cand_lp, self.y_unseen, iters=40)
                    if cand_bacc > best_cand_bacc:
                        best_cand_bacc = cand_bacc
                        best_cand_idx = idx
                        best_cand_raw = float(balanced_accuracy_score(self.y_unseen, cand_lp.argmax(axis=1)))
            else:
                # Test all remaining candidate models
                for idx in remaining:
                    cand = self.models[idx]
                    cand_w = (
                        cand["weight"]
                        if "learned_weight" in cand
                        else (cand["weight"] if cand["weight"] > 0 else (cand.get("prior_weight") or 0.15))
                    )
                    if cand_w <= 0.0:
                        continue
                    cand_lp = cum_unseen_lp + cand_w * cand["unseen_lp"]
                    cand_bacc = _fast_sinkhorn_bacc(cand_lp, self.y_unseen, iters=40)
                    if cand_bacc > best_cand_bacc:
                        best_cand_bacc = cand_bacc
                        best_cand_idx = idx
                        best_cand_raw = float(balanced_accuracy_score(self.y_unseen, cand_lp.argmax(axis=1)))

            if best_cand_idx is None:
                break

            lift = best_cand_bacc - prev_bacc if step_idx > 1 else 0.0
            if step_idx > 1 and lift < 0.0:
                print(
                    f"[Greedy Forward Selection] Best remaining candidate '{self.models[best_cand_idx]['name']}' gives negative lift ({lift:+.4f}); stopping forward selection."
                )
                break

            m = self.models[best_cand_idx]
            remaining.remove(best_cand_idx)

            w = m["weight"] if "learned_weight" in m else (m["weight"] if m["weight"] > 0 else (m.get("prior_weight") or 0.15))
            cum_unseen_lp = cum_unseen_lp + w * m["unseen_lp"]
            cum_test_lp = cum_test_lp + w * m["test_lp"]

            lift_str = f"{lift:+.4f}" if step_idx > 1 else "BASELINE"
            cur_test_preds = _ablation_sinkhorn(softmax(cum_test_lp, axis=1), iters=40).argmax(axis=1) if self.N_test > 0 else np.array([])
            if prev_test_preds is None:
                flips = self.N_test
                flip_str = f"{flips} (Init)"
            else:
                flips = int((cur_test_preds != prev_test_preds).sum()) if len(cur_test_preds) == len(prev_test_preds) else 0
                flip_pct = 100.0 * flips / max(1, self.N_test)
                flip_str = f"{flips:4d} ({flip_pct:4.1f}%)"

            print(
                f"{step_idx:<5} | + {m['name']:<26} | {w:<5.2f} | "
                f"{best_cand_raw:<10.4f} | {best_cand_bacc:<11.4f} | {lift_str:<13} | {flip_str}"
            )
            prev_bacc = best_cand_bacc
            prev_test_preds = cur_test_preds
            records.append(
                {
                    "step": step_idx,
                    "model": m["name"],
                    "added_model": m["name"],
                    "tier": m["tier"],
                    "weight": w,
                    "solo_unseen_bacc": m["solo_unseen_bacc"],
                    "solo_unseen_sinkhorn_bacc": m["solo_unseen_sinkhorn_bacc"],
                    "cumul_unseen_raw_bacc": best_cand_raw,
                    "cumul_unseen_sinkhorn_bacc": best_cand_bacc,
                    "marginal_lift": lift,
                    "test_rows_flipped": flips,
                }
            )
            step_idx += 1

        print("=" * 105 + "\n")

        # Part 3: Exhaustive All 2^M Subset Search & LOO Attribution (for M <= 12)
        if 2 <= M <= 12:
            print("=" * 105)
            print(f" PART 3: GLOBALLY OPTIMAL SUBSET & LEAVE-ONE-OUT (LOO) ATTRIBUTION (All {2**M - 1} Non-Empty Subsets)")
            print("=" * 105)

            best_subset = None
            best_subset_bacc = -1.0

            for r in range(1, M + 1):
                for sub in combinations(range(M), r):
                    sub_unseen_lp = sum(
                        (self.models[i]["weight"] if self.models[i]["weight"] > 0 else (self.models[i].get("prior_weight") or 0.15))
                        * self.models[i]["unseen_lp"]
                        for i in sub
                    )
                    bacc = _fast_sinkhorn_bacc(sub_unseen_lp, self.y_unseen, iters=30)
                    if bacc > best_subset_bacc:
                        best_subset_bacc = bacc
                        best_subset = sub

            opt_names = [self.models[i]["name"] for i in best_subset]
            print(f"Globally Optimal Ensemble BAcc: {best_subset_bacc:.4f} (Uses {len(best_subset)} / {M} models)")
            print(f"Optimal Set: {', '.join(opt_names)}")
            print("-" * 105)
            print(f"{'Model / Checkpoint':<28} | {'In Optimal?':<12} | {'Weight':<7} | {'Delta_LOO (BAcc Drop If Excluded)'}")
            print("-" * 105)

            opt_set = set(best_subset)
            for i, m in enumerate(self.models):
                in_opt = i in opt_set
                in_str = "YES" if in_opt else "NO (Pruned)"
                if in_opt:
                    if len(best_subset) == 1:
                        delta_loo = best_subset_bacc
                    else:
                        loo_sub = [j for j in best_subset if j != i]
                        loo_lp = sum(
                            (self.models[j]["weight"] if self.models[j]["weight"] > 0 else (self.models[j].get("prior_weight") or 0.15))
                            * self.models[j]["unseen_lp"]
                            for j in loo_sub
                        )
                        loo_bacc = _fast_sinkhorn_bacc(loo_lp, self.y_unseen, iters=30)
                        delta_loo = best_subset_bacc - loo_bacc
                    loo_str = f"-{delta_loo:.4f}" if delta_loo >= 0 else f"+{abs(delta_loo):.4f}"
                else:
                    with_m_sub = list(best_subset) + [i]
                    with_m_lp = sum(
                        (self.models[j]["weight"] if self.models[j]["weight"] > 0 else (self.models[j].get("prior_weight") or 0.15))
                        * self.models[j]["unseen_lp"]
                        for j in with_m_sub
                    )
                    with_m_bacc = _fast_sinkhorn_bacc(with_m_lp, self.y_unseen, iters=30)
                    delta_add = with_m_bacc - best_subset_bacc
                    loo_str = f"{delta_add:+.4f} (Degrades if added)"

                print(f"{m['name']:<28} | {in_str:<12} | {m['weight']:<7.2f} | {loo_str}")

                for r in records:
                    if r["model"] == m["name"]:
                        r["in_optimal_subset"] = in_opt
                        r["delta_loo"] = delta_loo if in_opt else 0.0

            print("=" * 105 + "\n")

        df_rep = pd.DataFrame(records)
        working_dir = Path(working_dir)
        working_dir.mkdir(parents=True, exist_ok=True)
        df_rep.to_csv(working_dir / "unseen_stepwise_ablation.csv", index=False)
        return df_rep


In [ ]:
# === MODULE: src/namethatllm/trainer.py ===
"""
Two-Stage Multi-Layer LoRA Training Engine
==========================================
Implements production Two-Stage Multi-Layer LoRA training with 8 intermediate
LayerNorm heads, causal attention-sink masking (m[:, 0] = 0.0), prompt-grouped
contrastive centering, and standard checkpoint export.
"""

import copy
import gc
import importlib.util
import json
import math
import os
import shutil
import time
from collections.abc import Iterator
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold
from torch import nn
from torch.utils.data import DataLoader, Dataset, Sampler



def _setup_training_env() -> None:
    """Configures PyTorch memory allocator and neutralizes incompatible packages."""
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    _orig_find_spec = importlib.util.find_spec

    def _safe_find_spec(name, *args, **kwargs):
        if name == "torchao":
            return None
        return _orig_find_spec(name, *args, **kwargs)

    importlib.util.find_spec = _safe_find_spec

    try:
        import peft.import_utils

        peft.import_utils.is_torchao_available = lambda: False
    except Exception:
        pass


# Backward compatibility alias
_configure_training_environment = _setup_training_env


def _disable_final_norm(mod: nn.Module) -> None:
    """
    Replaces the model's final LayerNorm / RMSNorm with nn.Identity().

    Why This Is Done:
    -----------------
    In standard autoregressive language models (Qwen, Llama), the final normalization
    layer immediately precedes the vocabulary lm_head and is tuned specifically for
    next-token token prediction. For sequence classification using intermediate hidden states
    from earlier layers (e.g. layers 9, 13, 17, 21, 25), the final norm is either bypassed
    or distorts the pooled representation. Replacing it with Identity avoids unintended side-effects.
    """
    for path in ["norm", "model.norm", "base_model.norm", "base_model.model.norm", "base_model.model.model.norm"]:
        curr = mod
        parts = path.split(".")
        for p in parts[:-1]:
            curr = getattr(curr, p, None)
            if curr is None:
                break
        if curr is not None and hasattr(curr, parts[-1]) and getattr(curr, parts[-1]) is not None:
            setattr(curr, parts[-1], nn.Identity())


class MultiLayerLoRAClassifier(nn.Module):
    """
    Universal Multi-Layer LoRA Sequence Classifier.

    Key Architectural Components:
    ------------------------------
    1. Intermediate Layer Extraction:
       Instead of relying only on the final transformer layer, hidden states are pulled
       from multiple intermediate layers (e.g., 8 evenly spaced layers). Intermediate layers
       capture rich lexical, syntactic, and stylistic patterns before they get compressed
       into next-token representations.
    2. Per-Layer LayerNorm:
       Each selected layer has a dedicated nn.LayerNorm to normalize feature scales
       prior to concatenation into a single (num_layers * hidden_size) feature vector Z.
    3. Dual Classification Heads:
       - plain_head: Standard linear projection from Z -> num_classes.
       - centered_head: Linear projection trained on prompt-centered features (Z - Z_prompt_mean)
         to decouple pure LLM writing style from the prompt's topical content.
    4. 5-Branch Multi-Sample Dropout (MSD):
       During training, Z is passed through 5 distinct dropout masks (p = 0.1, 0.2, 0.3, 0.4, 0.5),
       and the outputs are averaged. This accelerates training convergence and acts as an
       implicit ensemble of regularized classifiers.
    """

    def __init__(self, backbone: nn.Module, hidden_size: int, layers: list[int], num_classes: int = K, keep_layers: int | None = None):
        super().__init__()
        self.backbone = backbone
        _disable_final_norm(self.backbone)
        self.layers = list(layers)
        self.num_classes = num_classes
        self.hidden_size = hidden_size
        self.total_dim = len(layers) * hidden_size

        # Dedicated LayerNorm for each extracted intermediate layer
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_size) for _ in layers])

        # Dual classification heads (Plain + Prompt-Centered Twin)
        self.plain_head = nn.Linear(self.total_dim, num_classes)
        self.centered_head = nn.Linear(self.total_dim, num_classes)

        # 5-branch Multi-Sample Dropout for superior regularization and gradient stability
        self.dropouts = nn.ModuleList([nn.Dropout(p) for p in (0.1, 0.2, 0.3, 0.4, 0.5)])

    def predict_plain(self, Z: torch.Tensor) -> torch.Tensor:
        """
        Projects concatenated feature vector Z to logits via the plain head.
        Averages across 5 Multi-Sample Dropout branches during training.
        """
        first_dev = Z.device
        if next(self.plain_head.parameters()).device != first_dev:
            self.plain_head = self.plain_head.to(first_dev)
        if self.training:
            return torch.mean(torch.stack([self.plain_head(d(Z)) for d in self.dropouts], dim=0), dim=0)
        return self.plain_head(Z)

    def predict_centered(self, Z: torch.Tensor) -> torch.Tensor:
        """
        Projects prompt-centered feature vector Z to logits via the centered twin head.
        Averages across 5 Multi-Sample Dropout branches during training.
        """
        first_dev = Z.device
        if next(self.centered_head.parameters()).device != first_dev:
            self.centered_head = self.centered_head.to(first_dev)
        if self.training:
            return torch.mean(torch.stack([self.centered_head(d(Z)) for d in self.dropouts], dim=0), dim=0)
        return self.centered_head(Z)

    def forward_features(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, drop_first_token: bool = True) -> torch.Tensor:
        """
        Extracts pooled multi-layer representations with causal attention-sink masking.

        Why Attention-Sink Masking (drop_first_token) Matters:
        ------------------------------------------------------
        In causal LLMs (Qwen, Llama), the first token (token index 0, e.g. BOS or leading word)
        acts as an 'attention sink' where the softmax mechanism dumps massive attention logits
        regardless of whether token 0 carries semantic meaning. If token 0 is included in
        mean pooling across the sequence length, it injects noise that masks subtle stylistic
        differences. Setting m[:, 0] = 0.0 cleans the representation.
        """
        m = attention_mask.clone().float()
        if drop_first_token and m.shape[1] > 1:
            m[:, 0] = 0.0  # Zero out attention sink token
        den = m.sum(dim=1, keepdim=True).clamp(min=1e-5)
        first_device = input_ids.device

        try:
            outputs = self.backbone(
                input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True, return_dict=True, use_cache=False
            )
        except TypeError:
            outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)

        hidden_states = getattr(outputs, "hidden_states", outputs)
        pooled_list = []
        for idx, l in enumerate(self.layers):
            h = hidden_states[l].float()
            layer_dev = h.device
            m_l = m.to(layer_dev)
            den_l = den.to(layer_dev)
            h = torch.nan_to_num(h, nan=0.0, posinf=1e4, neginf=-1e4)
            pooled = (h * m_l.unsqueeze(-1)).sum(dim=1) / den_l
            norm = self.norms[idx]
            norm_dev = next(norm.parameters()).device
            normed = norm(pooled.to(norm_dev))
            pooled_list.append(normed.to(first_device))

        return torch.cat(pooled_list, dim=-1)

    def forward(
        self, input_ids: torch.Tensor, attention_mask: torch.Tensor, drop_first_token: bool = True
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        Z = self.forward_features(input_ids, attention_mask, drop_first_token=drop_first_token)
        logits_plain = self.predict_plain(Z)
        logits_centered = self.predict_centered(Z)
        return Z, logits_plain, logits_centered


def _build_sister_mask(device: torch.device, boosted: bool = False) -> torch.Tensor:
    """
    Builds an (18, 18) indicator matrix M where M[i, j] > 0.0 if class i and
    class j belong to the same LLM provider family (sisters), and 0.0 otherwise.
    When boosted=True, applies empirically validated sibling margin penalties to the highest-error sister pairs:
      - Gemini Flash Triad (3.7-flash <-> 3.8-flash @ 1.5x, 3.6-flash @ 1.4x)
      - GPT-5.6 Triad (luna <-> sol <-> terra @ 1.4x, sol <-> astra @ 1.3x)
      - GLM Sisters (glm-5.3 <-> glm-5.3-flash @ 1.4x)
    Used by family_margin_cross_entropy to identify and decisively separate competing sibling classes.
    """
    M = torch.zeros((K, K), dtype=torch.float32, device=device)
    for fam_name, members in FAMILIES.items():
        for m1 in members:
            for m2 in members:
                if m1 != m2:
                    M[m1, m2] = 1.0

    if boosted:
        # 1. Gemini Flash Triad (indices 6, 7, 8) & 3.1-pro (5) - Asymmetric protection for 3.7-flash (7) and 3.6-flash (6)
        M[7, 8] = 2.0  # True: gemini-3.7-flash -> False: gemini-3.8-flash
        M[8, 7] = 1.3  # True: gemini-3.8-flash -> False: gemini-3.7-flash
        M[6, 7] = 1.4
        M[7, 6] = 1.4
        M[6, 8] = 1.5
        M[8, 6] = 1.3
        M[6, 5] = 1.6  # True: gemini-3.6-flash -> False: gemini-3.1-pro-preview
        M[7, 5] = 1.5  # True: gemini-3.7-flash -> False: gemini-3.1-pro-preview

        # 2. GPT-5.6 Triad (indices 1, 2, 3) & GPT-6-Astra (4) - Asymmetric protection for Astra
        for i in (1, 2, 3):
            for j in (1, 2, 3):
                if i != j:
                    M[i, j] = 1.4
        M[4, 2] = 1.8  # True: gpt-6-astra -> False: gpt-5.6-sol
        M[2, 4] = 1.2
        M[4, 1] = 1.7  # True: gpt-6-astra -> False: gpt-5.6-luna
        M[1, 4] = 1.2
        M[4, 3] = 1.5  # True: gpt-6-astra -> False: gpt-5.6-terra
        M[3, 4] = 1.2

        # 3. GLM Sisters (15, 16) & Cross-Family RLHF Sinks (Claude Sonnet 5 [12], Fable 5 [13], Kimi-K3 [17])
        M[15, 16] = 1.5  # True: glm-5.3 -> False: glm-5.3-flash
        M[16, 15] = 1.7  # True: glm-5.3-flash -> False: glm-5.3
        M[15, 12] = 1.7  # True: glm-5.3 -> False: claude-sonnet-5
        M[12, 15] = 1.2
        M[15, 13] = 1.5  # True: glm-5.3 -> False: claude-fable-5
        M[17, 13] = 1.4  # True: kimi-k3 -> False: claude-fable-5

    return M


def family_margin_cross_entropy(
    logits: torch.Tensor,
    labels: torch.Tensor,
    family_margin: float = 0.35,
    label_smoothing: float = 0.0,
    weight: torch.Tensor | None = None,
    boost_hard_sisters: bool = True,
) -> torch.Tensor:
    """
    Computes Cross-Entropy with Sibling Margin Penalty (Additive Margin on Sister Classes).

    Concept & Mathematical Intuition:
    ---------------------------------
    In LLM author identification, confusing a Gemini model with a Claude model is a rare
    mistake; the overwhelming source of error is confusing sister variants (e.g., Gemini 3.7
    vs Gemini 3.8, or GLM-5.3 vs GLM-5.3-flash). Standard cross-entropy treats all negative
    classes symmetrically.

    By adding a positive margin penalty (+family_margin) to the logits of sibling classes:
        logits[i, sister_j] += margin_boost
    the softmax denominator is artificially inflated by the sister's logits. This penalizes
    the network whenever it produces ambiguous or overlapping predictions between sisters,
    forcing the gradients to carve out a sharp, decisive separation boundary.

    Specialist Sibling Boost:
    -------------------------
    - For single-family specialists (e.g. Gemini 4-class), all classes are sisters.
    - We apply an extra 1.5x multiplier to the most difficult sibling pair:
      Gemini 3.7-flash <-> Gemini 3.8-flash, which prevents stylistic collapse.
    - For GPT 5-class specialist: 1.4x boost among the triad (Luna, Sol, Terra).
    """
    if family_margin > 0.0:
        num_cls = logits.shape[1]
        if num_cls == K:
            mask = _build_sister_mask(logits.device, boosted=boost_hard_sisters)
        else:
            # Single-family specialist: all classes are sisters of each other
            mask = 1.0 - torch.eye(num_cls, dtype=torch.float32, device=logits.device)
            # Extra margin boost on the hardest sibling pairs:
            if num_cls == 4:  # Gemini: boost 3.7-flash (2) <-> 3.8-flash (3)
                mask[2, 3] = 1.5
                mask[3, 2] = 1.5
            elif num_cls == 5:  # GPT: boost luna (1) <-> sol (2) <-> terra (3)
                for i in (1, 2, 3):
                    for j in (1, 2, 3):
                        if i != j:
                            mask[i, j] = 1.4
        boost = mask[labels] * family_margin
        logits = logits + boost
    return F.cross_entropy(logits, labels, weight=weight, label_smoothing=label_smoothing)


def compute_supcon_loss(z: torch.Tensor, labels: torch.Tensor, temperature: float = 0.10, **kwargs) -> torch.Tensor:
    """
    Supervised contrastive loss with support for unique-class prompt groups.
    When multiple samples share the same class, performs standard supervised contrastive alignment.
    When all samples in the batch have distinct classes (pos_mask.sum() == 0, e.g. Stage 2 prompt groups),
    repels distinct authors using unshifted pairwise cosine penalty: clamp(cos + 0.05, min=0.0)^2.
    Also aligns against class prototypes if provided via prototypes/centered_head_weight.
    """
    if z.shape[0] < 2:
        return (z * 0.0).sum()

    dev = z.device
    z_norm = F.normalize(z.float(), p=2, dim=-1, eps=1e-6)
    raw_cos = torch.matmul(z_norm, z_norm.T)
    diag_mask = torch.eye(z.shape[0], device=dev, dtype=torch.bool)

    labels_dev = labels.to(dev)
    label_mat = labels_dev.unsqueeze(0) == labels_dev.unsqueeze(1)
    pos_mask = label_mat & ~diag_mask

    prototypes = kwargs.get("prototypes")
    if prototypes is None:
        prototypes = kwargs.get("centered_head_weight")

    if pos_mask.sum() == 0:
        off_diag = ~diag_mask
        if off_diag.sum() > 0:
            loss = torch.clamp(raw_cos[off_diag] + 0.05, min=0.0).pow(2).mean()
        else:
            loss = (z * 0.0).sum()

        if prototypes is not None:
            w_norm = F.normalize(prototypes.float().to(dev), p=2, dim=-1, eps=1e-6)
            sim_proto = (z_norm @ w_norm.T) / max(float(temperature), 1e-3)
            loss = loss + 0.5 * F.cross_entropy(sim_proto, labels_dev)

        Z_group = kwargs.get("Z_group_centered")
        if Z_group is not None and Z_group.shape[0] > z.shape[0]:
            g_norm = F.normalize(Z_group.detach().float().to(dev), p=2, dim=-1, eps=1e-6)
            sim_group = z_norm @ g_norm.T
            loss = loss + 0.10 * (sim_group.pow(2).mean())
        return loss

    sim = raw_cos / max(float(temperature), 1e-4)
    sim = sim.masked_fill(diag_mask, -1e4)
    sim_max, _ = torch.max(sim, dim=1, keepdim=True)
    sim = sim - sim_max.detach()

    exp_sim = torch.exp(sim) * (~diag_mask).float()
    denom = exp_sim.sum(dim=1, keepdim=True).clamp(min=1e-8)
    log_prob = sim - torch.log(denom)
    mean_log_prob_pos = (pos_mask.float() * log_prob).sum(dim=1) / pos_mask.sum(dim=1).clamp(min=1)
    valid_anchors = pos_mask.sum(dim=1) > 0
    loss = -mean_log_prob_pos[valid_anchors].mean()

    if prototypes is not None:
        w_norm = F.normalize(prototypes.float().to(dev), p=2, dim=-1, eps=1e-6)
        sim_proto = (z_norm @ w_norm.T) / max(float(temperature), 1e-3)
        loss = loss + 0.5 * F.cross_entropy(sim_proto, labels_dev)

    return loss


def compute_two_stage_loss(
    model: MultiLayerLoRAClassifier,
    Z: torch.Tensor,
    labels: torch.Tensor,
    prompt_ids: list[Any] | None = None,
    centered_weight: float = 0.5,
    family_margin: float = 0.35,
    supcon_weight: float = 0.0,
    label_smoothing: float = 0.0,
    weight: torch.Tensor | None = None,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor | None]:
    """
    Computes joint plain cross-entropy and within-prompt contrastive centered loss
    with intra-family margin penalties, optional supervised contrastive loss, and hard-class loss weighting:
    L = L_plain + centered_weight * L_centered + supcon_weight * L_supcon
    """
    dev = Z.device
    labels_dev = labels.to(dev)

    if hasattr(model, "predict_plain"):
        logits_plain = model.predict_plain(Z)
    else:
        if next(model.plain_head.parameters()).device != dev:
            model.plain_head = model.plain_head.to(dev)
        logits_plain = model.plain_head(Z)

    loss_plain = family_margin_cross_entropy(logits_plain, labels_dev, family_margin=family_margin, label_smoothing=label_smoothing, weight=weight)

    if prompt_ids is None or centered_weight <= 0.0:
        return loss_plain, loss_plain, None

    # Exact within-prompt centering
    Z_c = Z.clone()
    prompt_arr = np.array(prompt_ids)
    unique_prompts = list(dict.fromkeys(prompt_ids))
    for p in unique_prompts:
        mask = torch.tensor(prompt_arr == p, device=dev, dtype=torch.bool)
        if mask.sum() > 1:
            Z_c[mask] = Z[mask] - Z[mask].mean(dim=0, keepdim=True)
        else:
            Z_c[mask] = Z[mask] - Z[mask]

    if hasattr(model, "predict_centered"):
        logits_centered = model.predict_centered(Z_c)
    else:
        if next(model.centered_head.parameters()).device != dev:
            model.centered_head = model.centered_head.to(dev)
        logits_centered = model.centered_head(Z_c)

    loss_centered = family_margin_cross_entropy(logits_centered, labels_dev, family_margin=family_margin, label_smoothing=label_smoothing, weight=weight)

    total_loss = loss_plain + float(centered_weight) * loss_centered
    if supcon_weight > 0.0:
        loss_sup = compute_supcon_loss(Z_c, labels_dev, prototypes=model.centered_head.weight.detach())
        total_loss = total_loss + float(supcon_weight) * loss_sup

    return total_loss, loss_plain, loss_centered


class PromptGroupBatchSampler(Sampler):
    """
    Groups samples sharing the same prompt_id into complete batches (one prompt group per step)
    to enable two-pass micro-batched within-prompt contrastive centering during Stage 2 training.
    """

    def __init__(self, prompt_ids: list[Any], batch_size: int = 18, shuffle: bool = True, seed: int = 42):
        self.prompt_to_indices: dict[Any, list[int]] = {}
        for idx, p in enumerate(prompt_ids):
            self.prompt_to_indices.setdefault(p, []).append(idx)
        self.prompts = list(self.prompt_to_indices.keys())
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.seed = seed
        self.epoch = 0

    def __iter__(self) -> Iterator[list[int]]:
        rng = np.random.RandomState(self.seed + self.epoch)
        prompts = list(self.prompts)
        if self.shuffle:
            rng.shuffle(prompts)
        for p in prompts:
            idxs = list(self.prompt_to_indices[p])
            if self.shuffle:
                rng.shuffle(idxs)
            yield idxs

    def __len__(self) -> int:
        return len(self.prompts)

    def set_epoch(self, epoch: int):
        self.epoch = epoch


class TextSequenceDataset(Dataset):
    def __init__(
        self, texts: list[str], labels: list[int] | None, prompt_ids: list[Any] | None, tok: Any, max_len: int = 447, end_id: int = 151643
    ):
        self.texts = texts
        self.labels = labels
        self.prompt_ids = prompt_ids
        self.tok = tok
        self.max_len = max_len
        self.end_id = end_id

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> dict[str, Any]:
        full_ids = self.tok(str(self.texts[idx]), add_special_tokens=False, truncation=False)["input_ids"]
        if len(full_ids) > self.max_len:
            head_len = self.max_len // 2
            tail_len = self.max_len - head_len
            raw_ids = full_ids[:head_len] + full_ids[-tail_len:]
        else:
            raw_ids = full_ids
        input_ids = raw_ids + [self.end_id]
        item: dict[str, Any] = {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.ones(len(input_ids), dtype=torch.long),
        }
        if self.labels is not None:
            item["label"] = torch.tensor(self.labels[idx], dtype=torch.long)
        if self.prompt_ids is not None:
            item["prompt_id"] = self.prompt_ids[idx]
        return item


def collate_text_batch(batch: list[dict[str, Any]], pad_token_id: int = 151643) -> dict[str, Any]:
    input_ids = [item["input_ids"] for item in batch]
    masks = [item.get("attention_mask", torch.ones_like(item["input_ids"])) for item in batch]
    padded_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=pad_token_id)
    padded_mask = torch.nn.utils.rnn.pad_sequence(masks, batch_first=True, padding_value=0)
    res: dict[str, Any] = {"input_ids": padded_ids, "attention_mask": padded_mask}
    if "label" in batch[0]:
        res["labels"] = torch.stack([item["label"] for item in batch])
    if "prompt_id" in batch[0]:
        res["prompt_ids"] = [item["prompt_id"] for item in batch]
    return res


class BucketedLengthBatchSampler(Sampler):
    """
    Batches sequences of similar lengths together to minimize padding waste,
    while shuffling batches for stochastic gradient descent.
    """

    def __init__(self, lengths: list[int], batch_size: int, bucket_size_multiplier: int = 50, shuffle: bool = True, seed: int = 42):
        self.lengths = lengths
        self.batch_size = batch_size
        self.bucket_size = batch_size * bucket_size_multiplier
        self.shuffle = shuffle
        self.seed = seed
        self.epoch = 0

    def __iter__(self) -> Iterator[list[int]]:
        rng = np.random.RandomState(self.seed + self.epoch)
        indices = np.arange(len(self.lengths))
        if self.shuffle:
            rng.shuffle(indices)

        batches = []
        for i in range(0, len(indices), self.bucket_size):
            bucket = indices[i : i + self.bucket_size]
            sorted_bucket = bucket[np.argsort([self.lengths[idx] for idx in bucket])]
            for b in range(0, len(sorted_bucket), self.batch_size):
                batches.append(list(sorted_bucket[b : b + self.batch_size]))

        if self.shuffle:
            rng.shuffle(batches)
        for batch in batches:
            yield batch

    def __len__(self) -> int:
        return math.ceil(len(self.lengths) / self.batch_size)

    def set_epoch(self, epoch: int):
        self.epoch = epoch


def train_prompt_group_step(
    model: MultiLayerLoRAClassifier,
    opt: torch.optim.Optimizer,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    labels: torch.Tensor,
    micro_batch_size: int = 2,
    centered_weight: float = 0.5,
    family_margin: float = 0.35,
    supcon_weight: float = 0.20,
    label_smoothing: float = 0.10,
    accum_steps: int = 1,
    compute_dtype: torch.dtype = torch.float16,
    weight: torch.Tensor | None = None,
) -> tuple[float, float, float]:
    """
    Two-pass micro-batched training step for a single prompt group.
    Pass 1: Computes exact within-prompt mean vector mu_prompt without gradients in chunks of 4.
    Pass 2: Computes plain and centered cross-entropy losses over micro-batches of micro_batch_size,
            accumulating gradients directly with dynamic sequence length slicing, AMP fp16/bf16,
            family margin penalties, hard-class loss upweighting, and supervised contrastive loss.
    """
    N = input_ids.shape[0]
    dev = input_ids.device
    labels_dev = labels.to(dev)

    # Pass 1: Compute mu_prompt with no_grad in chunks of 4
    z_chunks = []
    with torch.no_grad():
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available(), dtype=compute_dtype):
            for s in range(0, N, 4):
                e = min(s + 4, N)
                mb_mask = attention_mask[s:e]
                max_l = max(int(mb_mask.sum(dim=1).max().item()), 2)
                z_c = model.forward_features(input_ids[s:e, :max_l], mb_mask[:, :max_l], drop_first_token=True)
                z_chunks.append(z_c)
    Z_all_no_grad = torch.cat(z_chunks, dim=0)
    # Variable-Size Sub-Cluster Centering Augmentation:
    # When N >= 8, with 50% probability compute mu_prompt using a random subset of k ~ Uniform(4, min(N, 14))
    # rows from Z_all_no_grad so centered_head generalizes to smaller test-time prompt clusters.
    if N >= 8 and torch.rand(1).item() < 0.5:
        max_k = min(N, 14)
        k = int(torch.randint(4, max_k + 1, (1,)).item())
        sub_indices = torch.randperm(N, device=Z_all_no_grad.device)[:k]
        mu_prompt = Z_all_no_grad[sub_indices].mean(dim=0, keepdim=True)
    else:
        mu_prompt = Z_all_no_grad.mean(dim=0, keepdim=True)
    Z_group_centered = (Z_all_no_grad - mu_prompt).detach()
    del z_chunks

    # Pass 2: Autograd micro-batches with AMP
    total_loss_val = 0.0
    plain_loss_val = 0.0
    centered_loss_val = 0.0

    model.train()
    for s in range(0, N, micro_batch_size):
        e = min(s + micro_batch_size, N)
        mb_mask = attention_mask[s:e]
        max_l = max(int(mb_mask.sum(dim=1).max().item()), 2)
        mb_ids = input_ids[s:e, :max_l]
        mb_mask_slice = mb_mask[:, :max_l]

        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available(), dtype=compute_dtype):
            Z_mb = model.forward_features(mb_ids, mb_mask_slice, drop_first_token=True)
            mb_dev = Z_mb.device

            logits_p = model.predict_plain(Z_mb)
            loss_p = family_margin_cross_entropy(
                logits_p, labels_dev[s:e].to(mb_dev), family_margin=family_margin, label_smoothing=label_smoothing, weight=weight
            )

            Z_c_mb = Z_mb - mu_prompt.to(mb_dev)
            logits_c = model.predict_centered(Z_c_mb)
            loss_c = family_margin_cross_entropy(
                logits_c, labels_dev[s:e].to(mb_dev), family_margin=family_margin, label_smoothing=label_smoothing, weight=weight
            )

            loss_mb = loss_p + float(centered_weight) * loss_c
            if supcon_weight > 0.0:
                loss_sup = compute_supcon_loss(
                    Z_c_mb, labels_dev[s:e], temperature=0.10, prototypes=model.centered_head.weight.detach(), Z_group_centered=Z_group_centered
                )
                loss_mb = loss_mb + float(supcon_weight) * loss_sup

            weight_frac = (e - s) / float(N)
            scaled_loss = (loss_mb * weight_frac) / float(max(accum_steps, 1))

        scaled_loss.backward()

        total_loss_val += loss_mb.item() * weight_frac
        plain_loss_val += loss_p.item() * weight_frac
        centered_loss_val += loss_c.item() * weight_frac

    return total_loss_val, plain_loss_val, centered_loss_val


def fit_ridge_heads(
    Z: np.ndarray,
    y: np.ndarray,
    prompts: np.ndarray | None = None,
    alpha: float = 10.0,
    label_smoothing: float = 0.10,
    num_classes: int = K,
    scale_by_trace: bool = True,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Fits closed-form dual Ridge regression heads for plain and centered representations
    with label smoothing and feature-dimension trace scaling.
    Uses dual formulation (N x N) for ultra-fast, numerically stable solution.
    """
    N, D = Z.shape
    Y = np.zeros((N, num_classes), dtype=np.float32)
    for i, lab in enumerate(y):
        Y[i, lab] = 1.0
    if label_smoothing > 0.0:
        Y = (1.0 - label_smoothing) * Y + (label_smoothing / num_classes)

    # Plain head: Z_tilde = [Z, 1]
    Z_tilde = np.hstack([Z, np.ones((N, 1), dtype=np.float32)])
    K_mat = Z_tilde @ Z_tilde.T
    if scale_by_trace:
        mean_diag_p = float(np.mean(np.sum(Z * Z, axis=1)))
        eff_alpha_p = float(alpha) * max(mean_diag_p / 512.0, 1.0)
    else:
        eff_alpha_p = float(alpha)
    K_mat.flat[:: N + 1] += eff_alpha_p
    A = np.linalg.solve(K_mat, Y)
    W_tilde = (A.T @ Z_tilde).astype(np.float32)
    W_p = W_tilde[:, :-1]
    b_p = W_tilde[:, -1]

    # Centered head: Z_c
    Z_c = Z.copy()
    if prompts is not None:
        u_p = np.unique(prompts)
        for p in u_p:
            mask = prompts == p
            if mask.sum() > 1:
                Z_c[mask] = Z[mask] - Z[mask].mean(axis=0, keepdims=True)
            else:
                Z_c[mask] = 0.0

    Z_c_tilde = np.hstack([Z_c, np.ones((N, 1), dtype=np.float32)])
    K_mat_c = Z_c_tilde @ Z_c_tilde.T
    if scale_by_trace:
        mean_diag_c = float(np.mean(np.sum(Z_c * Z_c, axis=1)))
        eff_alpha_c = float(alpha) * max(mean_diag_c / 512.0, 1.0)
    else:
        eff_alpha_c = float(alpha)
    K_mat_c.flat[:: N + 1] += eff_alpha_c
    A_c = np.linalg.solve(K_mat_c, Y)
    W_c_tilde = (A_c.T @ Z_c_tilde).astype(np.float32)
    W_c = W_c_tilde[:, :-1]
    b_c = W_c_tilde[:, -1]

    return W_p, b_p, W_c, b_c


def compute_grouped_inference_logits(
    Z: np.ndarray,
    plain_weight: np.ndarray,
    plain_bias: np.ndarray,
    centered_weight: np.ndarray,
    centered_bias: np.ndarray,
    groups: list[Any] | np.ndarray | None = None,
    blend_centered: float = 0.20,
) -> dict[str, np.ndarray]:
    """
    Computes plain, gated prompt-group centered, and blended logits for inference.
    When groups are available, only centers groups that have >= 6 samples and >= 4 distinct
    plain-head predicted classes to prevent self-subtraction on homogeneous/small clusters.
    Uses shrinkage centering (0.75x mean subtraction) for valid groups.
    When groups are absent and batch size >= 18, centers globally across the batch.
    Computes prior-centered Sinkhorn-balanced probabilities on both logits_plain and logits_blend.
    """
    logits_plain = Z @ plain_weight.T + plain_bias
    if groups is not None and len(groups) == len(Z):
        Z_c = Z.copy()
        singleton_mask = np.zeros(len(Z), dtype=bool)
        grps_arr = np.asarray(groups)
        plain_preds = logits_plain.argmax(axis=1)
        for g in np.unique(grps_arr):
            mask = (grps_arr == g)
            distinct_classes = len(np.unique(plain_preds[mask]))
            if mask.sum() >= 6 and distinct_classes >= 4:
                Z_c[mask] = Z[mask] - 0.75 * Z[mask].mean(axis=0, keepdims=True)
            else:
                singleton_mask[mask] = True
        logits_centered = Z_c @ centered_weight.T + centered_bias
        if np.any(singleton_mask):
            logits_centered[singleton_mask] = logits_plain[singleton_mask]
    elif groups is None and len(Z) >= 18:
        Z_c = Z - 0.75 * Z.mean(axis=0, keepdims=True)
        logits_centered = Z_c @ centered_weight.T + centered_bias
    else:
        logits_centered = logits_plain.copy()

    # Standardize per-sample logit scale before blending
    row_mean_p = logits_plain.mean(axis=1, keepdims=True)
    row_std_p = np.clip(logits_plain.std(axis=1, keepdims=True), 1e-4, None)
    norm_plain = (logits_plain - row_mean_p) / row_std_p

    row_mean_c = logits_centered.mean(axis=1, keepdims=True)
    row_std_c = np.clip(logits_centered.std(axis=1, keepdims=True), 1e-4, None)
    norm_centered = (logits_centered - row_mean_c) / row_std_c

    logits_blend = (1.0 - blend_centered) * norm_plain + blend_centered * norm_centered

    probs_plain_sinkhorn = _apply_prior_and_sinkhorn(logits_plain)
    probs_sinkhorn = _apply_prior_and_sinkhorn(logits_blend)

    return {
        "logits_plain": logits_plain,
        "logits_centered": logits_centered,
        "logits_blend": logits_blend,
        "probs_plain_sinkhorn": probs_plain_sinkhorn,
        "probs_sinkhorn": probs_sinkhorn,
    }


def _blend_head(head_linear: nn.Linear, W_r: np.ndarray, b_r: np.ndarray, w_neural: float = 0.65, w_ridge: float = 0.35) -> None:
    """Blends neural head with closed-form ridge head after matching Frobenius norm."""
    with torch.no_grad():
        W_n = head_linear.weight.detach().cpu().float().numpy()
        b_n = head_linear.bias.detach().cpu().float().numpy()
        norm_n = float(np.linalg.norm(W_n))
        norm_r = float(np.linalg.norm(W_r))
        scale = (norm_n / norm_r) if (norm_r > 1e-6 and norm_n > 1e-6) else 1.0
        W_r_scaled = W_r * scale
        b_r_centered = (b_r - b_r.mean()) * scale + b_n.mean()
        W_blend = w_neural * W_n + w_ridge * W_r_scaled
        b_blend = w_neural * b_n + w_ridge * b_r_centered
        head_linear.weight.copy_(torch.from_numpy(W_blend).to(head_linear.weight.device, dtype=head_linear.weight.dtype))
        head_linear.bias.copy_(torch.from_numpy(b_blend).to(head_linear.bias.device, dtype=head_linear.bias.dtype))


def evaluate_stage2_oof_cv(
    Z: np.ndarray,
    y: np.ndarray,
    prompts: np.ndarray,
    n_splits: int = 5,
    alpha: float = 10.0,
    label_smoothing: float = 0.10,
    num_classes: int = K,
    class_labels: list[str] | None = None,
    output_dir: Path | None = None,
) -> dict[str, Any]:
    """
    Executes 5-Stage GroupKFold Cross-Validation on Stage 2 neural representations
    grouped strictly by prompt_id to evaluate out-of-fold generalization and check overfitting.

    Why This Is Done:
    -----------------
    In authorship attribution, autoregressive models can memorize prompt topics rather than
    learning stylistic traits. Evaluating on the training prompts produces artificially high
    accuracy (e.g. 95%+). By partitioning prompts into 5 disjoint folds, we evaluate on
    passages from prompts the heads never saw during training, giving an honest, unbiased
    out-of-fold (OOF) accuracy and quantifying the exact Overfitting Gap (Train BAcc - Val BAcc).
    """
    u_prompts = np.unique(prompts)
    actual_splits = min(n_splits, len(u_prompts))
    if actual_splits < 2:
        print(f"[Stage 2 CV] Insufficient prompt groups ({len(u_prompts)}) for cross-validation; skipping.")
        return {}

    labels = class_labels or list(LABELS[:num_classes])
    gkf = GroupKFold(n_splits=actual_splits)

    oof_plain_lp = np.zeros((len(y), num_classes), dtype=np.float32)
    oof_centered_lp = np.zeros((len(y), num_classes), dtype=np.float32)
    oof_blend_lp = np.zeros((len(y), num_classes), dtype=np.float32)
    fold_results = []

    print("\n==========================================================================================")
    print(f" STAGE 2: {actual_splits}-STAGE GROUP-KFOLD CV & OVERFITTING AUDIT (grouped by prompt_id)")
    print("==========================================================================================")
    print(" Fold   | Train Prompts | Val Prompts | Train BAcc | Val Plain  | Val Center | Val Sink   | Overfit Gap")
    print("------------------------------------------------------------------------------------------")

    for fold_idx, (tr_idx, val_idx) in enumerate(gkf.split(Z, y, groups=prompts)):
        Z_tr, y_tr, p_tr = Z[tr_idx], y[tr_idx], prompts[tr_idx]
        Z_val, y_val, p_val = Z[val_idx], y[val_idx], prompts[val_idx]

        W_p, b_p, W_c, b_c = fit_ridge_heads(Z_tr, y_tr, p_tr, alpha=alpha, label_smoothing=label_smoothing, num_classes=num_classes)

        # Train evaluation (to measure overfitting gap)
        tr_plain = Z_tr @ W_p.T + b_p
        Z_tr_c = Z_tr.copy()
        for p in np.unique(p_tr):
            m = p_tr == p
            if m.sum() > 1:
                Z_tr_c[m] -= Z_tr_c[m].mean(axis=0, keepdims=True)
            else:
                Z_tr_c[m] = 0.0
        tr_centered = Z_tr_c @ W_c.T + b_c
        tr_blend = 0.5 * tr_plain + 0.5 * tr_centered
        train_bacc = float(balanced_accuracy_score(y_tr, tr_blend.argmax(axis=1)))

        # Validation evaluation (strictly unseen prompt groups)
        val_plain = Z_val @ W_p.T + b_p
        Z_val_c = Z_val.copy()
        singleton_mask = np.zeros(len(p_val), dtype=bool)
        for p in np.unique(p_val):
            m = p_val == p
            if m.sum() > 1:
                Z_val_c[m] -= Z_val_c[m].mean(axis=0, keepdims=True)
            else:
                Z_val_c[m] = 0.0
                singleton_mask[m] = True
        val_centered = Z_val_c @ W_c.T + b_c
        if np.any(singleton_mask):
            val_centered[singleton_mask] = val_plain[singleton_mask]
        val_blend = 0.5 * val_plain + 0.5 * val_centered

        val_plain_bacc = float(balanced_accuracy_score(y_val, val_plain.argmax(axis=1)))
        val_center_bacc = float(balanced_accuracy_score(y_val, val_centered.argmax(axis=1)))
        val_blend_bacc = float(balanced_accuracy_score(y_val, val_blend.argmax(axis=1)))

        # Val Soft Sinkhorn (scale ridge outputs to standard logit std ~4.0 before softmax)
        val_std = float(np.std(val_blend, axis=1).mean())
        val_scale = (4.0 / val_std) if val_std > 1e-4 else 1.0
        P_val = softmax(val_blend * val_scale, axis=1)
        P_val_sk = sinkhorn_balancing(P_val, iters=60, sinkhorn_strength=0.65, target_mix=0.35)
        val_sink_bacc = float(balanced_accuracy_score(y_val, P_val_sk.argmax(axis=1)))

        gap = train_bacc - val_sink_bacc

        oof_plain_lp[val_idx] = val_plain
        oof_centered_lp[val_idx] = val_centered
        oof_blend_lp[val_idx] = val_blend

        fold_res = {
            "fold": fold_idx + 1,
            "train_prompts": int(len(np.unique(p_tr))),
            "val_prompts": int(len(np.unique(p_val))),
            "train_bacc": train_bacc,
            "val_plain_bacc": val_plain_bacc,
            "val_center_bacc": val_center_bacc,
            "val_blend_bacc": val_blend_bacc,
            "val_sink_bacc": val_sink_bacc,
            "overfit_gap": gap,
        }
        fold_results.append(fold_res)

        print(
            f" Fold {fold_idx + 1:<2} | {len(np.unique(p_tr)):<13} | {len(np.unique(p_val)):<11} | "
            f"{train_bacc * 100:5.2f}%    | {val_plain_bacc * 100:5.2f}%    | "
            f"{val_center_bacc * 100:5.2f}%    | {val_sink_bacc * 100:5.2f}%   | {gap * 100:+5.2f}%"
        )

    # Global OOF Metrics across all N samples
    oof_plain_bacc = float(balanced_accuracy_score(y, oof_plain_lp.argmax(axis=1)))
    oof_center_bacc = float(balanced_accuracy_score(y, oof_centered_lp.argmax(axis=1)))
    oof_blend_bacc = float(balanced_accuracy_score(y, oof_blend_lp.argmax(axis=1)))

    oof_std = float(np.std(oof_blend_lp, axis=1).mean())
    oof_scale = (4.0 / oof_std) if oof_std > 1e-4 else 1.0
    P_oof = softmax(oof_blend_lp * oof_scale, axis=1)
    P_oof_sk = sinkhorn_balancing(P_oof, iters=60, sinkhorn_strength=0.65, target_mix=0.35)
    oof_sink_bacc = float(balanced_accuracy_score(y, P_oof_sk.argmax(axis=1)))

    mean_tr_bacc = float(np.mean([f["train_bacc"] for f in fold_results]))
    total_overfit_gap = mean_tr_bacc - oof_sink_bacc

    if total_overfit_gap <= 0.08:
        verdict = "LOW (Excellent Generalization across Unseen Prompts)"
    elif total_overfit_gap <= 0.18:
        verdict = "MODERATE (Normal Stylistic Domain Variance)"
    else:
        verdict = "HIGH (Prompt Memorization Risk - Increase supcon/family margin)"

    print("------------------------------------------------------------------------------------------")
    print(
        f" OOF   | {len(u_prompts):<13} | {len(u_prompts):<11} | "
        f"{mean_tr_bacc * 100:5.2f}%    | {oof_plain_bacc * 100:5.2f}%    | "
        f"{oof_center_bacc * 100:5.2f}%    | {oof_sink_bacc * 100:5.2f}%   | {total_overfit_gap * 100:+5.2f}%"
    )
    print("==========================================================================================")
    print(f" Overfitting Assessment: {verdict}")
    print("==========================================================================================\n")

    report = {
        "n_splits": actual_splits,
        "n_samples": int(len(y)),
        "n_prompts": int(len(u_prompts)),
        "mean_train_bacc": mean_tr_bacc,
        "oof_plain_bacc": oof_plain_bacc,
        "oof_center_bacc": oof_center_bacc,
        "oof_blend_bacc": oof_blend_bacc,
        "oof_sink_bacc": oof_sink_bacc,
        "overfit_gap": total_overfit_gap,
        "verdict": verdict,
        "folds": fold_results,
    }

    if output_dir is not None:
        out_p = Path(output_dir)
        out_p.mkdir(parents=True, exist_ok=True)
        rep_file = out_p / "stage2_cv_report.json"
        with open(rep_file, "w", encoding="utf-8") as f:
            json.dump(report, f, indent=2)
        print(f"[Stage 2 CV] Saved cross-validation and overfitting report to: {rep_file}")

        oof_npz = out_p / "stage2_oof_predictions.npz"
        np.savez_compressed(
            str(oof_npz),
            oof_plain_lp=oof_plain_lp,
            oof_centered_lp=oof_centered_lp,
            oof_blend_lp=oof_blend_lp,
            oof_sink_probs=P_oof_sk,
            y_true=y,
            prompts=prompts,
            classes=np.array(labels),
        )
        print(f"[Stage 2 CV] Saved out-of-fold predictions to: {oof_npz}")

    return report


def export_compatible_checkpoint(
    model: MultiLayerLoRAClassifier,
    output_dir: Path,
    layers: list[int],
    keep: int,
    max_len: int = 447,
    end_id: int = 151643,
    backbone_name: str | None = None,
    Z_train: np.ndarray | None = None,
    y_train: np.ndarray | None = None,
    prompts_train: np.ndarray | None = None,
    class_labels: list[str] | None = None,
    target_family: str | None = None,
) -> Path:
    """
    Exports standard checkpoint artifacts strictly compatible with inference loaders:
    1. adapter/ (LoRA adapter weights and config)
    2. heads.safetensors (norms.{i}.weight, norms.{i}.bias, plain.weight, plain.bias, centered.weight, centered.bias)
    3. meta.json (keep, layers, max_len, end_id, labels, backbone, classes, target_family)
    4. wPCA_bank.npz (shared Gaussian covariance / whitening basis)
    """
    from safetensors.torch import save_file

    out_p = Path(output_dir)
    out_p.mkdir(parents=True, exist_ok=True)

    # 1. Adapter weights
    adapter_dir = out_p / "adapter"
    adapter_dir.mkdir(parents=True, exist_ok=True)
    if hasattr(model.backbone, "save_pretrained"):
        model.backbone.save_pretrained(str(adapter_dir))
    elif hasattr(model, "save_pretrained"):
        model.save_pretrained(str(adapter_dir))

    # 2. Heads dictionary
    heads_dict = {}
    for i, norm in enumerate(model.norms):
        heads_dict[f"norms.{i}.weight"] = norm.weight.detach().cpu().contiguous()
        if hasattr(norm, "bias") and norm.bias is not None:
            heads_dict[f"norms.{i}.bias"] = norm.bias.detach().cpu().contiguous()

    heads_dict["plain.weight"] = model.plain_head.weight.detach().cpu().contiguous()
    heads_dict["plain.bias"] = model.plain_head.bias.detach().cpu().contiguous()
    heads_dict["centered.weight"] = model.centered_head.weight.detach().cpu().contiguous()
    heads_dict["centered.bias"] = model.centered_head.bias.detach().cpu().contiguous()

    save_file(heads_dict, str(out_p / "heads.safetensors"))

    # 3. Meta JSON
    active_labels = list(class_labels) if class_labels is not None else list(LABELS)
    meta = {
        "labels": active_labels,
        "classes": active_labels,
        "target_family": str(target_family or ""),
        "backbone": str(backbone_name) if backbone_name else "",
        "cfg": {
            "keep": int(keep),
            "layers": [int(x) for x in layers],
            "max_len": int(max_len),
            "end_token_id": int(end_id),
            "backbone": str(backbone_name) if backbone_name else "",
            "target_family": str(target_family or ""),
        },
    }
    (out_p / "meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

    # 4. wPCA Bank precomputation (only when standard 18-class)
    if len(active_labels) == K and Z_train is not None and y_train is not None and prompts_train is not None:
        d_target = min(256, Z_train.shape[1])
        res = fit_shared_gaussian_lowram(Z_train, y_train, prompts_train, d_target=d_target)
        np.savez_compressed(
            str(out_p / "wPCA_bank.npz"),
            mx=res["mx"].astype(np.float32),
            sx=res["sx"].astype(np.float32),
            mu=res["mx"].astype(np.float32),
            sd=res["sx"].astype(np.float32),
            W=res["W"].astype(np.float32),
            m=res["m"].astype(np.float32),
            lam=res["lam"].astype(np.float32),
            V_top64=res["V_top64"].astype(np.float32),
            layers=np.array(layers, dtype=np.int32),
        )

    return out_p


def _get_loraplus_param_groups(
    model: MultiLayerLoRAClassifier,
    lr_lora: float,
    lr_head: float,
    lora_plus_ratio: float = 2.5,
    include_centered_head: bool = False,
) -> list[dict[str, Any]]:
    """
    Builds LoRA+ asymmetric parameter groups where lora_B learns at lora_plus_ratio * lr_lora
    relative to lora_A to accelerate feature learning in high-dimensional hidden layers.
    """
    lora_a_params = []
    lora_b_params = []
    other_backbone_params = []
    for name, param in model.backbone.named_parameters():
        if not param.requires_grad:
            continue
        name_lower = name.lower()
        if "lora_b" in name_lower:
            lora_b_params.append(param)
        elif "lora_a" in name_lower:
            lora_a_params.append(param)
        else:
            other_backbone_params.append(param)

    head_params = list(model.norms.parameters()) + list(model.plain_head.parameters())
    if include_centered_head and hasattr(model, "centered_head"):
        head_params += list(model.centered_head.parameters())

    groups = []
    if lora_a_params:
        groups.append({"params": lora_a_params, "lr": float(lr_lora)})
    if lora_b_params:
        groups.append({"params": lora_b_params, "lr": float(lr_lora) * float(lora_plus_ratio)})
    if other_backbone_params:
        groups.append({"params": other_backbone_params, "lr": float(lr_lora)})
    if head_params:
        groups.append({"params": head_params, "lr": float(lr_head)})
    return groups


def run_training_pipeline(
    train_cfg: dict[str, Any] | None = None,
    step_callback: Any | None = None,
) -> Path:
    """
    Executes complete Two-Stage Multi-Layer LoRA training pipeline:
    Stage 1: Multi-corpus pretraining on up to 90k passages with family margin loss.
    Stage 2: Prompt-grouped contrastive fine-tuning on train.csv with SupCon loss.
    """
    _setup_training_env()

    from peft import LoraConfig, get_peft_model
    from transformers import (
        AutoConfig,
        AutoModel,
        AutoTokenizer,
        get_cosine_schedule_with_warmup,
    )

    if train_cfg is None:
        train_cfg = {}

    backbone_name = train_cfg.get("backbone", "Qwen/Qwen3-14B-Base")
    keep_layers = int(train_cfg.get("keep_layers", 27))
    layers = list(train_cfg.get("layers", [2, 4, 6, 9, 13, 17, 22, 27]))
    max_len = int(train_cfg.get("max_len", 384))
    end_id = int(train_cfg.get("end_id", 151643))
    out_dir = Path(train_cfg.get("output_dir", "export_champion_qwen3_14b"))
    family_margin = float(train_cfg.get("family_margin", 0.35))
    supcon_weight = float(train_cfg.get("supcon_weight", 0.20))
    is_deberta = "deberta" in backbone_name.lower()

    input_dir = Path(train_cfg.get("input_dir", "/kaggle/input"))

    # Preserve existing 18-class in-house champion checkpoint if present in input_dir
    working_champ = Path("/kaggle/working/export_champion_qwen3_14b")
    if input_dir.exists() and not working_champ.exists():
        for cand in input_dir.rglob("export_champion_qwen3_14b"):
            if cand.is_dir() and (cand / "heads.safetensors").exists():
                try:
                    shutil.copytree(str(cand), str(working_champ))
                    print(f"[Trainer] Preserved existing in-house checkpoint from {cand} to {working_champ}")
                    break
                except Exception as e:
                    print(f"[Trainer] Note on preserving in-house checkpoint: {e}")

    # Sequential execution of specialist jobs if configured
    if train_cfg.get("specialist_jobs") and isinstance(train_cfg["specialist_jobs"], list):
        final_out = out_dir
        for job in train_cfg["specialist_jobs"]:
            job_cfg = copy.deepcopy(train_cfg)
            job_cfg.pop("specialist_jobs", None)
            for k, v in job.items():
                if isinstance(v, dict) and isinstance(job_cfg.get(k), dict):
                    job_cfg[k] = {**job_cfg[k], **v}
                else:
                    job_cfg[k] = v
            final_out = run_training_pipeline(job_cfg, step_callback=step_callback)
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        return final_out

    target_family = train_cfg.get("target_family")
    is_specialist = bool(target_family and target_family in FAMILIES)
    if is_specialist:
        fam_cols = FAMILIES[target_family]
        active_labels = [LABELS[i] for i in fam_cols]
        sub_lab2i = {m: i for i, m in enumerate(active_labels)}
        num_classes = len(active_labels)
        print(f"[Trainer] Training {target_family} family specialist with {num_classes} classes: {active_labels}")
    else:
        active_labels = list(LABELS)
        sub_lab2i = dict(LAB2I)
        num_classes = K
    local_base = None
    if input_dir.exists():
        if is_deberta:
            for cfg_p in input_dir.rglob("config.json"):
                try:
                    with open(cfg_p, "r", encoding="utf-8") as f:
                        c_data = json.load(f)
                    if "deberta" in str(c_data.get("model_type", "")).lower():
                        local_base = cfg_p.parent
                        break
                except Exception:
                    continue
        elif "2.5" in backbone_name.lower() or "qwen25" in backbone_name.lower():
            local_base = find_base_model(input_dir, "qwen2", 5120, 48) or find_base_model(input_dir, "qwen2", 3584, 28)
        elif "14b" in backbone_name.lower():
            local_base = find_base_model(input_dir, "qwen3", 5120, 40) or find_base_model(input_dir, "qwen2", 5120, 48)
        elif "8b" in backbone_name.lower():
            local_base = find_base_model(input_dir, "qwen3", 4096, 36)
    model_source = str(local_base) if local_base else backbone_name

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    n_gpus = torch.cuda.device_count()

    # Hardware capability detection: Ampere/Hopper (A100/H100) vs. Turing/earlier (T4)
    is_ampere_or_newer = False
    has_high_vram_single_gpu = False
    total_vram_gb = 0.0
    if torch.cuda.is_available():
        major_cap = torch.cuda.get_device_capability(0)[0]
        is_ampere_or_newer = major_cap >= 8
        total_vram_bytes = torch.cuda.get_device_properties(0).total_memory
        total_vram_gb = total_vram_bytes / (1024**3)
        has_high_vram_single_gpu = bool(n_gpus == 1 and total_vram_bytes >= 40 * 1024**3)

    use_bf16 = bool(train_cfg.get("bf16", False) or is_ampere_or_newer)
    compute_dtype = torch.bfloat16 if use_bf16 else (torch.float16 if torch.cuda.is_available() else torch.float32)

    use_fa2 = False
    if has_high_vram_single_gpu and use_bf16 and not is_deberta:
        try:
            import flash_attn  # noqa: F401

            use_fa2 = True
        except ImportError:
            use_fa2 = False

    print(
        f"[Trainer] Device: {device}, Available GPUs: {n_gpus} (VRAM: {total_vram_gb:.1f} GiB), "
        f"Dtype: {compute_dtype}, FlashAttn-2: {use_fa2}, Model Source: {model_source}, Keep: {keep_layers}"
    )

    train_path, test_path, pool_file = resolve_dataset_paths(input_dir if input_dir.exists() else Path("."))
    all_pools = resolve_all_pool_files(input_dir if input_dir.exists() else Path("."))

    tok = AutoTokenizer.from_pretrained(model_source, trust_remote_code=True, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    vocab_sz = len(tok)
    if end_id >= vocab_sz:
        safe_end_id = int(tok.eos_token_id or tok.sep_token_id or 2)
        print(f"[Trainer] Config end_id ({end_id}) >= vocab size ({vocab_sz}); adjusting to {safe_end_id}")
        end_id = safe_end_id

    base_cfg = AutoConfig.from_pretrained(model_source, trust_remote_code=True)
    base_cfg.num_hidden_layers = keep_layers
    base_cfg.use_cache = False
    if getattr(base_cfg, "layer_types", None) is not None:
        base_cfg.layer_types = base_cfg.layer_types[:keep_layers]

    extra_model_kwargs = {}
    if use_fa2:
        extra_model_kwargs["attn_implementation"] = "flash_attention_2"

    # Memory strategy: Single high-VRAM GPU (A100/H100) vs. Multi-GPU 2xT4 balancing vs. Single T4 vs. CPU
    if has_high_vram_single_gpu:
        print(f"[Trainer] High-VRAM single GPU detected ({total_vram_gb:.1f} GiB). Loading directly on {device} in {compute_dtype}...")
        base_model = AutoModel.from_pretrained(
            model_source,
            config=base_cfg,
            torch_dtype=compute_dtype,
            low_cpu_mem_usage=True,
            trust_remote_code=True,
            **extra_model_kwargs,
        ).to(device)
    elif n_gpus >= 2:
        max_memory = {0: "12.5GiB", 1: "13.5GiB"}
        base_model = AutoModel.from_pretrained(
            model_source,
            config=base_cfg,
            torch_dtype=compute_dtype,
            device_map="auto",
            max_memory=max_memory,
            low_cpu_mem_usage=True,
            trust_remote_code=True,
        )
    elif n_gpus == 1:
        base_model = AutoModel.from_pretrained(
            model_source,
            config=base_cfg,
            torch_dtype=compute_dtype,
            device_map="auto",
            low_cpu_mem_usage=True,
            trust_remote_code=True,
        )
    else:
        base_model = AutoModel.from_pretrained(
            model_source,
            config=base_cfg,
            torch_dtype=torch.float32,
            low_cpu_mem_usage=True,
            trust_remote_code=True,
        )

    try:
        import peft.import_utils
        import peft.tuners.lora.torchao

        peft.import_utils.is_torchao_available = lambda: False
        peft.tuners.lora.torchao.is_torchao_available = lambda: False
    except Exception:
        pass

    _disable_final_norm(base_model)

    if hasattr(base_model, "enable_input_require_grads"):
        base_model.enable_input_require_grads()

    linear_mod_names = set()
    for name, module in base_model.named_modules():
        if isinstance(module, nn.Linear):
            linear_mod_names.add(name.split(".")[-1])

    configured_targets = list(train_cfg.get("lora_target_modules", []))
    if not configured_targets:
        configured_targets = (
            ["query_proj", "key_proj", "value_proj", "dense"]
            if is_deberta
            else ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
        )

    active_targets = [m for m in configured_targets if m in linear_mod_names]
    if not active_targets:
        active_targets = list(linear_mod_names)[:7] if linear_mod_names else configured_targets
        print(f"[Trainer] Configured LoRA targets not found; falling back to: {active_targets}")
    else:
        print(f"[Trainer] Active LoRA target modules: {active_targets}")

    lora_cfg = LoraConfig(
        r=int(train_cfg.get("lora_r", 32)),
        lora_alpha=int(train_cfg.get("lora_alpha", 64)),
        lora_dropout=float(train_cfg.get("lora_dropout", 0.05)),
        target_modules=active_targets,
        bias="none",
        task_type="FEATURE_EXTRACTION",
    )
    base_model = get_peft_model(base_model, lora_cfg)

    if hasattr(base_model, "gradient_checkpointing_enable"):
        try:
            base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        except TypeError:
            base_model.gradient_checkpointing_enable()
    for mod in base_model.modules():
        if hasattr(mod, "gradient_checkpointing"):
            mod.gradient_checkpointing = True

    model = MultiLayerLoRAClassifier(base_model, hidden_size=base_cfg.hidden_size, layers=layers, num_classes=num_classes)
    # If device_map was used, keep classification heads and layer norms on the main device (device / cuda:0)
    # to avoid unnecessary PCI-e device thrashing while preserving device_map on the backbone.
    if hasattr(base_model, "hf_device_map") and len(getattr(base_model, "hf_device_map", {})) > 1:
        model.plain_head = model.plain_head.to(device)
        model.centered_head = model.centered_head.to(device)
        model.dropouts = model.dropouts.to(device)
        for idx in range(len(model.norms)):
            model.norms[idx] = model.norms[idx].to(device)
    else:
        model = model.to(device)

    # Stage 1: Pool Pretraining
    stage1_cfg = train_cfg.get("stage1", {})
    if stage1_cfg.get("enabled", True) and (all_pools or pool_file):
        max_stage1_hours = float(stage1_cfg.get("max_hours", 12.0))
        s1_start = time.time()
        max_s1_sec = max_stage1_hours * 3600.0
        pool_list = all_pools if all_pools else ([pool_file] if pool_file else [])
        print(f"[Trainer] Stage 1 Pretraining on {len(pool_list)} pool file(s)...")
        dfs = []
        pool_source_counts = {}
        for p in pool_list:
            try:
                if str(p).endswith(".parquet"):
                    df_one = pd.read_parquet(p, columns=["text", "model"])
                else:
                    df_one = pd.read_csv(p, usecols=["text", "model"])
                pool_source_counts[Path(p).name] = len(df_one)
                dfs.append(df_one)
            except Exception as e:
                print(f"[Trainer] Could not read pool {p}: {e}")

        total_raw_rows = sum(len(df) for df in dfs)
        if dfs:
            df_p = pd.concat(dfs, ignore_index=True)
        else:
            df_p = pd.DataFrame(columns=["text", "model"])

        df_p = df_p.dropna(subset=["text", "model"])
        df_p["text"] = df_p["text"].astype(str)
        df_p = df_p[df_p["text"].str.strip().str.len() >= 40]
        df_p = df_p[df_p["model"].isin(active_labels)]

        df_p["_hash"] = [_text_hash(t) for t in df_p["text"]]
        df_p = df_p.drop_duplicates(subset=["_hash"]).reset_index(drop=True)
        dedup_rows = len(df_p)

        # Build unseen holdout to quarantine holdout passages from Stage 1 pretraining
        quarantine_dropped = 0
        try:
            _, holdout_hashes = build_unseen_holdout_dataset(input_dir, train_path)
            if holdout_hashes:
                mask_clean = ~df_p["_hash"].isin(holdout_hashes)
                quarantine_dropped = int(len(df_p) - mask_clean.sum())
                if quarantine_dropped > 0:
                    print(f"[Trainer] Quarantined {quarantine_dropped} pool rows that overlapped with unseen holdout.")
                df_p = df_p[mask_clean].reset_index(drop=True)
        except Exception as e:
            print(f"[Trainer] Note on holdout quarantine: {e}")

        # Sacred Data Integrity Report before Stage 1 begins
        word_lens = [len(t.split()) for t in df_p["text"]]
        tok_lens = [min(len(t) // 4 + 1, max_len) for t in df_p["text"]]
        class_dist = df_p["model"].value_counts().to_dict()

        def _calc_percentiles(vals: list[int]) -> dict[str, float]:
            if not vals:
                return {}
            v = np.array(vals)
            return {
                "p10": float(np.percentile(v, 10)),
                "p25": float(np.percentile(v, 25)),
                "p50": float(np.percentile(v, 50)),
                "p75": float(np.percentile(v, 75)),
                "p90": float(np.percentile(v, 90)),
                "p99": float(np.percentile(v, 99)),
            }

        audit_report = {
            "source_files": pool_source_counts,
            "total_raw_rows": int(total_raw_rows),
            "deduplicated_rows": int(dedup_rows),
            "quarantined_holdout_rows": int(quarantine_dropped),
            "retained_pool_rows": int(len(df_p)),
            "class_distribution": {str(k): int(v) for k, v in class_dist.items()},
            "word_length_percentiles": _calc_percentiles(word_lens),
            "token_length_percentiles": _calc_percentiles(tok_lens),
        }
        out_dir.mkdir(parents=True, exist_ok=True)
        audit_file = out_dir / "data_audit_report.json"
        with open(audit_file, "w", encoding="utf-8") as f:
            json.dump(audit_report, f, indent=2)

        print("\n==========================================================================================")
        print("                         SACRED DATA INTEGRITY AUDIT REPORT                               ")
        print("==========================================================================================")
        print(f" Source Files Ingested ({len(pool_source_counts)}):")
        for fname, cnt in pool_source_counts.items():
            print(f"   - {fname:<32}: {cnt:>8,} rows")
        print(f" Total Raw Pool Rows     : {total_raw_rows:,}")
        print(f" Deduplicated Pool Rows  : {dedup_rows:,}")
        print(f" Quarantined Holdout Rows: {quarantine_dropped:,}")
        print(f" Clean Retained Pool Rows: {len(df_p):,}")
        print(" Per-Class Distribution (Active Models):")
        for lbl in active_labels:
            c_cnt = class_dist.get(lbl, 0)
            pct = (c_cnt / max(len(df_p), 1)) * 100
            print(f"   - {lbl:<22}: {c_cnt:>6,} ({pct:5.2f}%)")
        w_p = audit_report["word_length_percentiles"]
        t_p = audit_report["token_length_percentiles"]
        if w_p:
            print(
                f" Word Lengths (words)   : p10={w_p['p10']:.0f} | p25={w_p['p25']:.0f} | p50={w_p['p50']:.0f} | p75={w_p['p75']:.0f} | p90={w_p['p90']:.0f} | p99={w_p['p99']:.0f}"
            )
        if t_p:
            print(
                f" Token Lengths (tokens) : p10={t_p['p10']:.0f} | p25={t_p['p25']:.0f} | p50={t_p['p50']:.0f} | p75={t_p['p75']:.0f} | p90={t_p['p90']:.0f} | p99={t_p['p99']:.0f}"
            )
        print(f" Report Saved To        : {audit_file}")
        print("==========================================================================================\n")

        # Raise default max_samples to 150000 and upweight HARD_CLASS_INDICES by 1.30x
        max_s = int(stage1_cfg.get("max_samples", 2000 * num_classes if is_specialist else 150000))
        hard_set = set(HARD_CLASS_INDICES)
        if len(df_p) > max_s:
            cls_multipliers = {lbl: (1.30 if LAB2I[lbl] in hard_set else 1.0) for lbl in active_labels}
            total_multiplier = sum(cls_multipliers.values())
            target_per_c = {lbl: int(max_s * (cls_multipliers[lbl] / total_multiplier)) for lbl in active_labels}
            sampled = []
            remainder = []
            for lbl, group in df_p.groupby("model"):
                target_k = target_per_c.get(lbl, max_s // num_classes)
                if len(group) <= target_k:
                    sampled.append(group)
                else:
                    s_grp = group.sample(n=target_k, random_state=42)
                    sampled.append(s_grp)
                    remainder.append(group.drop(s_grp.index))
            res_df = pd.concat(sampled, ignore_index=True)
            needed = max_s - len(res_df)
            if needed > 0 and remainder:
                rem_df = pd.concat(remainder, ignore_index=True)
                take_extra = rem_df.sample(n=min(needed, len(rem_df)), random_state=42)
                res_df = pd.concat([res_df, take_extra], ignore_index=True)
            df_p = res_df.sample(frac=1.0, random_state=42).reset_index(drop=True)
        else:
            df_p = df_p.sample(frac=1.0, random_state=42).reset_index(drop=True)

        lengths = [min(len(t) // 4 + 1, max_len) for t in df_p["text"]]
        bs1 = int(stage1_cfg.get("batch_size", 2))
        accum1 = int(stage1_cfg.get("gradient_accumulation_steps", 8))
        if torch.cuda.is_available():
            vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
            if vram_gb < 24.0 and bs1 > 4:
                old_bs1 = bs1
                bs1 = 2
                accum1 = max(accum1 * (old_bs1 // 2), 8)
                print(f"[Trainer] Low VRAM GPU ({vram_gb:.1f} GiB) detected: adjusted Stage 1 batch_size from {old_bs1} to {bs1}, accum_steps to {accum1}")
        epochs1 = int(stage1_cfg.get("epochs", 2 if not is_specialist else 1))
        ds1 = TextSequenceDataset(df_p["text"].tolist(), [sub_lab2i[m] for m in df_p["model"]], None, tok, max_len=max_len, end_id=end_id)
        bucket_sampler = BucketedLengthBatchSampler(lengths, batch_size=bs1, shuffle=True)
        loader1 = DataLoader(ds1, batch_sampler=bucket_sampler, collate_fn=collate_text_batch)

        lora_ratio1 = float(stage1_cfg.get("lora_plus_ratio", 2.5))
        param_groups1 = _get_loraplus_param_groups(
            model,
            lr_lora=float(stage1_cfg.get("lr_lora", 1.5e-4)),
            lr_head=float(stage1_cfg.get("lr_head", 8e-4)),
            lora_plus_ratio=lora_ratio1,
            include_centered_head=False,
        )
        opt1 = torch.optim.AdamW(param_groups1, weight_decay=float(train_cfg.get("weight_decay", 0.01)))
        total_steps1 = len(loader1) * epochs1
        sched1 = get_cosine_schedule_with_warmup(
            opt1, num_warmup_steps=int(stage1_cfg.get("warmup_steps", 100)), num_training_steps=math.ceil(total_steps1 / accum1)
        )

        stage1_hard_weight = float(stage1_cfg.get("hard_class_weight", train_cfg.get("hard_class_weight", 1.0)))
        stage1_class_weights = torch.ones(num_classes, device=device, dtype=torch.float32)
        for lbl, local_idx in sub_lab2i.items():
            if LAB2I.get(lbl) in hard_set:
                stage1_class_weights[local_idx] = stage1_hard_weight

        model.train()
        opt1.zero_grad()
        global_step1 = 0
        stop_stage1 = False
        for ep in range(epochs1):
            if stop_stage1:
                break
            bucket_sampler.set_epoch(ep)
            for step_idx, batch in enumerate(loader1):
                global_step1 += 1
                mb_ids = batch["input_ids"].to(device)
                mb_mask = batch["attention_mask"].to(device)
                max_l = max(int(mb_mask.sum(dim=1).max().item()), 2)
                mb_ids = mb_ids[:, :max_l]
                mb_mask = mb_mask[:, :max_l]
                labels_dev = batch["labels"].to(device)

                with torch.amp.autocast("cuda", enabled=torch.cuda.is_available(), dtype=compute_dtype):
                    Z_s1, logits_p, _ = model(mb_ids, mb_mask, drop_first_token=not is_deberta)
                    loss_ce = family_margin_cross_entropy(logits_p, labels_dev, family_margin=family_margin, weight=stage1_class_weights)
                    loss_sup = compute_supcon_loss(Z_s1, labels_dev, temperature=0.10, prototypes=model.plain_head.weight.detach())
                    loss = (loss_ce + 0.15 * loss_sup) / accum1

                loss.backward()
                if global_step1 % accum1 == 0 or global_step1 == total_steps1:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt1.step()
                    sched1.step()
                    opt1.zero_grad()

                if global_step1 % 200 == 0:
                    elapsed_m = (time.time() - s1_start) / 60.0
                    print(
                        f"[Trainer Stage 1] Epoch {ep + 1}/{epochs1} Step {step_idx + 1}/{len(loader1)} (Global {global_step1}/{total_steps1}) | Loss: {loss.item() * accum1:.4f} | Elapsed: {elapsed_m:.1f}m"
                    )

                save_steps = int(stage1_cfg.get("save_steps", 500))
                if global_step1 % save_steps == 0:
                    try:
                        export_compatible_checkpoint(
                            model=model,
                            output_dir=out_dir,
                            layers=layers,
                            keep=keep_layers,
                            max_len=max_len,
                            end_id=end_id,
                            backbone_name=backbone_name,
                            class_labels=active_labels,
                            target_family=target_family if is_specialist else None,
                        )
                        print(f"[Trainer Stage 1] Intermediate checkpoint auto-saved at step {global_step1} to {out_dir}")
                        if step_callback is not None:
                            step_callback("checkpoint_saved", global_step1)
                    except Exception as e:
                        print(f"[Trainer Stage 1 Warning] Auto-save failed at step {global_step1}: {e}")

                if (time.time() - s1_start) >= max_s1_sec:
                    print(
                        f"[Trainer Stage 1] Reached {stage1_cfg.get('max_hours', 12.0)}h time budget at global step {global_step1}; advancing to Stage 2."
                    )
                    stop_stage1 = True
                    break

        del ds1, loader1, opt1, sched1, df_p
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # Intermediate checkpoint export after Stage 1
        try:
            export_compatible_checkpoint(
                model=model,
                output_dir=out_dir,
                layers=layers,
                keep=keep_layers,
                max_len=max_len,
                end_id=end_id,
                backbone_name=backbone_name,
                class_labels=active_labels,
                target_family=target_family if is_specialist else None,
            )
            print(f"[Trainer] Safe intermediate Stage 1 checkpoint exported to {out_dir}")
            if step_callback is not None:
                step_callback("checkpoint_saved", "stage1_finished")
        except Exception as e:
            print(f"[Trainer] Warning: could not export Stage 1 intermediate checkpoint: {e}")

    # Stage 2: Prompt-Grouped Fine-Tuning on train.csv
    stage2_cfg = train_cfg.get("stage2", {})
    freeze_backbone_s2 = bool(stage2_cfg.get("freeze_backbone", False))
    if freeze_backbone_s2:
        print("[Trainer] Freezing backbone parameters for Stage 2 (exclusively training norms, plain_head, and centered_head)...")
        for p in model.backbone.parameters():
            p.requires_grad = False
    df_tr = pd.read_csv(train_path)
    if is_specialist:
        df_tr = df_tr[df_tr["model"].isin(active_labels)].reset_index(drop=True)
    print(f"[Trainer] Stage 2 Fine-Tuning on {train_path} ({len(df_tr)} samples, {num_classes} classes)...")
    bs2 = int(stage2_cfg.get("batch_size", 2))
    accum2 = int(stage2_cfg.get("gradient_accumulation_steps", 1))
    ds2 = TextSequenceDataset(
        df_tr["text"].tolist(), [sub_lab2i[m] for m in df_tr["model"]], df_tr["prompt_id"].tolist(), tok, max_len=max_len, end_id=end_id
    )
    sampler2 = PromptGroupBatchSampler(df_tr["prompt_id"].tolist(), batch_size=bs2, shuffle=True)
    loader2 = DataLoader(ds2, batch_sampler=sampler2, collate_fn=collate_text_batch)

    lora_ratio2 = float(stage2_cfg.get("lora_plus_ratio", 2.5))
    param_groups2 = _get_loraplus_param_groups(
        model,
        lr_lora=float(stage2_cfg.get("lr_lora", 3e-5)),
        lr_head=float(stage2_cfg.get("lr_head", 2e-4)),
        lora_plus_ratio=lora_ratio2,
        include_centered_head=True,
    )
    opt2 = torch.optim.AdamW(param_groups2, weight_decay=float(train_cfg.get("weight_decay", 0.01)))
    epochs2 = int(stage2_cfg.get("epochs", 2))
    total_steps2 = len(loader2) * epochs2
    sched2 = get_cosine_schedule_with_warmup(
        opt2, num_warmup_steps=int(stage2_cfg.get("warmup_steps", 50)), num_training_steps=math.ceil(total_steps2 / accum2)
    )

    # SWA setup: average parameters across the final epoch of Stage 2 (accumulate in FP32)
    swa_enabled = bool(stage2_cfg.get("swa", True))
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    swa_shadow = {p: torch.zeros_like(p.data, dtype=torch.float32, device="cpu") for p in trainable_params} if swa_enabled else {}
    swa_steps = 0

    # Hard-Class Loss Upweighting for Stage 2 fine-tuning (default to 1.0 to preserve unbiased prior margins when Sinkhorn is disabled)
    stage2_hard_weight = float(stage2_cfg.get("hard_class_weight", 1.0))
    stage2_class_weights = torch.ones(num_classes, device=device, dtype=torch.float32)
    if stage2_hard_weight != 1.0:
        hard_set = set(HARD_CLASS_INDICES)
        for lbl, local_idx in sub_lab2i.items():
            if LAB2I.get(lbl) in hard_set:
                stage2_class_weights[local_idx] = stage2_hard_weight

    model.train()
    global_step2 = 0
    for ep in range(epochs2):
        sampler2.set_epoch(ep)
        opt2.zero_grad()
        for step_idx, batch in enumerate(loader2):
            global_step2 += 1
            train_prompt_group_step(
                model=model,
                opt=opt2,
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
                labels=batch["labels"].to(device),
                micro_batch_size=bs2,
                centered_weight=float(stage2_cfg.get("centered_weight", 0.60)),
                family_margin=family_margin,
                supcon_weight=supcon_weight,
                label_smoothing=0.10,
                accum_steps=accum2,
                compute_dtype=compute_dtype,
                weight=stage2_class_weights,
            )
            if global_step2 % accum2 == 0 or (step_idx + 1) == len(loader2):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt2.step()
                sched2.step()
                opt2.zero_grad()

                if swa_enabled and ep == (epochs2 - 1):
                    swa_steps += 1
                    for p in trainable_params:
                        swa_shadow[p].add_(p.data.detach().cpu())

    if swa_enabled and swa_steps > 0:
        print(f"[Trainer] Applying SWA weight averaging across {swa_steps} steps in final Stage 2 epoch...")
        for p in trainable_params:
            p.data.copy_((swa_shadow[p] / float(swa_steps)).to(p.device, p.dtype))
        del swa_shadow

    # Feature extraction for wPCA bank & checkpoint export
    del loader2, opt2, sched2, sampler2
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model.eval()
    loader_eval = DataLoader(ds2, batch_size=4, shuffle=False, collate_fn=collate_text_batch)
    Z_list = []
    with torch.no_grad():
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available(), dtype=compute_dtype):
            for batch in loader_eval:
                b_ids = batch["input_ids"].to(device)
                b_mask = batch["attention_mask"].to(device)
                max_l = max(int(b_mask.sum(dim=1).max().item()), 2)
                z_b = model.forward_features(b_ids[:, :max_l], b_mask[:, :max_l], drop_first_token=not is_deberta)
                Z_list.append(z_b.cpu().numpy())
    Z_train = np.vstack(Z_list)
    y_train = np.array([sub_lab2i[m] for m in df_tr["model"]])
    prompts_train = df_tr["prompt_id"].values

    # 5-Stage Cross-Validation & Overfitting Audit on Stage 2 Representations
    n_cv_splits = int(stage2_cfg.get("n_splits", 5))
    if stage2_cfg.get("check_overfitting", True) and len(np.unique(prompts_train)) >= 2:
        evaluate_stage2_oof_cv(
            Z=Z_train,
            y=y_train,
            prompts=prompts_train,
            n_splits=n_cv_splits,
            alpha=8.0 if is_specialist else 10.0,
            label_smoothing=0.05 if is_specialist else 0.10,
            num_classes=num_classes,
            class_labels=active_labels,
            output_dir=out_dir,
        )

    # Closed-form dual Ridge head blending:
    # Combines neural margin boundaries with exact L2-regularized closed-form ridge centroids over Z_train.
    # Applied to both specialist models (0.65 neural / 0.35 ridge) and 18-class champion model (0.75 neural / 0.25 ridge).
    r_alpha = 8.0 if is_specialist else 10.0
    r_smoothing = 0.05 if is_specialist else 0.10
    w_n = 0.65 if is_specialist else 0.75
    w_r = 0.35 if is_specialist else 0.25

    W_p_r, b_p_r, W_c_r, b_c_r = fit_ridge_heads(
        Z_train, y_train, prompts_train, alpha=r_alpha, label_smoothing=r_smoothing, num_classes=num_classes
    )
    _blend_head(model.plain_head, W_p_r, b_p_r, w_neural=w_n, w_ridge=w_r)
    _blend_head(model.centered_head, W_c_r, b_c_r, w_neural=w_n, w_ridge=w_r)
    print(f"[Trainer] Successfully blended dual Ridge heads into neural classifier (w_neural={w_n:.2f}, w_ridge={w_r:.2f}).")

    export_compatible_checkpoint(
        model=model,
        output_dir=out_dir,
        layers=layers,
        keep=keep_layers,
        max_len=max_len,
        end_id=end_id,
        backbone_name=backbone_name,
        Z_train=Z_train,
        y_train=y_train,
        prompts_train=prompts_train,
        class_labels=active_labels,
        target_family=target_family if is_specialist else None,
    )
    if step_callback is not None:
        try:
            step_callback("checkpoint_saved", "training_complete")
        except Exception as e:
            print(f"[Trainer Callback Warning] {e}")
    print(f"[Trainer] Checkpoint exported to {out_dir}")
    return out_dir


In [ ]:
if __name__ == '__main__':
    run_training_pipeline(TRAINING_CONFIG)
